# [프로젝트] 세그먼트 8개 요율표 — 추정 방법 5개 비교

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## 오늘 우리가 하게 될 일

<div style="background:#FFFFFF;color:#000000;padding:40px;border-radius:12px;border:1px solid #E6E6E6;border-left:4px solid #051C2C">
<div style="font-size:0.9em;color:#656565">모형추정 심화통계 · 복습 프로젝트</div>
<h1 style="color:#051C2C;margin:8px 0 16px">세그먼트 8개 요율표: 추정 방법 5개 비교</h1>
<div style="font-size:1.05em;color:#333333">재무팀이 세그먼트 8개를 담은 요율표 한 장을 요청했습니다. 미션 하나가 이 표의 열을 1개에서 4개까지 채우고, 다음 미션은 앞 미션이 해결하지 못한 문제에서 출발합니다.</div>
</div>

## 오늘 할 일

5일 동안 배운 추정 방법 5개를 같은 요율표 한 장에서 비교합니다. 미션이 끝날 때마다 지금까지 채운 표가 화면에 출력되고, 그 표에 남은 문제가 다음 미션의 출발점이 됩니다. 이 복습 프로젝트는 앞의 5권보다 쉽게 구성해, 각 날의 실습을 하지 못한 사람도 이 한 권만으로 추정 방법 5개를 모두 한 번씩 실행할 수 있습니다.

| 미션 | 앞 미션에서 받는 한계 | 그날 배운 방법 | 채울 열 |
|---|---|---|---|
| 1 | 표에 계약 수와 사고 건수와 노출만 있습니다 | 첫날 최대우도추정값, 넷째 날 표준오차 구간 | `mle_rate` · `se` · `lo_2se` · `hi_2se` |
| 2 | 미션 1 의 값과 구간은 그 세그먼트의 자료만 씁니다 | 둘째 날 감마-포아송 켤레 사후평균 | `post_mean_100` |
| 3 | 미션 2 가 더 쓰는 것은 계약 전체의 값 하나뿐입니다 | 셋째 날 로그 노출 오프셋 포아송 회귀 | `glm_rate` |
| 4 | 미션 1 의 구간은 좌우 대칭을 전제한 정규근사입니다 | 넷째 날 부트스트랩 백분위 구간 | `boot_lo` · `boot_hi` |
| 5 | 미션 4 까지는 값과 구간이 서로 다른 방법에서 나왔습니다 | 다섯째 날 닫힌 형태 부분 풀링 | `pooled_400` · `cred_lo_400` · `cred_hi_400` |
| 과제 | 같은 세그먼트에 값 4개와 구간 3벌이 남아 있습니다 | 값 하나를 골라 적고 심사 경로를 나눕니다 | `report_rate` · `decision` |

미션 5 의 **닫힌 형태**는 표본을 만들지 않고 식에 값을 대입해 바로 얻는 계산입니다.

완성할 요율표는 아래 모양입니다. 지금은 왼쪽 4개 열만 채워져 있습니다.

| segment | n | claims | exposure | mle_rate | post_mean_100 | glm_rate | boot_lo · boot_hi | pooled_400 | report_rate · decision |
|---|---|---|---|---|---|---|---|---|---|
| R24 B2 | 채움 | 채움 | 채움 | 미션 1 | 미션 2 | 미션 3 | 미션 4 | 미션 5 | 과제 |
| R21 B11 | 채움 | 채움 | 채움 | 미션 1 | 미션 2 | 미션 3 | 미션 4 | 미션 5 | 과제 |

## 오늘 쓰는 데이터

프랑스 자동차보험 계약 678,013건입니다. 지역과 브랜드를 짝지으면 세그먼트 242개가 나오고, 그 가운데 8개를 오늘의 요율표에 담습니다. 노출이 큰 세그먼트 4개와 사고가 1건 이상이면서 노출이 작은 세그먼트 4개이고, 노출이 6.62년뿐인 세그먼트도 그 안에 있습니다.

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Japanese_car_accident.jpg" width="640" height="360"/>
<p align="center"><em>▲ 교차로 접촉 사고 — 오늘 쓰는 계약 678,013건은 이런 사고가 몇 건 일어났는지와 그 계약이 몇 년 유지되었는지 두 숫자로 기록되어 있다 (출처: Wikipedia, Japanese_car_accident.jpg)</em></p>

<img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_tweedie_regression_insurance_claims_001.png" width="800" height="400"/>
<p align="center"><em>▲ 같은 프랑스 자동차보험 포트폴리오를 운전자 나이·차량 나이·보너스말러스로 나눠 사고 빈도를 그린 4장 — 계약이 몰린 구간에서는 관측과 모형 값이 거의 겹치는데 계약이 드문 양끝에서는 관측이 크게 변동한다 (출처: scikit-learn documentation)</em></p>

이 노트북에서 다루는 값은 하나뿐입니다. **연간 사고율 λ(건/년)** 입니다. 사고를 겪은 계약의 비율은 오늘 쓰지 않습니다. 사고 건수 합 대신 사고를 겪은 계약 수를, 노출 합 대신 계약 수를 섞으면 값이 낮아지는데, 그 실수를 미션 2 에서 직접 고쳐 봅니다.

오늘 만드는 표는 요율을 이루는 두 요소 가운데 빈도 한쪽입니다. 실제 요율은 여기에 건당 손해액(심도)을 곱해 정하고, 심도는 이 과정의 범위 밖입니다.

## 이 노트북을 쓰는 법

미션마다 셀이 4개입니다. 무엇을 만들지 적은 설명, 값을 만들어 주는 「준비 셀」, 지시를 적는 「PROMPT 셀」, 그리고 확인 문항입니다. 「PROMPT 셀」을 실행하면 그 셀의 출력과 그림을 읽어 바로 아래에 채점 결과가 출력됩니다.

미션 하나만 하고 싶으면 맨 위 설치 셀과 그 미션의 「준비 셀」을 차례로 실행합니다. 「준비 셀」이 앞 미션의 열을 참조 값으로 다시 만들어 두므로 가운데 미션부터 시작해도 그 미션이 단독으로 돌아갑니다.

채점은 LMS 에서 자동으로 준비되므로 [채점] 셀은 화면에 보이지 않습니다. 이 노트북을 내려받아 Colab 이나 로컬에서 열었다면 설치 셀 다음에 그 셀을 한 번 실행합니다. 그러면 판정과 힌트와 완성본 풀이가 LMS 화면과 똑같이 나옵니다.

In [ ]:
# 실습 준비 - 패키지 설치
!pip install -q "numpy>=1.26" "pandas>=2.0" "scipy>=1.11" "matplotlib>=3.7" "statsmodels>=0.14" "scikit-learn>=1.3"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
import warnings; warnings.filterwarnings("ignore")

# 한글 폰트 — Colab 이면 나눔고딕을 설치한 뒤 그 ttf 를 폰트 목록에 **직접** 등록한다.
#  설치는 디스크에 파일만 만든다. 이미 떠 있는 폰트 목록은 그 사실을 모르므로 addfont 로
#  등록해야 이름이 보인다(등록하지 않으면 DejaVu Sans 로 밀려 한글이 전부 □ 로 나온다).
import glob, shutil, subprocess
if shutil.which("apt-get"):
    subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"], check=False)
for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):   # 설치 여부와 무관하게 한 번
    try: fm.fontManager.addfont(_p)
    except Exception: pass
try: fm._get_font.cache_clear()
except Exception: pass
installed = {f.name for f in fm.fontManager.ttflist}
FONT = next((n for n in ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"] if n in installed), "DejaVu Sans")
plt.rcParams["font.family"] = FONT
plt.rcParams["axes.unicode_minus"] = FONT in {"Pretendard", "DejaVu Sans"}   # 나머지 폰트에는 U+2212 가 없다
%config InlineBackend.print_figure_kwargs = {"bbox_inches": None}   # 그림마다 폭이 깎이지 않게
plt.rcParams["figure.dpi"] = plt.rcParams["savefig.dpi"] = 150
plt.rcParams["figure.autolayout"] = True
plt.rcParams.update({"font.size": 11, "axes.titlesize": 15, "axes.labelsize": 12,
                     "legend.fontsize": 11, "figure.titlesize": 17, "font.weight": "regular",
                     "text.color": "black", "axes.labelcolor": "black", "axes.titlecolor": "black",
                     "axes.edgecolor": "#757575", "xtick.color": "#757575", "ytick.color": "#757575",
                     "xtick.labelcolor": "black", "ytick.labelcolor": "black",
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False})

# 오늘 그림 전체가 함께 쓰는 색 — 기준선=남색, 관측=파랑, 구간·강조=하늘색, 문맥=회색
COL = {"남색": "#051C2C", "파랑": "#2251FF", "하늘색": "#00A9F4",
       "회색": "#757575", "연회색": "#B3B3B3"}
FIG_W = 10.0     # 캔버스 폭은 오늘 그림 모두 이 값으로 고정하고 높이만 그림마다 조절한다

print("사용 폰트:", FONT,
      "| 색 팔레트 준비 완료 — 맨 위 [채점] 셀을 한 번 실행하면 판정과 힌트가 함께 나옵니다")

In [ ]:
#@title [채점] 이 셀을 먼저 실행하세요 (한 번이면 됩니다)
#   미션의 「PROMPT 셀」을 실행하면 그 셀의 출력과 그림을 읽어 바로 아래에 채점 결과를 인쇄합니다.
#   통과하면 그 자리에서 완성본 풀이가 열리고, 아니면 항목별 힌트와 튜터에 보낼 질문이 나옵니다.
#   읽지 않고 실행만 해도 됩니다 — 기대값과 풀이는 이 셀에만 있고 「PROMPT 셀」에는 없습니다.
import base64
import glob
import io
import json
import re
import shutil
import subprocess
import sys

import matplotlib.font_manager as fm      # 아래 폰트 자급이 쓴다
import matplotlib.pyplot as plt
import numpy as np


def _font_bootstrap():
    """한글 폰트 자급 — 맨 위 설치·폰트 셀을 건너뛰어도 그래프의 한글이 깨지지 않게.

    LMS 는 이 셀만 자동 실행하므로 폰트 준비도 여기서 한 번 더 한다. 설치·폰트 셀과 같은
    규칙이라 둘 다 실행돼도 결과가 같고, apt-get 이 없는 자리(로컬)에서는 곧바로 지나간다.
    설치한 ttf 는 `addfont` 로 폰트 목록에 직접 등록한다 — 등록하지 않으면 목록이 옛 상태라
    한글이 □ 로 깨진다(2026-09-09 Colab 실사고).
    """
    try:
        cands = ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"]
        installed = {f.name for f in fm.fontManager.ttflist}
        if not any(n in installed for n in cands) and shutil.which("apt-get"):
            subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"],
                           check=False, timeout=120)
        # 설치는 파일만 만든다 — 이미 떠 있는 폰트 목록에 addfont 로 등록해야 이름이 보인다.
        # (설치를 건너뛴 자리도 나눔 ttf 가 이미 있으면 이 행들만으로 잡힌다.)
        for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):
            try: fm.fontManager.addfont(_p)
            except Exception: pass
        try: fm._get_font.cache_clear()
        except Exception: pass
        installed = {f.name for f in fm.fontManager.ttflist}
        _font = next((n for n in cands if n in installed), "DejaVu Sans")
        plt.rcParams["font.family"] = _font
        plt.rcParams["axes.unicode_minus"] = _font in {"Pretendard", "DejaVu Sans"}
    except Exception:
        print("한글 폰트를 못 찾아 그래프 글자가 깨질 수 있습니다 —"
              " 맨 위 설치·폰트 셀을 실행하세요")


_font_bootstrap()


def _tail_numbers(tails):
    """라벨 뒤 꼬리에서 괄호 설명을 걷어내고 첫 숫자 하나씩만 모읍니다.

    꼬리 전체의 숫자를 담으면 괄호 안 설명에 우연히 든 수가 기준 범위에 들어 틀린 값을
    통과시킵니다. 예를 들어 「… = 0.074955 건/년 (사고 0건 재표본 112벌)」 은 기준이 0 부근인
    항목에서 괄호 안의 0 으로 열립니다. 그래서 괄호 안은 버리고 맨 앞 숫자 하나만 봅니다.
    """
    picked = []
    for one in tails:
        plain, depth = [], 0
        for ch in one:
            if ch in "([":
                depth += 1
            elif ch in ")]":
                depth = max(0, depth - 1)
            elif depth == 0:
                plain.append(ch)
        got = _nums("".join(plain))
        if got:
            picked.append(got[0])
    return picked


def _dec_json(s):                          # base64(JSON) → 값. 리스트는 원래 튜플이라 되돌립니다
    def _t(v):
        return ({k: _t(x) for k, x in v.items()} if isinstance(v, dict)
                else tuple(_t(x) for x in v) if isinstance(v, list) else v)
    return _t(json.loads(base64.b64decode(s).decode("utf-8")))


# 채점 기준 — base64(JSON), 통과 판정에만 쓴다
_ANS = _dec_json(
           "eyJtMSI6IHsidmFsdWVzIjogeyJSMjEgQjExIOydmCDstZzrjIDsmrDrj4TstpTs"
           "oJXqsJIgKOqxtC/rhYQpIjogWzAuMjk5NSwgMC4zMDQ1LCAi6rG0L+uFhCJdLCAi"
           "7ZWY7ZWc7J20IOydjOyImOyduCDshLjqt7jrqLztirgg7IiYICjqsJwpIjogWzQs"
           "IDQsICLqsJwiXX0sICJwbG90IjogeyJraW5kIjogImludGVydmFsIiwgIm5fc3Bh"
           "bnMiOiBbOCwgMTBdLCAibl9wb2ludHMiOiBbOCwgMTBdLCAic3Bhbl9taW4iOiBb"
           "MC4wMDUsIDAuMDA4XSwgInNwYW5fbWF4IjogWzAuOCwgMC45XSwgInhfcmFuZ2Ui"
           "OiBbLTAuMTI1MSwgMC43Mjk0XSwgIuy2leudvOuyqCI6IHRydWV9fSwgIm0yIjog"
           "eyJ2YWx1ZXMiOiB7IlIyNCBCMiDsnZgg7IKs7ZuE7Y+J6regICjqsbQv64WEKSI6"
           "IFswLjA4ODUsIDAuMDg5NiwgIuqxtC/rhYQiXSwgIlIyMSBCMTEg7J2YIOyCrO2b"
           "hO2Pieq3oCAo6rG0L+uFhCkiOiBbMC4xMTI2LCAwLjExMzcsICLqsbQv64WEIl19"
           "LCAicGxvdCI6IHsia2luZCI6ICJzY2F0dGVyIiwgIm5fcG9pbnRzIjogWzgsIDEw"
           "XSwgInhfcmFuZ2UiOiBbMC4wODgsIDAuMzA5Ml0sICJ5X3JhbmdlIjogWzAuMDg4"
           "LCAwLjE1NDNdLCAiZ2FwX21pbiI6IDAuMTUsICLstpXrnbzrsqgiOiB0cnVlfX0s"
           "ICJtMyI6IHsidmFsdWVzIjogeyJSMTEgQjEyIOydmCDsmIjsuKEg7IKs6rOg7Jyo"
           "ICjqsbQv64WEKSI6IFswLjE1NTYsIDAuMTU4NSwgIuqxtC/rhYQiXSwgIuu4jOue"
           "nOuTnCBCMTIg7J2YIOyCrOqzoOycqCDrsLDsnKggKOuwsCkiOiBbMS40MjUsIDEu"
           "NDc1LCAi67CwIl19LCAicGxvdCI6IHsia2luZCI6ICJiYXIiLCAibiI6IDE2LCAi"
           "eW1heCI6IFswLjE1LCAwLjE2NV0sICLstpXrnbzrsqgiOiB0cnVlfX0sICJtNCI6"
           "IHsidmFsdWVzIjogeyJSMjQgQjIg7J2YIOu2gO2KuOyKpO2KuOueqSDqtazqsIQg"
           "6ri47J20ICjqsbQv64WEKSI6IFswLjAwNTUsIDAuMDA3MiwgIuqxtC/rhYQiXSwg"
           "IlIyMSBCMTEg7J2YIOu2gO2KuOyKpO2KuOueqSDqtazqsIQg7ZWY7ZWcICjqsbQv"
           "64WEKSI6IFswLjAsIDAuMDAwNiwgIuqxtC/rhYQiXX0sICJwbG90IjogeyJraW5k"
           "IjogImJhciIsICJuIjogMTYsICJ5bWF4IjogWzAuOCwgMS4wNV0sICLstpXrnbzr"
           "sqgiOiB0cnVlfX0sICJtNSI6IHsidmFsdWVzIjogeyJSMjEgQjExIOydmCDrtoDr"
           "toQg7ZKA66eBIOqwkiAo6rG0L+uFhCkiOiBbMC4xMDMsIDAuMTA0OCwgIuqxtC/r"
           "hYQiXSwgIlIyMSBCMTEg7J2YIOyLoOyaqeq1rOqwhCDquLjsnbQgKOqxtC/rhYQp"
           "IjogWzAuMDYxNSwgMC4wNjM2LCAi6rG0L+uFhCJdfSwgInBsb3QiOiB7ImtpbmQi"
           "OiAic2NhdHRlciIsICJuX3BvaW50cyI6IFsxNiwgMjBdLCAieF9yYW5nZSI6IFs2"
           "LjYsIDM2MDAwLjBdLCAieV9yYW5nZSI6IFswLjAxNjMsIDAuOTk3M10sICJ4c2Nh"
           "bGUiOiAibG9nIiwgIuy2leudvOuyqCI6IHRydWV9fSwgIuqzvOygnCI6IHsidmFs"
           "dWVzIjogeyLsoIDsnqXtlZwg7JqU7Jyo7ZGc7J2YIO2WiSDsiJggKO2WiSkiOiBb"
           "OCwgOCwgIu2WiSJdLCAi7JqU7Jyo7ZGcIOqwkuydtCDsi6DsmqnqtazqsIQg7JWI"
           "7JeQIOuToCDruYTsnKggKCUpIjogWzk5LjksIDEwMC4xLCAiJSJdfSwgInBsb3Qi"
           "OiB7ImtpbmQiOiAic2NhdHRlciIsICJuX3BvaW50cyI6IFs4LCAxMF0sICJ4X3Jh"
           "bmdlIjogWzYuNiwgMzYwMDAuMF0sICJ5X3JhbmdlIjogWzAuMDY4OSwgMC42MDkx"
           "XSwgInhzY2FsZSI6ICJsb2ciLCAi7LaV652867KoIjogdHJ1ZX19fQ==")
_REVEAL = {"m1":                   # 완성본 풀이 — base64, 통과할 때만 디코드해 인쇄합니다
           "8J+UkyDrr7jshZggMSDsmYTshLHrs7gg4oCUIOOAjOyXrOq4sOu2gO2EsCBBSSDs"
           "vZTrk5zjgI0g7JWE656Y7JeQIOuTpOyWtOyZgOyVvCDtlZjripQg7L2U65Oc64qU"
           "IOydtCDsoJXrj4TsnoXri4jri6QKCuKWoCDsnbQg7L2U65Oc66W8IOu2gOuluCDt"
           "lITroaztlITtirgg7JiICgogICAg7J2066+4IOunjOuTpOyWtOyguCDsnojripQg"
           "ZGYg7JmAIHJhdGVfdGFibGUsIGxhbV9hbGwsIHdhdGNoX25hbWUsIHNob3dfcmF0"
           "ZV90YWJsZSDsnYQg6re464yA66GcIOyTsOqzoCDri6Tsi5wg66eM65Ok7KeA64qU"
           "IOunkOyVhCDspJguIHJhdGVfdGFibGUg7JeQIG1sZV9yYXRlIOyXtOydhCDsgqzq"
           "s6Ag6rG07IiYIO2VqQogICAgY2xhaW1zIOulvCDrhbjstpwg7ZWpIGV4cG9zdXJl"
           "IOuhnCDrgpjriIgg6rCS7Jy866GcIOuNlO2VmOqzoCwgc2Ug7Je07J2EIGNsYWlt"
           "cyDsnZgg7KCc6rOx6re87J2EIGV4cG9zdXJlIOuhnCDrgpjriIgg6rCS7Jy866Gc"
           "IOuNlO2VtCDspJguIOqxsOq4sOyEnCBsb18yc2Ug64qUIG1sZV9yYXRlIOyXkOyE"
           "nCBzZSDsnZggMuuwsOulvAogICAg67qAIOqwkiwgaGlfMnNlIOuKlCDrjZTtlZwg"
           "6rCS7J207JW8LiDqt7jri6TsnYwgIlIyMSBCMTEg7J2YIOy1nOuMgOyasOuPhOy2"
           "lOygleqwkiA9ICjqsJIpIOqxtC/rhYQiIOydhCDshozsiJjsoJAg7JWE656YIDbs"
           "npDrpqzroZwg7ZWcIO2WiSwgIu2VmO2VnOydtCDsnYzsiJjsnbgg7IS46re466i8"
           "7Yq4IOyImCA9ICjqsJIpIOqwnCIg66W8IOygleyImOuhnCDtlZwg7ZaJCiAgICBw"
           "cmludCDtlZjqs6Agc2hvd19yYXRlX3RhYmxlKCkg66GcIOyngOq4iOq5jOyngCDs"
           "sYTsmrQg7Je07J2EIOuqqOuRkCDri7TsnYAg7ZGc64+EIOy2nOugpe2VtCDspJgu"
           "IOq3uOuemO2UhOuKlCDtlZwg7J6l66eMIOKAlCDshLjqt7jrqLztirggOOqwnOul"
           "vCDshLjroZzroZwg67Cw7LmY7ZWY6rOgIOyEuOq3uOuovO2KuOuniOuLpCBsb18y"
           "c2Ug67aA7YSwIGhpXzJzZSDquYzsp4Drpbwg6rW17J2AIOqwgOuhnAogICAg66eJ"
           "64yA66GcIOq4i+qzoCBtbGVfcmF0ZSDsnITsuZjsl5Ag66eI7Luk66W8IO2VmOuC"
           "mOyUqSDtkZzsi5ztlZwg65KkLCBsYW1fYWxsIOychOy5mOyXkCDshLjroZwg7KCQ"
           "7ISg7J2EIOq4i+qzoCDshLjroZzstpUg64iI6riI7JeQIOyEuOq3uOuovO2KuCDs"
           "nbTrpoTsnYQg7KCB6rOgIOqwgOuhnOy2leqzvCDshLjroZzstpXsl5Ag652867Ko"
           "7J2EIOuLrOyVhCDspJguCgogICAgcmF0ZV90YWJsZVsibWxlX3JhdGUiXSA9IHJh"
           "dGVfdGFibGVbImNsYWltcyJdIC8gcmF0ZV90YWJsZVsiZXhwb3N1cmUiXQogICAg"
           "cmF0ZV90YWJsZVsic2UiXSA9IG5wLnNxcnQocmF0ZV90YWJsZVsiY2xhaW1zIl0p"
           "IC8gcmF0ZV90YWJsZVsiZXhwb3N1cmUiXSAgICMgU0UgPSDiiJoo7IKs6rOgIO2V"
           "qSkgw7cg64W47LacIO2VqQogICAgcmF0ZV90YWJsZVsibG9fMnNlIl0gPSByYXRl"
           "X3RhYmxlWyJtbGVfcmF0ZSJdIC0gMiAqIHJhdGVfdGFibGVbInNlIl0KICAgIHJh"
           "dGVfdGFibGVbImhpXzJzZSJdID0gcmF0ZV90YWJsZVsibWxlX3JhdGUiXSArIDIg"
           "KiByYXRlX3RhYmxlWyJzZSJdCgogICAgd2F0Y2ggPSByYXRlX3RhYmxlLmxvY1ty"
           "YXRlX3RhYmxlWyJzZWdtZW50Il0gPT0gd2F0Y2hfbmFtZV0uaWxvY1swXQogICAg"
           "cHJpbnQoZiJ7d2F0Y2hfbmFtZX0g7J2YIOy1nOuMgOyasOuPhOy2lOygleqwkiA9"
           "IHt3YXRjaFsnbWxlX3JhdGUnXTouNmZ9IOqxtC/rhYQiKQogICAgcHJpbnQoZiLt"
           "lZjtlZzsnbQg7J2M7IiY7J24IOyEuOq3uOuovO2KuCDsiJggPSB7aW50KChyYXRl"
           "X3RhYmxlWydsb18yc2UnXSA8IDApLnN1bSgpKX0g6rCcIikKICAgIHNob3dfcmF0"
           "ZV90YWJsZSgpCgogICAgeXBvcyA9IG5wLmFyYW5nZShsZW4ocmF0ZV90YWJsZSkp"
           "Wzo6LTFdICAgICAgICAgICAgICAgICAjIOychOyXkOyEnCDslYTrnpjroZwg7IS4"
           "6re466i87Yq4IDjqsJwKICAgIGZpZywgYXggPSBwbHQuc3VicGxvdHMoZmlnc2l6"
           "ZT0oRklHX1cgKiAwLjgsIDMuOCkpCiAgICBheC5obGluZXMoeXBvcywgcmF0ZV90"
           "YWJsZVsibG9fMnNlIl0sIHJhdGVfdGFibGVbImhpXzJzZSJdLCBjb2xvcj1DT0xb"
           "Iu2VmOuKmOyDiSJdLCBsdz00KQogICAgYXgucGxvdChyYXRlX3RhYmxlWyJtbGVf"
           "cmF0ZSJdLCB5cG9zLCAibyIsIGNvbG9yPUNPTFsi64Ko7IOJIl0pCiAgICBheC5h"
           "eHZsaW5lKGxhbV9hbGwsIGNvbG9yPUNPTFsi7ZqM7IOJIl0sIGxzPSI6IikgICAg"
           "ICAgICAgICMg6rOE7JW9IOyghOyytOydmCDqsJIKICAgIGF4LnNldF95dGlja3Mo"
           "eXBvcykKICAgIGF4LnNldF95dGlja2xhYmVscyhyYXRlX3RhYmxlWyJzZWdtZW50"
           "Il0pCiAgICBheC5zZXRfeGxhYmVsKCLsl7DqsIQg7IKs6rOg7JyoICjqsbQv64WE"
           "KSIpCiAgICBheC5zZXRfeWxhYmVsKCLsmpTsnKjtkZzsnZgg7IS46re466i87Yq4"
           "IikKICAgIHBsdC5zaG93KCkKCu2VtOyEnQoKLSDrhbjstpzsnbQg7YGwIOyEuOq3"
           "uOuovO2KuOydmCDqtazqsIQg6ri47J2064qUIDAuMDA2IOqxtC/rhYQg7IiY7KSA"
           "7J206rOgLCDqs4Tslb0gMTLqsbTsnbgg7IS46re466i87Yq464qUIDAuODUg6rG0"
           "L+uFhOydtOudvCAxMzbrsLAg7LCo7J207J6F64uI64ukLiDqtazqsIQg6ri47J20"
           "66W8IOydtOunjO2BvCDri6TrpbTqsowg66eM65OgIOqyg+ydgCDqs4Tslb0g7IiY"
           "6rCAIOyVhOuLiOudvCDriITsoIHrkJwg7IKs6rOgIOqxtOyImOyeheuLiOuLpC4g"
           "7IOB64yAIO2RnOykgOyYpOywqOqwgCAxIMO3IOKImijsgqzqs6Ag6rG07IiYIO2V"
           "qSkg7Jy866GcIOygleumrOuQmOq4sCDrlYzrrLjsnoXri4jri6QuCi0g7ZWY7ZWc"
           "7J20IOydjOyImOyduCDshLjqt7jrqLztirggNOqwnOuKlCDsgqzqs6DsnKjsnbQg"
           "7J2M7IiY7J28IOyImCDsnojri6TripQg65y77J20IOyVhOuLmeuLiOuLpC4g7LaU"
           "7KCV6rCSIMKxIDIgw5cg7ZGc7KSA7Jik7LCoIOudvOuKlCDsoJXqt5zqt7zsgqzq"
           "sIAg6re4IOyEuOq3uOuovO2KuOyXkOyEnCDshLHrpr3tlZjsp4Ag7JWK64qU64uk"
           "64qUIOucu+yeheuLiOuLpC4g7IKs6rOg6rCAIDLqsbTrv5Dsnbgg7IS46re466i8"
           "7Yq47JeQ7IScIOy2lOygleqwkuydmCDrtoTtj6zripQg64yA7Lmt7J20IOyVhOuL"
           "iOudvCDsmKTrpbjsqr3snLzroZwg6ri46rKMIOy5mOyasOyzkCDsnojsirXri4jr"
           "i6QuCi0g6re4656Y7IScIOydtCDsl7Trp4zsnLzroZzripQg7JqU7Jyo7J2EIOyg"
           "le2VoCDsiJgg7JeG64qUIOyEuOq3uOuovO2KuOqwgCA06rCcIOuCqOyKteuLiOuL"
           "pC4g66+47IWYIDIg67aA7YSw64qUIOq3uCA06rCc7JeQIOqzhOyVvSDsoITssrTs"
           "nZgg6rCS7J2EIOyWvOuniOuCmCDrsJjsmIHtlaDsp4Drpbwg64uk66O564uI64uk"
           "Lgo=",
           "m2":
           "8J+UkyDrr7jshZggMiDsmYTshLHrs7gg4oCUIOOAjOyXrOq4sOu2gO2EsCBBSSDs"
           "vZTrk5zjgI0g7JWE656Y7JeQIOuTpOyWtOyZgOyVvCDtlZjripQg7L2U65Oc64qU"
           "IOydtCDsoJXrj4TsnoXri4jri6QKCuKWoCDsnbQg7L2U65Oc66W8IOu2gOuluCDt"
           "lITroaztlITtirgg7JiICgogICAg7KCE7J6EIOuLtOuLueyekCDsvZTrk5zsl5Ds"
           "hJwg7IKs7ZuE7Y+J6regIO2VnCDtlonrp4wg6rOg7LOQIOykmC4g67aE7J6Q64qU"
           "IGFscGhhX3ByaW9yIOyXkCDqt7gg7IS46re466i87Yq47J2YIOyCrOqzoCDqsbTs"
           "iJgg7ZWpIGNsYWltcyDrpbwg642U7ZWcIOqwkuydtOqzoCwg67aE66qo64qUIHBy"
           "aW9yX2V4cG9zdXJlIOyXkCDrhbjstpwg7ZWpCiAgICBleHBvc3VyZSDrpbwg642U"
           "7ZWcIOqwkuydtOyVvC4g6rOE7JW9IOyImCBuIOqzvCDsgqzqs6Drpbwg6rKq7J2A"
           "IOqzhOyVvSDsiJggbl9jbGFpbV9jb250cmFjdHMg64qUIOuRmCDri6Qg7JOw7KeA"
           "IOunkOyVhCDspJguIOuCmOuouOyngOuKlCDqt7jrjIDroZwg65GQ6rOgLCAiUjI0"
           "IEIyIOydmCDsgqztm4Ttj4nqt6AgPSAo6rCSKSDqsbQv64WEIgogICAg6rO8ICJS"
           "MjEgQjExIOydmCDsgqztm4Ttj4nqt6AgPSAo6rCSKSDqsbQv64WEIiDsnYQg7IaM"
           "7IiY7KCQIOyVhOuemCA27J6Q66as66GcIO2VnCDtlonslKkgcHJpbnQg7ZWY6rOg"
           "IHNob3dfcmF0ZV90YWJsZSgpIOuhnCDsp4DquIjquYzsp4Ag7LGE7Jq0IOyXtOyd"
           "hCDrqqjrkZAg64u07J2AIO2RnOulvCDstpzroKXtlbQg7KSYLiDqt7jrnpjtlITr"
           "ipQg7ZWcIOyepeunjCDigJQKICAgIOqwgOuhnOy2leyXkCByYXRlX3RhYmxlIOyd"
           "mCBtbGVfcmF0ZSwg7IS466Gc7LaV7JeQIHBvc3RfbWVhbl8xMDAg7J2EIOuRlCDs"
           "grDsoJDrj4TroZwg7KCQIDjqsJzrpbwg7LCN6rOgIOqwgOuhnOy2leqzvCDshLjr"
           "oZzstpXsl5Ag652867Ko7J2EIOuLrOyVhCDspJguCgogICAgcHJpb3JfZXhwb3N1"
           "cmUgPSAxMDAuMAogICAgYWxwaGFfcHJpb3IgPSBsYW1fYWxsICogcHJpb3JfZXhw"
           "b3N1cmUKICAgIHJhdGVfdGFibGVbInBvc3RfbWVhbl8xMDAiXSA9ICgoYWxwaGFf"
           "cHJpb3IgKyByYXRlX3RhYmxlWyJjbGFpbXMiXSkgICAgICAgICMg67aE7J6QID0g"
           "7IKs6rOgIOqxtOyImCDtlakKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAg"
           "ICAgICAvIChwcmlvcl9leHBvc3VyZSArIHJhdGVfdGFibGVbImV4cG9zdXJlIl0p"
           "KSAgIyDrtoTrqqggPSDrhbjstpwg7ZWpCiAgICBmb3IgbmFtZSBpbiAoIlIyNCBC"
           "MiIsICJSMjEgQjExIik6CiAgICAgICAgcm93ID0gcmF0ZV90YWJsZS5sb2NbcmF0"
           "ZV90YWJsZVsic2VnbWVudCJdID09IG5hbWVdLmlsb2NbMF0KICAgICAgICBwcmlu"
           "dChmIntuYW1lfSDsnZgg7IKs7ZuE7Y+J6regID0ge3Jvd1sncG9zdF9tZWFuXzEw"
           "MCddOi42Zn0g6rG0L+uFhCIpCiAgICBzaG93X3JhdGVfdGFibGUoKQoKICAgIGZp"
           "ZywgYXggPSBwbHQuc3VicGxvdHMoZmlnc2l6ZT0oRklHX1cgKiAwLjcsIDQuMCkp"
           "CiAgICBheC5zY2F0dGVyKHJhdGVfdGFibGVbIm1sZV9yYXRlIl0sIHJhdGVfdGFi"
           "bGVbInBvc3RfbWVhbl8xMDAiXSwgY29sb3I9Q09MWyLtlZjripjsg4kiXSkKICAg"
           "IGF4LnNldF94bGFiZWwoIuy1nOuMgOyasOuPhOy2lOygleqwkiAo6rG0L+uFhCki"
           "KQogICAgYXguc2V0X3lsYWJlbCgi7IKs7ZuE7Y+J6regICjqsbQv64WEKSIpCiAg"
           "ICBwbHQuc2hvdygpCgrtlbTshJ0KCi0g7KCE7J6EIOuLtOuLueyekCDsvZTrk5zq"
           "sIAg64K4IOqwkuydtCDrkZAg7IS46re466i87Yq4IOuqqOuRkCDrgq7slZjrjZgg"
           "7JuQ7J247J2AIO2VnCDqs7PsnbQg7JWE64uI6528IOuRkCDqs7PsnoXri4jri6Qu"
           "IOu2hOyekOulvCDsgqzqs6Drpbwg6rKq7J2AIOqzhOyVvSDsiJjroZwg64yA7J6F"
           "7ZW0IOyekeyVhOyhjOqzoCwg67aE66qo66W8IOqzhOyVvSDsiJjroZwg64yA7J6F"
           "7ZW0IOy7pOyhjOyKteuLiOuLpC4g6rOE7JW9IO2VnCDqsbTsnZgg64W47Lac7J20"
           "IDHrhYTrs7Tri6Qg7Ken7Jy866+A66GcIOqzhOyVvSDsiJjripQg64W47LacIO2V"
           "qeuztOuLpCDtgazqs6AsIO2VnCDqs4Tslb3sl5DshJwg7IKs6rOg6rCAIDLqsbQg"
           "7J207IOBIOuCnCDqsr3smrDqsIAg7J6I7Jy866+A66GcIOyCrOqzoCDqsbTsiJgg"
           "7ZWp7J2AIOyCrOqzoOulvCDqsqrsnYAg6rOE7JW9IOyImOuztOuLpCDtgb3ri4jr"
           "i6QuCi0g6rOg7LmcIOuSpOydmCDqsJLsnYQg67O066m0IOuFuOy2nOydtCAzNSw5"
           "NDnrhYTsnbgg7IS46re466i87Yq47JeQ7ISc64qUIOyCrOyghCDqsJXrj4QgMTAw"
           "64WE7J20IOqwgOykkey5mOydmCAwLjMl67CW7JeQIOywqOyngO2VmOyngCDrqrvt"
           "lbQg7LWc64yA7Jqw64+E7LaU7KCV6rCS6rO8IOyGjOyImOygkCDslYTrnpggNOye"
           "kOumrOq5jOyngCDqsJnsirXri4jri6QuIOuFuOy2nCA2LjYy64WE7J24IOyEuOq3"
           "uOuovO2KuOyXkOyEnOuKlCDqt7ggMTAw64WE7J20IOqwgOykkey5mOydmCA5NCXr"
           "pbwg7LCo7KeA7ZW0IDAuMzAyIOqxtC/rhYTsnbQgMC4xMTMg6rG0L+uFhOycvOuh"
           "nCDqsJDshoztlanri4jri6QuCi0g7IKs7KCEIOqwleuPhOulvCDqsIDsg4Eg64W4"
           "7LacIOuqhyDrhYTsnLzroZwg7KCB7Jy866m0IOydtCDruYTqtZDqsIAg6rOn67CU"
           "66GcIOydve2emeuLiOuLpC4gMTAw64WE7J20652864qUIOqwkuydgCDqs4Tslb0g"
           "MTAw6rG07J20IOyVhOuLiOudvCDrhbjstpwgMTAw64WE7Ja07LmY7J2YIOqyve2X"
           "mOydhCDrr7jrpqwg67CY7JiB7ZaI64uk64qUIOucu+yeheuLiOuLpC4K",
           "m3":
           "8J+UkyDrr7jshZggMyDsmYTshLHrs7gg4oCUIOOAjOyXrOq4sOu2gO2EsCBBSSDs"
           "vZTrk5zjgI0g7JWE656Y7JeQIOuTpOyWtOyZgOyVvCDtlZjripQg7L2U65Oc64qU"
           "IOydtCDsoJXrj4TsnoXri4jri6QKCuKWoCDsnbQg7L2U65Oc66W8IOu2gOuluCDt"
           "lITroaztlITtirgg7JiICgogICAg7J2066+4IOygge2VqeydtCDrgZ3rgpwgZ2xt"
           "X3JlcyDsmYAgc2VnX2ZyYW1lLCBicmFuZF9yZWYsIHJhdGVfdGFibGUg7J2EIOq3"
           "uOuMgOuhnCDsk7Dqs6Ag64uk7IucIOygge2Vqe2VmOyngOuKlCDrp5DslYQg7KSY"
           "LiBzZWdfZnJhbWUg7J2EIGdsbV9yZXMg7J2YIOyYiOy4oeyXkCDsoITri6ztlZjq"
           "s6Ag7Jik7ZSE7IWL7JeQ64qUIDDsnLzroZwKICAgIOyxhOyatCDrsLDsl7TsnYQg"
           "7KCE64us7ZW07IScIOuFuOy2nCAx64WE7J24IOqzhOyVveydmCDsmIjsuKEg7IKs"
           "6rOg7Jyo7J2EIOuwm+qzoCwg6re4IOqwkuydhCByYXRlX3RhYmxlIOydmCBnbG1f"
           "cmF0ZSDsl7TroZwg642U7ZW0IOykmC4g67iM656c65OcIEIxMiDsnZgg6rOE7IiY"
           "7JeQIOyngOyImOulvCDsoIHsmqntlZwg6rCS7J20IOq4sOykgCDruIzrnpzrk5wK"
           "ICAgIOuMgOu5hCDrsLDsnKjsnbTslbwuIOq3uOuLpOydjCAiUjExIEIxMiDsnZgg"
           "7JiI7LihIOyCrOqzoOycqCA9ICjqsJIpIOqxtC/rhYQiIOydhCDshozsiJjsoJAg"
           "7JWE656YIDbsnpDrpqzroZwg7ZWcIO2WiSwgIuu4jOuenOuTnCBCMTIg7J2YIOyC"
           "rOqzoOycqCDrsLDsnKggPSAo6rCSKSDrsLAiIOulvCDshozsiJjsoJAg7JWE656Y"
           "IDTsnpDrpqzroZwg7ZWcIO2WiQogICAgcHJpbnQg7ZWY6rOgLCDrsLDsnKjsnYQg"
           "7J2964qUIOusuOyepeuPhCDtlZwg7ZaJIOuNlCDstpzroKXtlbQg7KSYLiBzaG93"
           "X3JhdGVfdGFibGUoKSDroZwg7KeA6riI6rmM7KeAIOyxhOyatCDsl7TsnYQg66qo"
           "65GQIOuLtOydgCDtkZzrj4Qg7Lac66Cl7ZW0IOykmC4g6re4656Y7ZSE64qUIO2V"
           "nCDsnqXrp4wg4oCUIOyEuOq3uOuovO2KuOuniOuLpCDrr7jshZggMiDsnZggcG9z"
           "dF9tZWFuXzEwMCDqs7wKICAgIOydtOuyiCDrr7jshZjsnZggZ2xtX3JhdGUg66W8"
           "IOuCmOuegO2eiCDrkZQg66eJ64yAIDE26rCc66W8IOq3uOumrOqzoCDrspTroYDs"
           "mYAg6rCA66Gc7LaV6rO8IOyEuOuhnOy2lSDrnbzrsqjsnYQg64us7JWEIOykmC4K"
           "CiAgICByYXRlX3RhYmxlWyJnbG1fcmF0ZSJdID0gbnAuYXNhcnJheSgKICAgICAg"
           "ICBnbG1fcmVzLnByZWRpY3Qoc2VnX2ZyYW1lLCBvZmZzZXQ9bnAuemVyb3MobGVu"
           "KHNlZ19mcmFtZSkpKSkgICAgICMg7Jik7ZSE7IWLIDAgPSDrhbjstpwgMeuFhAog"
           "ICAgYjEyX3JhdGlvID0gZmxvYXQobnAuZXhwKGdsbV9yZXMucGFyYW1zWyJDKFZl"
           "aEJyYW5kKVtULkIxMl0iXSkpICAgICAgICMg6rOE7IiY7JeQIOyngOyImCDihpIg"
           "7IKs6rOg7JyoIOuwsOycqAoKICAgIHJvdyA9IHJhdGVfdGFibGUubG9jW3JhdGVf"
           "dGFibGVbInNlZ21lbnQiXSA9PSAiUjExIEIxMiJdLmlsb2NbMF0KICAgIHByaW50"
           "KGYiUjExIEIxMiDsnZgg7JiI7LihIOyCrOqzoOycqCA9IHtyb3dbJ2dsbV9yYXRl"
           "J106LjZmfSDqsbQv64WEIikKICAgIHByaW50KGYi67iM656c65OcIEIxMiDsnZgg"
           "7IKs6rOg7JyoIOuwsOycqCA9IHtiMTJfcmF0aW86LjRmfSDrsLAiKQogICAgcHJp"
           "bnQoZiLquLDspIAg67iM656c65OcIHticmFuZF9yZWZ9IOqzvCDruYTqtZDtlZjr"
           "qbQgQjEyIOydmCDsl7DqsIQg7IKs6rOg7Jyo7J20IHtiMTJfcmF0aW86LjRmfSDr"
           "sLDsnoXri4jri6QiKQogICAgc2hvd19yYXRlX3RhYmxlKCkKCiAgICB4cG9zID0g"
           "bnAuYXJhbmdlKGxlbihyYXRlX3RhYmxlKSkKICAgIGZpZywgYXggPSBwbHQuc3Vi"
           "cGxvdHMoZmlnc2l6ZT0oRklHX1cgKiAwLjksIDMuOCkpCiAgICBheC5iYXIoeHBv"
           "cyAtIDAuMiwgcmF0ZV90YWJsZVsicG9zdF9tZWFuXzEwMCJdLCB3aWR0aD0wLjQs"
           "CiAgICAgICAgICAgY29sb3I9Q09MWyLsl7Dtmozsg4kiXSwgbGFiZWw9IuuvuOyF"
           "mCAyIOydmCDsgqztm4Ttj4nqt6AiKQogICAgYXguYmFyKHhwb3MgKyAwLjIsIHJh"
           "dGVfdGFibGVbImdsbV9yYXRlIl0sIHdpZHRoPTAuNCwKICAgICAgICAgICBjb2xv"
           "cj1DT0xbIu2VmOuKmOyDiSJdLCBsYWJlbD0i66+47IWYIDMg7J2YIOyYiOy4oSDs"
           "gqzqs6DsnKgiKQogICAgYXguc2V0X3h0aWNrcyh4cG9zKQogICAgYXguc2V0X3h0"
           "aWNrbGFiZWxzKHJhdGVfdGFibGVbInNlZ21lbnQiXSwgcm90YXRpb249MzAsIGhh"
           "PSJyaWdodCIpCiAgICBheC5zZXRfeGxhYmVsKCLsmpTsnKjtkZzsnZgg7IS46re4"
           "66i87Yq4IikKICAgIGF4LnNldF95bGFiZWwoIuyXsOqwhCDsgqzqs6DsnKggKOqx"
           "tC/rhYQpIikKICAgIGF4LmxlZ2VuZCgpCiAgICBwbHQuc2hvdygpCgrtlbTshJ0K"
           "Ci0g6rOE7JW9IDEy6rG07J24IOyEuOq3uOuovO2KuOyXkOuPhCDrqqjtmJXsnYAg"
           "MC4xMTgg6rG0L+uFhOydhCDsgrDstpztlanri4jri6QuIOydtCDqsJLsnYAg6re4"
           "IDEy6rG07JeQ7IScIOuCmOyYqCDqsJLsnbQg7JWE64uI6528IOqwmeydgCDsp4Ds"
           "l63snZgg6rOE7JW9IOyghOyytOyZgCDqsJnsnYAg67iM656c65Oc7J2YIOqzhOyV"
           "vSDsoITssrTqsIAg7ZWo6ruYIOunjOuToCDqsJLsnoXri4jri6QuIOyngOyXrSDt"
           "mqjqs7zsmYAg67iM656c65OcIO2aqOqzvOulvCDrjZTtlZwg65KkIOyngOyImOul"
           "vCDsoIHsmqntlZwg6rKw6rO87J206riwIOuVjOusuOyeheuLiOuLpC4KLSDruIzr"
           "npzrk5wgQjEyIOydmCDrsLDsnKggMS40NDkxIOydgCDquLDspIAg67iM656c65Oc"
           "IEIxIOqzvCDruYTqtZDtlojsnYQg65WM7J2YIOqwkuyeheuLiOuLpC4g6rCZ7J2A"
           "IOyngOyXrSDslYjsl5DshJwgQjEg7J2YIOyYiOy4oSDsgqzqs6DsnKjsl5Ag7J20"
           "IOqwkuydhCDqs7HtlZjrqbQgQjEyIOydmCDsmIjsuKEg7IKs6rOg7Jyo7J20IOuQ"
           "qeuLiOuLpC4g7KO87Zqo6rO866eMIOuLtOydgCDrqqjtmJXsnbTrnbwg7J20IOuw"
           "sOycqOydgCDrqqjrk6Ag7KeA7Jet7JeQ7IScIOqwmeyKteuLiOuLpC4KLSDqt7jr"
           "przsnZgg66eJ64yAIDLqsJzrpbwg7IS46re466i87Yq466eI64ukIOu5hOq1kO2V"
           "mOuptCDsnbQg7Je07J20IOuvuOyFmCAyIOyZgCDshLHqsqnsnbQg64uk66W4IOqy"
           "g+ydtCDrs7TsnoXri4jri6QuIFI0MyBCMTMg7JeQ7IScIOuvuOyFmCAyIOydmCDs"
           "gqztm4Ttj4nqt6DsnYAgMC4xMDAyNTcg6rG0L+uFhOyduOuNsCDrr7jshZggMyDs"
           "nZgg7JiI7LihIOyCrOqzoOycqOydgCAwLjA5MDE3MSDqsbQv64WE7J6F64uI64uk"
           "LiDrr7jshZggMiDripQg6rOE7JW9IOyghOyytOydmCDqsJIg7ZWY64KY66W8IOuw"
           "mOyYge2WiOqzoCwg66+47IWYIDMg7J2AIOq3uCDshLjqt7jrqLztirjsmYAg7KeA"
           "7JetwrfruIzrnpzrk5zrpbwg6rO17Jyg7ZWY64qUIOqzhOyVveydmCDqsJLsnYQg"
           "67CY7JiB7ZWY6riwIOuVjOusuOyeheuLiOuLpC4K",
           "m4":
           "8J+UkyDrr7jshZggNCDsmYTshLHrs7gg4oCUIOOAjOyXrOq4sOu2gO2EsCBBSSDs"
           "vZTrk5zjgI0g7JWE656Y7JeQIOuTpOyWtOyZgOyVvCDtlZjripQg7L2U65Oc64qU"
           "IOydtCDsoJXrj4TsnoXri4jri6QKCuKWoCDsnbQg7L2U65Oc66W8IOu2gOuluCDt"
           "lITroaztlITtirgg7JiICgogICAg7J2066+4IOunjOuTpOyWtOyguCDsnojripQg"
           "ZGYg7JmAIHJhdGVfdGFibGUsIHNlZ19tYXNrcywgYm9vdF9iLCBzZWVkIOulvCDq"
           "t7jrjIDroZwg7JOw6rOgIOuLpOyLnCDrp4zrk6Tsp4DripQg66eQ7JWEIOykmC4g"
           "c2VnX21hc2tzIOuKlCDshLjqt7jrqLztirgg7J2066aE7J2EIOq3uCDshLjqt7jr"
           "qLztirjsnZgg6rOE7JW97J2EIOqzoOultOuKlAogICAg67aI66as7Ja4IOuwsOyX"
           "tOyXkCDsp53sp4DsnYAg7IKs7KCEIDjqsJzslbwuIOyEuOq3uOuovO2KuOuniOuL"
           "pCDqt7gg6rOE7JW97J2EIOuzteybkOy2lOy2nOuhnCDsm5Ag7ZGc67O46rO8IOqw"
           "meydgCDqsbTsiJjrp4ztgbwgYm9vdF9iIOuyiCDri6Tsi5wg7LaU7Lac7ZWY6rOg"
           "LCDsnqztkZzrs7jrp4jri6Qg7IKs6rOgIOqxtOyImCBDbGFpbU5iIOydmCDtlans"
           "nYQg64W47LacCiAgICBFeHBvc3VyZSDsnZgg7ZWp7Jy866GcIOuCmOuIiCDsgqzq"
           "s6DsnKjsnYQg66qo7JWEIOykmC4g64Kc7IiY64qUIHNlZWQg66GcIOyLnOyeke2V"
           "mOqzoCDrtoTrqqjripQg6rOE7JW9IOyImOqwgCDslYTri4jrnbwg64W47LacIO2V"
           "qeydtOyVvC4g6re4IGJvb3RfYiDqsJzsl5DshJwgMi41JSDsmYAgOTcuNSUg67Cx"
           "67aE7JyE66W8IOyekOuluCDqsJLsnYQKICAgIGJvb3RfbG8g7JmAIGJvb3RfaGkg"
           "7Je066GcIOuNlO2VnCDri6TsnYwgIlIyNCBCMiDsnZgg67aA7Yq47Iqk7Yq4656p"
           "IOq1rOqwhCDquLjsnbQgPSAo6rCSKSDqsbQv64WEIiDqs7wgIlIyMSBCMTEg7J2Y"
           "IOu2gO2KuOyKpO2KuOueqSDqtazqsIQg7ZWY7ZWcID0gKOqwkikg6rG0L+uFhCIg"
           "7J2EIOyGjOyImOygkCDslYTrnpggNuyekOumrOuhnAogICAg7ZWcIO2WieyUqSBw"
           "cmludCDtlZjqs6Agc2hvd19yYXRlX3RhYmxlKCkg66GcIOyngOq4iOq5jOyngCDs"
           "sYTsmrQg7Je07J2EIOuqqOuRkCDri7TsnYAg7ZGc64+EIOy2nOugpe2VtCDspJgu"
           "IOq3uOuemO2UhOuKlCDtlZwg7J6l66eMIOKAlCDshLjqt7jrqLztirjrp4jri6Qg"
           "7ZGc7KSA7Jik7LCoIOq1rOqwhOydmCDquLjsnbTsmYAg67aA7Yq47Iqk7Yq4656p"
           "IOq1rOqwhOydmCDquLjsnbTrpbwg64KY656A7Z6IIOuRlAogICAg66eJ64yAIDE2"
           "6rCc66W8IOq3uOumrOqzoCDrspTroYDsmYAg6rCA66Gc7LaV6rO8IOyEuOuhnOy2"
           "lSDrnbzrsqjsnYQg64us7JWEIOykmC4KCiAgICBybmcgPSBucC5yYW5kb20uZGVm"
           "YXVsdF9ybmcoc2VlZCkKICAgIGJvb3RfbG8sIGJvb3RfaGkgPSBbXSwgW10KICAg"
           "IGZvciBuYW1lIGluIHJhdGVfdGFibGVbInNlZ21lbnQiXToKICAgICAgICBwYXJ0"
           "ID0gZGZbc2VnX21hc2tzW25hbWVdXQogICAgICAgIHkgPSBwYXJ0WyJDbGFpbU5i"
           "Il0udG9fbnVtcHkoZHR5cGU9ZmxvYXQpCiAgICAgICAgZSA9IHBhcnRbIkV4cG9z"
           "dXJlIl0udG9fbnVtcHkoZHR5cGU9ZmxvYXQpCiAgICAgICAgcmF0ZXMgPSBucC5l"
           "bXB0eShib290X2IpCiAgICAgICAgZm9yIGIgaW4gcmFuZ2UoYm9vdF9iKToKICAg"
           "ICAgICAgICAgaWR4ID0gcm5nLmludGVnZXJzKDAsIGxlbih5KSwgbGVuKHkpKSAg"
           "ICAgICMg7J6s7ZGc67O4IO2BrOq4sOuKlCDsm5Ag7ZGc67O46rO8IOqwmeuLpAog"
           "ICAgICAgICAgICByYXRlc1tiXSA9IHlbaWR4XS5zdW0oKSAvIGVbaWR4XS5zdW0o"
           "KSAgICAgIyDrtoTrqqjripQg6rOE7JW9IOyImOqwgCDslYTri4jrnbwg64W47Lac"
           "IO2VqQogICAgICAgIGJvb3RfbG8uYXBwZW5kKGZsb2F0KG5wLnBlcmNlbnRpbGUo"
           "cmF0ZXMsIDIuNSkpKQogICAgICAgIGJvb3RfaGkuYXBwZW5kKGZsb2F0KG5wLnBl"
           "cmNlbnRpbGUocmF0ZXMsIDk3LjUpKSkKICAgIHJhdGVfdGFibGVbImJvb3RfbG8i"
           "XSwgcmF0ZV90YWJsZVsiYm9vdF9oaSJdID0gYm9vdF9sbywgYm9vdF9oaQoKICAg"
           "IGJpZyA9IHJhdGVfdGFibGUuaWxvY1swXQogICAgd2F0Y2ggPSByYXRlX3RhYmxl"
           "LmxvY1tyYXRlX3RhYmxlWyJzZWdtZW50Il0gPT0gd2F0Y2hfbmFtZV0uaWxvY1sw"
           "XQogICAgcHJpbnQoZiJSMjQgQjIg7J2YIOu2gO2KuOyKpO2KuOueqSDqtazqsIQg"
           "6ri47J20ID0ge2JpZ1snYm9vdF9oaSddIC0gYmlnWydib290X2xvJ106LjZmfSDq"
           "sbQv64WEIikKICAgIHByaW50KGYie3dhdGNoX25hbWV9IOydmCDrtoDtirjsiqTt"
           "irjrnqkg6rWs6rCEIO2VmO2VnCA9IHt3YXRjaFsnYm9vdF9sbyddOi42Zn0g6rG0"
           "L+uFhCIpCiAgICBzaG93X3JhdGVfdGFibGUoKQoKICAgIHdpZHRoXzJzZSA9IHJh"
           "dGVfdGFibGVbImhpXzJzZSJdIC0gcmF0ZV90YWJsZVsibG9fMnNlIl0KICAgIHdp"
           "ZHRoX2Jvb3QgPSByYXRlX3RhYmxlWyJib290X2hpIl0gLSByYXRlX3RhYmxlWyJi"
           "b290X2xvIl0KICAgIHhwb3MgPSBucC5hcmFuZ2UobGVuKHJhdGVfdGFibGUpKQog"
           "ICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPShGSUdfVyAqIDAuOSwg"
           "My44KSkKICAgIGF4LmJhcih4cG9zIC0gMC4yLCB3aWR0aF8yc2UsIHdpZHRoPTAu"
           "NCwgY29sb3I9Q09MWyLsl7Dtmozsg4kiXSwgbGFiZWw9Iu2RnOykgOyYpOywqCDq"
           "tazqsIQiKQogICAgYXguYmFyKHhwb3MgKyAwLjIsIHdpZHRoX2Jvb3QsIHdpZHRo"
           "PTAuNCwgY29sb3I9Q09MWyLtlZjripjsg4kiXSwgbGFiZWw9Iuu2gO2KuOyKpO2K"
           "uOueqSDqtazqsIQiKQogICAgYXguc2V0X3h0aWNrcyh4cG9zKQogICAgYXguc2V0"
           "X3h0aWNrbGFiZWxzKHJhdGVfdGFibGVbInNlZ21lbnQiXSwgcm90YXRpb249MzAs"
           "IGhhPSJyaWdodCIpCiAgICBheC5zZXRfeGxhYmVsKCLsmpTsnKjtkZzsnZgg7IS4"
           "6re466i87Yq4IikKICAgIGF4LnNldF95bGFiZWwoIuq1rOqwhCDquLjsnbQgKOqx"
           "tC/rhYQpIikKICAgIGF4LmxlZ2VuZCgpCiAgICBwbHQuc2hvdygpCgrtlbTshJ0K"
           "Ci0g64W47Lac7J20IO2BsCDshLjqt7jrqLztirjsl5DshJzripQg65GQIOq1rOqw"
           "hOydmCDquLjsnbTqsIAg7IKs7Iuk7IOBIOqwmeyKteuLiOuLpC4gMC4wMDYzIOqz"
           "vCAwLjAwNjQg7LKY65+8IOyGjOyImOygkCDslYTrnpggNOyekOumrOyXkOyEnCDr"
           "i6zrnbzsp5Hri4jri6QuIOyCrOqzoOqwgCAzLDE5OeqxtCDriITsoIHrkJwg7IS4"
           "6re466i87Yq47JeQ7ISc64qUIOygleq3nOq3vOyCrOqwgCDqt7jrp4ztgbwg7J6Y"
           "IOunnuuKlOuLpOuKlCDrnLvsnoXri4jri6QuCi0g6rOE7JW9IDEy6rG07J24IOyE"
           "uOq3uOuovO2KuOyXkOyEnOuKlCDrtoDtirjsiqTtirjrnqnsnbQg7J2M7IiY66W8"
           "IOyCsOy2nO2VmOyngCDslYrsirXri4jri6QuIOyerO2RnOuzuCDsgqzqs6DsnKjs"
           "nYAg7Ja47KCc64KYIDAg7J207IOB7J206riwIOuVjOusuOyeheuLiOuLpC4g6re4"
           "IOuMgOyLoCDqtazqsIQg6ri47J2064qUIDAuODIg6rG0L+uFhOycvOuhnCDrr7js"
           "hZggMSDsnZggMC44NSDqsbQv64WE6rO8IOqxsOydmCDqsJnsirXri4jri6QuIOyd"
           "jOyImOqwgCDsgqzrnbzsoYzsnYQg67+QIOygleuwgOuPhOuKlCDrgpjslYTsp4Ds"
           "p4Ag7JWK7JWY7Iq164uI64ukLgotIOq3uOuemOyEnCDrr7jshZggMSDqs7wg66+4"
           "7IWYIDQg64qUIOqwmeydgCDqsrDroaDsl5Ag7J2066aF64uI64ukLiDqt7gg7IS4"
           "6re466i87Yq47J2YIOyekOujjOunjOycvOuhnOuKlCDsmpTsnKjsnYQg7KCV7ZWg"
           "IOyImCDsl4bsirXri4jri6QuIOuvuOyFmCA1IOuKlCDshLjqt7jrqLztirgg67CW"
           "7J2YIOygleuztOulvCDslrzrp4jrgpgg67CY7JiB7ZWg7KeA66W8IOyekOujjOqw"
           "gCDsoJXtlZjqsowg7ZWp64uI64ukLgo=",
           "m5":
           "8J+UkyDrr7jshZggNSDsmYTshLHrs7gg4oCUIOOAjOyXrOq4sOu2gO2EsCBBSSDs"
           "vZTrk5zjgI0g7JWE656Y7JeQIOuTpOyWtOyZgOyVvCDtlZjripQg7L2U65Oc64qU"
           "IOydtCDsoJXrj4TsnoXri4jri6QKCuKWoCDsnbQg7L2U65Oc66W8IOu2gOuluCDt"
           "lITroaztlITtirgg7JiICgogICAg7J2066+4IOunjOuTpOyWtOyguCDsnojripQg"
           "cmF0ZV90YWJsZSDqs7wgbGFtX2FsbCwgcHJpb3JfZXhwb3N1cmUsIGJfc3Ryb25n"
           "LCBhbHBoYV9zdHJvbmcsIHdhdGNoX25hbWUg7J2EIOq3uOuMgOuhnCDsk7Dqs6Ag"
           "64uk7IucIOunjOuTpOyngOuKlCDrp5DslYQg7KSYLiDqsIDspJHsuZggdyDrpbwg"
           "64W47LacIO2VqQogICAgZXhwb3N1cmUg66W8IGV4cG9zdXJlIOyZgCBiX3N0cm9u"
           "ZyDsnZgg7ZWp7Jy866GcIOuCmOuIiCDqsJLsnLzroZwg6rWs7ZWY6rOgLCDrtoDr"
           "toQg7ZKA66eBIOqwkiBwb29sZWRfNDAwIOydhCB3IOqzse2VmOq4sCBtbGVfcmF0"
           "ZSDsl5AgMSDrubzquLAgdyDqs7HtlZjquLAgbGFtX2FsbCDsnYQg642U7ZWcIOqw"
           "kuycvOuhnCDqtaztlbQg7KSYLgogICAg7Iug7Jqp6rWs6rCE7J2AIOqwkOuniOu2"
           "hO2PrOydmCDrtoTsnITsiJjslbwg4oCUIO2Yle2DnCDrqqjsiJjripQgYWxwaGFf"
           "c3Ryb25nIOyXkCBjbGFpbXMg66W8IOuNlO2VnCDqsJIsIOu5hOycqCDrqqjsiJjr"
           "ipQgYl9zdHJvbmcg7JeQIGV4cG9zdXJlIOulvCDrjZTtlZwg6rCS7J206rOgIHNj"
           "aXB5LnN0YXRzLmdhbW1hIOydmCBzY2FsZQogICAg7JeQ64qUIOu5hOycqCDrqqjs"
           "iJjsnZgg7Jet7IiY66W8IOyghOuLrO2VtC4gMi41JSDsmYAgOTcuNSUg67aE7JyE"
           "7IiY66W8IGNyZWRfbG9fNDAwIOqzvCBjcmVkX2hpXzQwMCDsl7TroZwg642U7ZW0"
           "IOykmC4g7IKs7KCEIOqwleuPhCBwcmlvcl9leHBvc3VyZSDroZwg6rCA7KSR7LmY"
           "66W8IO2VnCDrsowg642UIOq1rO2VtCDspJguCiAgICDqt7gg6rCA7KSR7LmY64qU"
           "IOq3uOuemO2UhOyXkOunjCDsk7Dqs6Ag7ZGc7JeQ64qUIOyDiCDsl7TsnYQg66eM"
           "65Ok7KeAIOunkOyVhCDspJguIOyCrOyghCDqsJXrj4QgMTAw64WE7J28IOuVjOyd"
           "mCDrtoDrtoQg7ZKA66eBIOqwkuydgCDrr7jshZggMiDsnZggcG9zdF9tZWFuXzEw"
           "MCDqs7wg6rCZ7J2AIOqwkuydtOuLiOq5jC4KICAgIOq3uOuLpOydjCAiUjIxIEIx"
           "MSDsnZgg67aA67aEIO2SgOungSDqsJIgPSAo6rCSKSDqsbQv64WEIiDqs7wgIlIy"
           "MSBCMTEg7J2YIOyLoOyaqeq1rOqwhCDquLjsnbQgPSAo6rCSKSDqsbQv64WEIiDs"
           "nYQg7IaM7IiY7KCQIOyVhOuemCA27J6Q66as66GcIO2VnCDtlonslKkgcHJpbnQg"
           "7ZWY6rOgCiAgICBzaG93X3JhdGVfdGFibGUoKSDroZwg7KeA6riI6rmM7KeAIOyx"
           "hOyatCDsl7TsnYQg66qo65GQIOuLtOydgCDtkZzrj4Qg7Lac66Cl7ZW0IOykmC4g"
           "6re4656Y7ZSE64qUIO2VnCDsnqXrp4wg4oCUIOqwgOuhnOy2leyXkCDrhbjstpwg"
           "7ZWp7J2EIOuRkOqzoCDroZzqt7gg64iI6riI7J2EIOyggeyaqe2VnCDri6TsnYwg"
           "7IS466Gc7LaV7JeQIOqwgOykkey5mOulvCDrkZQg7IKw7KCQ64+E7JeQIOyCrOyg"
           "hCDqsJXrj4Qg65GQIOqwgOyngOulvCDrqqjrkZAKICAgIOywjeyWtCDsoJAgMTbq"
           "sJzrpbwg66eM65Ok6rOgIOuylOuhgOyZgCDqsIDroZzstpXqs7wg7IS466Gc7LaV"
           "IOudvOuyqOydhCDri6zslYQg7KSYLgoKICAgIHc0MDAgPSByYXRlX3RhYmxlWyJl"
           "eHBvc3VyZSJdIC8gKHJhdGVfdGFibGVbImV4cG9zdXJlIl0gKyBiX3N0cm9uZykK"
           "ICAgIHJhdGVfdGFibGVbInBvb2xlZF80MDAiXSA9IHc0MDAgKiByYXRlX3RhYmxl"
           "WyJtbGVfcmF0ZSJdICsgKDEgLSB3NDAwKSAqIGxhbV9hbGwKICAgIHJhdGVfdGFi"
           "bGVbImNyZWRfbG9fNDAwIl0gPSBzdGF0cy5nYW1tYS5wcGYoCiAgICAgICAgMC4w"
           "MjUsIGE9YWxwaGFfc3Ryb25nICsgcmF0ZV90YWJsZVsiY2xhaW1zIl0sCiAgICAg"
           "ICAgc2NhbGU9MS4wIC8gKGJfc3Ryb25nICsgcmF0ZV90YWJsZVsiZXhwb3N1cmUi"
           "XSkpICAgICAgICAgICAgICAgICAjIHNjYWxlID0g67mE7JyoIOuqqOyImOydmCDs"
           "l63siJgKICAgIHJhdGVfdGFibGVbImNyZWRfaGlfNDAwIl0gPSBzdGF0cy5nYW1t"
           "YS5wcGYoCiAgICAgICAgMC45NzUsIGE9YWxwaGFfc3Ryb25nICsgcmF0ZV90YWJs"
           "ZVsiY2xhaW1zIl0sCiAgICAgICAgc2NhbGU9MS4wIC8gKGJfc3Ryb25nICsgcmF0"
           "ZV90YWJsZVsiZXhwb3N1cmUiXSkpCgogICAgdzEwMCA9IHJhdGVfdGFibGVbImV4"
           "cG9zdXJlIl0gLyAocmF0ZV90YWJsZVsiZXhwb3N1cmUiXSArIHByaW9yX2V4cG9z"
           "dXJlKSAgICMg6re4656Y7ZSE7JeQ66eMIOyUgeuLiOuLpAogICAgIyAgdzEwMCDs"
           "nLzroZwg6rCZ7J2AIOqwgOykke2Pieq3oOydhCDrp4zrk6TrqbQg66+47IWYIDIg"
           "7J2YIHBvc3RfbWVhbl8xMDAg6rO8IOqwkuydtCDqsJnslYQg7IOIIOyXtOydhCDr"
           "kZDsp4Ag7JWK7Iq164uI64ukCgogICAgd2F0Y2ggPSByYXRlX3RhYmxlLmxvY1ty"
           "YXRlX3RhYmxlWyJzZWdtZW50Il0gPT0gd2F0Y2hfbmFtZV0uaWxvY1swXQogICAg"
           "cHJpbnQoZiJ7d2F0Y2hfbmFtZX0g7J2YIOu2gOu2hCDtkoDrp4Eg6rCSID0ge3dh"
           "dGNoWydwb29sZWRfNDAwJ106LjZmfSDqsbQv64WEIikKICAgIHByaW50KGYie3dh"
           "dGNoX25hbWV9IOydmCDsi6DsmqnqtazqsIQg6ri47J20ID0ge3dhdGNoWydjcmVk"
           "X2hpXzQwMCddIC0gd2F0Y2hbJ2NyZWRfbG9fNDAwJ106LjZmfSDqsbQv64WEIikK"
           "ICAgIHNob3dfcmF0ZV90YWJsZSgpCgogICAgZmlnLCBheCA9IHBsdC5zdWJwbG90"
           "cyhmaWdzaXplPShGSUdfVyAqIDAuNywgNC4wKSkKICAgIGF4LnNjYXR0ZXIocmF0"
           "ZV90YWJsZVsiZXhwb3N1cmUiXSwgdzQwMCwgY29sb3I9Q09MWyLtlZjripjsg4ki"
           "XSwgbGFiZWw9IuyCrOyghCDqsJXrj4QgNDAw64WEIikKICAgIGF4LnNjYXR0ZXIo"
           "cmF0ZV90YWJsZVsiZXhwb3N1cmUiXSwgdzEwMCwgY29sb3I9Q09MWyLsl7Dtmozs"
           "g4kiXSwgbGFiZWw9IuyCrOyghCDqsJXrj4QgMTAw64WEIikKICAgIGF4LnNldF94"
           "c2NhbGUoImxvZyIpCiAgICBheC5zZXRfeGxhYmVsKCLshLjqt7jrqLztirjsnZgg"
           "64W47LacIO2VqSAo64WEKSIpCiAgICBheC5zZXRfeWxhYmVsKCLrgrQg7IS46re4"
           "66i87Yq4IOqwkuyXkCDsi6TrpqzripQg6rCA7KSR7LmYIikKICAgIGF4LmxlZ2Vu"
           "ZCgpCiAgICBwbHQuc2hvdygpCgrtlbTshJ0KCi0g7IKs7KCEIOqwleuPhOulvCAx"
           "MDDrhYTsl5DshJwgNDAw64WE7Jy866GcIOuGkuydtOuptCDqtIDsuKHsnbQg7KCB"
           "7J2AIOyEuOq3uOuovO2KuOydmCDqsJLsnbQg6rOE7JW9IOyghOyytOydmCDqsJIg"
           "7Kq97Jy866GcIOuNlCDsiJjstpXtlZjqs6Ag7Iug7Jqp6rWs6rCEIOq4uOydtOqw"
           "gCAwLjEyNyDqsbQv64WE7JeQ7IScIDAuMDYzIOqxtC/rhYTsnLzroZwg7KCI67CY"
           "7J20IOuQqeuLiOuLpC4g6rCA7KSR7LmY66GcIOuztOuptCAwLjA2MiDsl5DshJwg"
           "MC4wMTYg7Jy866GcIOqwkOyGjO2VnCDqsoPsnoXri4jri6QuCi0g7Ken7JWE7KeE"
           "IOq1rOqwhOydgCDsnpDro4zqsIAg64qY7Ja07IScIOyWu+ydgCDqsoPsnbQg7JWE"
           "64uZ64uI64ukLiDsgqzsoITrtoTtj6zrpbwg642UIOqwle2VmOqyjCDqsIDsoJXt"
           "lbTshJwg7Ja77J2AIOqyg+yeheuLiOuLpC4g6re4IOyEuOq3uOuovO2KuOqwgCDs"
           "i6TsoJzroZwg64uk66W4IOyngOyXreqzvCDri6TrpbTri6TrqbQg6re4IOywqOyd"
           "tOuPhCDtlajqu5gg7IKs65287KeR64uI64ukLgotIOuFuOy2nOydtCDtgbAg7IS4"
           "6re466i87Yq47JeQ7ISc64qUIOuRkCDsgqzsoIQg6rCV64+E6rCAIOyCrOyLpOyD"
           "gSDqsJnsnYAg6rCS7J2EIOyCsOy2nO2VqeuLiOuLpC4g6rCA7KSR7LmY6rCAIDAu"
           "OTg5IOyZgCAwLjk5NyDsnbTrnbwg7Ja064qQIOyqveydhCDqs6jrnbzrj4Qg7IaM"
           "7IiY7KCQIOyVhOuemCAz7J6Q66as6rmM7KeAIOqwmeyKteuLiOuLpC4g7IKs7KCE"
           "IOqwleuPhOydmCDshKDtg53snYAg7Ja47KCc64KYIOq0gOy4oeydtCDsoIHsnYAg"
           "7IS46re466i87Yq47JeQ7IScIOusuOygnOqwgCDrkKnri4jri6QuCg==",
           "과제":
           "8J+UkyDqs7zsoJwg7JmE7ISx67O4IOKAlCDssYTsoJDsnpDqsIAg67O064qUIOqy"
           "gwoK4pagIOydtCDsvZTrk5zrpbwg67aA66W4IO2UhOuhrO2UhO2KuCDsmIgKCiAg"
           "ICDspIDruYQg7IWA7J20IOunjOuTpOyWtCDrkZQgcmF0ZV90YWJsZSDqs7wgb3V0"
           "X3BhdGgg66W8IOq3uOuMgOuhnCDsk7Dqs6Ag64uk7IucIOunjOuTpOyngOuKlCDr"
           "p5DslYQg7KSYLiByZXBvcnRfcmF0ZSDsl7Tsl5Ag64K06rCAIOqzoOuluCDrsKnr"
           "spXsnZgg6rCS7J2EIOuzteyCrO2VmOqzoCwgcmVsX3dpZHRoIOyXtOydhCBjcmVk"
           "X2hpXzQwMAogICAg7JeQ7IScIGNyZWRfbG9fNDAwIOydhCDruoAg6rCS7J2EIHBv"
           "b2xlZF80MDAg7Jy866GcIOuCmOuIiCDqsJLsnLzroZwg6rWs7ZW0IOykmC4g64K0"
           "6rCAIOygle2VnCDquLDspIDqsJLsnLzroZwgZGVjaXNpb24g7Je07J2EIOOAjOye"
           "kOuPmSDsirnsnbjjgI0g6rO8IOOAjOyCrOuejCDsi6zsgqzjgI0g66GcIOuCmOuI"
           "iCDrkqQg7ZGc66W8IG91dF9wYXRoIOuhnAogICAg7KCA7J6l7ZWY6rOgIOuLpOyL"
           "nCDsnb3slrQg7ZaJIOyImOulvCDstpzroKXtlZwg64uk7J2MIHNob3dfcmF0ZV90"
           "YWJsZSgpIOuhnCDsmYTshLHrkJwg7ZGc66W8IO2ZlOuptOyXkCDstpzroKXtlbQg"
           "7KSYLiByZXBvcnRfcmF0ZSDqsIAgY3JlZF9sb180MDAg6rO8IGNyZWRfaGlfNDAw"
           "IOyCrOydtOyXkCDsnojripQg7IS46re466i87Yq47J2YIOu5hOycqOuPhCDrsLHr"
           "toTsnKjroZwg7ZWcIO2WiSDstpzroKXtlbQg7KSYLiDqt7jrnpjtlITripQg7ZWc"
           "IOyepeunjCDigJQKICAgIOqwgOuhnOy2leyXkCDrhbjstpwg7ZWp7J2EIOuRkOqz"
           "oCDroZzqt7gg64iI6riI7J2EIOyggeyaqe2VnCDri6TsnYwg7IS466Gc7LaV7JeQ"
           "IHJlbF93aWR0aCDrpbwg65GUIOyCsOygkOuPhOyXkCDsoJAgOOqwnOulvCDssI3q"
           "s6Ag6riw7KSA6rCSIOychOy5mOyXkCDqsIDroZzshKDsnYQg6re47J2AIOuSpCDq"
           "sIDroZzstpXqs7wg7IS466Gc7LaV7JeQIOudvOuyqOydhCDri6zslYQg7KSYLgoK"
           "7LGE7KCQ7J6Q6rCAIOuztOuKlCDqsoMKCi0g6rCSIDLqsJwg4oCUIOyggOyepe2V"
           "nCDtjIzsnbzsnYQg64uk7IucIOydveydgCDtlokg7IiY6rCAIDgg7J206rOgLCDs"
           "mpTsnKjtkZzsl5Ag7KCB7J2AIOqwkuydtCDsi6DsmqnqtazqsIQg7JWI7JeQIOuT"
           "oCDruYTsnKjsnbQgMTAwJSDsnoXri4jri6QuIOyggOyepe2VnCDrkqQg7ZmU66m0"
           "7JeQIOy2nOugpe2VnCDsmYTshLEg7JqU7Jyo7ZGc64qUIOyxhOygkCDtla3rqqns"
           "nbQg7JWE64uI6528IOuCtOqwgCDriIjsnLzroZwg7ZmV7J247ZWY64qUIO2Vreuq"
           "qeyeheuLiOuLpC4g65GQIOuyiOynuCDqsJLsnYAg64K06rCAIOyggeydgCDqsJLq"
           "s7wg64K06rCAIOyggeydgCDqtazqsITsnbQg7ISc66GcIOunnuuKlOyngOulvCDt"
           "mZXsnbjtlZjripQg7ZWt66qp7J6F64uI64ukLgotIOq3uOumvCDtlZwg7J6lIOKA"
           "lCDqsIDroZzstpXsnbQg64W47LacIO2VqeydmCDroZzqt7gg64iI6riI7J206rOg"
           "IOyEuOuhnOy2leydtCDsg4HrjIAg6rWs6rCEIOq4uOydtOyduCDsgrDsoJDrj4Ts"
           "l5Ag7KCQIDjqsJzsnoXri4jri6QuIOuCtOqwgCDsoJXtlZwg6riw7KSA6rCS7J2E"
           "IOuCmO2DgOuCtOuKlCDqsIDroZzshKDsnYAg7JyE7LmY66W8IOuCtOqwgCDsoJXt"
           "lZjrr4DroZwg7LGE7KCQIOuMgOyDgeydtCDslYTri5nri4jri6QuCi0g7ISg7YOd"
           "IDPqsIDsp4Ag4oCUIOyalOycqO2RnOyXkCDsoIHsnYQg6rCSLCDsi6zsgqzrpbwg"
           "6rCA66W8IOq4sOykgOqzvCDquLDspIDqsJIsIOyCrOuejCDsi6zsgqwg7IS46re4"
           "66i87Yq466W8IOyggeuKlCDrsKnrspXsnoXri4jri6QuIOygleuLteydgCDsl4bq"
           "s6Ag67iM66as7ZWR7JeQIOydtOycoOqwgCDsoIHtmIAg7J6I7Jy866m0IOuQqeuL"
           "iOuLpC4KLSDsoJXtlakg7KGw6rG0IOKAlCDstZzrjIDsmrDrj4TstpTsoJXqsJLr"
           "p4wg6rOo6528IOyggeycvOuptCDruYTsnKjsnbQgNjIuNSUg6rCAIOuQqeuLiOuL"
           "pC4g6rSA7Lih7J20IOyggeydgCDshLjqt7jrqLztirggM+qwnOyXkOyEnCDqt7gg"
           "6rCS7J20IOyLoOyaqeq1rOqwhCDrsJbsl5Ag7J6I6riwIOuVjOusuOyeheuLiOuL"
           "pC4g6rCS6rO8IOq1rOqwhOydhCDri6Trpbgg67Cp67KV7JeQ7IScIOqwgOyguOyY"
           "pOuptCDtlZwg7ZaJIOyViOyXkOyEnCDshJzroZwg7Ja06riL64Kc64uk64qUIOqy"
           "g+ydhCDrs7Tsl6wg7KO864qUIO2VreuqqeyeheuLiOuLpC4KLSDruIzrpqztlZEg"
           "M+usuOyepSDigJQg6rKw66Gg6rO8IOq3vOqxsOyZgCDtlZzqs4TqsIAg6rCBIO2V"
           "nCDrrLjsnqXslKkg7J6I6rOgLCDtlZzqs4Qg66y47J6l7JeQIOyCrOyghCDqsJXr"
           "j4QgNDAw64WE7J2YIOy2nOyymOyZgCDshLjqt7jrqLztirjqsIAgOOqwnOu/kOyd"
           "tOudvOuKlCDsoJDsnbQg7KCB7ZiAIOyeiOycvOuptCDrkKnri4jri6QuCg==",
           }

#  힌트는 (한 문장 힌트, 튜터에게 물을 것) 짝입니다 — 채점이 ❌ 인 항목에만 나옵니다
# 항목별 힌트 — base64(JSON)
_HINT = _dec_json(
           "eyJtMSI6IHsiUjIxIEIxMSDsnZgg7LWc64yA7Jqw64+E7LaU7KCV6rCSICjqsbQv"
           "64WEKSI6IFsi64KY64iX7IWI7J2YIOu2hOuqqOqwgCDqs4Tslb0g7IiY6rCAIOyV"
           "hOuLiOudvCDrhbjstpwg7ZWp7J247KeAIO2ZleyduO2VmOyEuOyalC4g6rOE7JW9"
           "IOyImOuhnCDrgpjriITrqbQg64uo7JyE6rCAIOqxtC/qs4Tslb3snbQg65CY7Ja0"
           "IOqwkuydtCDsoIjrsJgg7JWE656Y66GcIOqwkOyGjO2VqeuLiOuLpCIsICLsnbQg"
           "7L2U65Oc6rCAIOyCrOqzoOycqOydmCDrtoTrqqjroZwg7IKs7Jqp7ZWcIOyXtOyd"
           "tCDrrLTsl4fsnbjqsIAiXSwgIu2VmO2VnOydtCDsnYzsiJjsnbgg7IS46re466i8"
           "7Yq4IOyImCAo6rCcKSI6IFsi65GQIOqwgOyngOulvCDssKjroYDroZwg7ZmV7J24"
           "7ZWY7IS47JqULiDtkZzspIDsmKTssKjrpbwg4oiaKOyCrOqzoCDqsbTsiJgg7ZWp"
           "KSDDtyDrhbjstpwg7ZWpIOycvOuhnCDqs4TsgrDtlojripTsp4AsIOq3uOumrOqz"
           "oCDstpTsoJXqsJLsl5DshJwg6re4IOqwkuydmCAy67Cw66W8IOu6kOuKlOyngOye"
           "heuLiOuLpCIsICLsnbQg7L2U65Oc6rCAIO2RnOykgOyYpOywqOulvCDslrTripAg"
           "7Iud7Jy866GcIOqzhOyCsO2WiOqzoCDrqocg67Cw66W8IOu6kOuKlOqwgCJdLCAi"
           "cGxvdCI6IFsi6rCA66GcIOunieuMgCA46rCc7JmAIOuniOy7pCA46rCc6rCAIOye"
           "iOuKlOyngCwg6rCA66Gc7LaV6rO8IOyEuOuhnOy2leyXkCDsnbTrpoTsnbQg7J6I"
           "64qU7KeAIOywqOuhgOuhnCDtmZXsnbjtlZjshLjsmpQuIGxhbV9hbGwg7JyE7LmY"
           "7J2YIOyEuOuhnOyEoOydgCDshKDtg50g7ZWt66qp7J206528IOyxhOygkO2VmOyn"
           "gCDslYrsirXri4jri6QiLCAi64K0IOq3uOumvOydmCDslrTripAg66eJ64yAwrfs"
           "lrTripAg66eI7Luk6rCAIOq4sOykgOqzvCDri6Trpbjsp4AiXSwgImVycm9yIjog"
           "WyLsmKTrpZgg66y47J6lIOyyqyDtlonsnYQg6re464yA66GcIEFJIO2KnO2EsOyX"
           "kCDrs7Tsl6wg7KO86rOgLCDjgIzspIDruYQg7IWA44CN7J20IOunjOuToCDsnbTr"
           "poQoZGYgwrcgcmF0ZV90YWJsZSDCtyBsYW1fYWxsIMK3IHNlZ19uYW1lcyDCtyB3"
           "YXRjaF9uYW1lKeydgCDri6Tsi5wg66eM65Ok7KeAIOunkOqzoCDqt7jrjIDroZwg"
           "7IKs7Jqp7ZWY65286rOgIOuqheyLnO2VmOyEuOyalCIsICLsnbQg7Jik66WY6rCA"
           "IOyWtOuKkCDtlonsl5DshJwg7JmcIOuCrOuKlOyngCJdfSwgIm0yIjogeyJSMjQg"
           "QjIg7J2YIOyCrO2bhO2Pieq3oCAo6rG0L+uFhCkiOiBbWyIx7LCoIOKAlCDrtoTs"
           "npDsl5Ag64yA7J6F7ZWcIOyXtOqzvCDrtoTrqqjsl5Ag64yA7J6F7ZWcIOyXtCDr"
           "kZAg6rOz7J2EIOuztOyEuOyalCIsICIy7LCoIOKAlCDrtoTsnpDripQg7IKs6rOg"
           "IOqxtOyImCDtlakgY2xhaW1zIOydtOqzoCDrtoTrqqjripQg64W47LacIO2VqSBl"
           "eHBvc3VyZSDsnoXri4jri6QuIOqzhOyVvSDsiJggbiDqs7wg7IKs6rOg66W8IOqy"
           "quydgCDqs4Tslb0g7IiYIG5fY2xhaW1fY29udHJhY3RzIOuKlCDrkZgg7Ja065SU"
           "7JeQ64+EIOuTpOyWtOqwgOyngCDslYrsirXri4jri6QiXSwgIuydtCDsvZTrk5zq"
           "sIAg7IKs7ZuE7Y+J6reg7J2YIOu2hOyekOyZgCDrtoTrqqjsl5Ag64yA7J6F7ZWc"
           "IOyXtOydtCDqsIHqsIEg66y07JeH7J246rCAIl0sICJSMjEgQjExIOydmCDsgqzt"
           "m4Ttj4nqt6AgKOqxtC/rhYQpIjogW1siMeywqCDigJQg6rCS7J20IOuRkCDshLjq"
           "t7jrqLztirjsl5DshJwg66qo65GQIOuCruqyjCDrgpjsmZTri6TrqbQg7JuQ7J24"
           "7J2AIO2VnCDqs7PsnoXri4jri6QiLCAiMuywqCDigJQg6rOE7JW9IO2VnCDqsbTs"
           "nZgg64W47Lac7J2AIDHrhYTrs7Tri6Qg7Ken7JWEIOqzhOyVvSDsiJjqsIAg64W4"
           "7LacIO2VqeuztOuLpCDtgb3ri4jri6QuIOu2hOuqqOulvCDqs4Tslb0g7IiY66Gc"
           "IOuMgOyehe2VmOuptCDrqqjrk6Ag7IS46re466i87Yq47J2YIOqwkuydtCDtlajq"
           "u5gg64K066Ck6rCR64uI64ukIl0sICLsnbQg7L2U65Oc7J2YIOu2hOuqqOqwgCDq"
           "s4Tslb0g7IiY7J247KeAIOuFuOy2nCDtlansnbjsp4AiXSwgInBsb3QiOiBbIuyE"
           "uOuhnOy2lSDqsJLsnbQg6rCA66Gc7LaVIOqwkuuztOuLpCDslrzrp4jrgpgg6rCQ"
           "7IaM7ZaI64qU7KeAIO2ZleyduO2VmOyEuOyalC4g6rOg7LmcIOuSpOyXkOuKlCDq"
           "tIDsuKHsnbQg7KCB7J2AIOyEuOq3uOuovO2KuOyXkOyEnCDqsIDroZzstpUg6rCS"
           "6rO8IOyEuOuhnOy2lSDqsJLsnZgg7LCo7J206rCAIO2BrOqyjCDsu6Tsp5Hri4jr"
           "i6QiLCAi64K0IOq3uOumvOydmCDshLjroZzstpUg6rCS7J20IOyWtOuKkCDsl7Ts"
           "l5DshJwg64KY7JioIOqwkuyduOyngCJdLCAiZXJyb3IiOiBbIuyYpOulmCDrrLjs"
           "nqUg7LKrIO2WieydhCDqt7jrjIDroZwgQUkg7Yqc7YSw7JeQIOuztOyXrCDso7zq"
           "s6AsIOOAjOykgOu5hCDshYDjgI3snbQg66eM65OgIOydtOumhChkZiDCtyByYXRl"
           "X3RhYmxlIMK3IGxhbV9hbGwgwrcgcHJpb3JfZXhwb3N1cmUgwrcgYWxwaGFfcHJp"
           "b3Ip7J2AIOuLpOyLnCDrp4zrk6Tsp4Ag66eQ6rOgIOq3uOuMgOuhnCDsgqzsmqnt"
           "lZjrnbzqs6Ag66qF7Iuc7ZWY7IS47JqUIiwgIuydtCDsmKTrpZjqsIAg7Ja064qQ"
           "IO2WieyXkOyEnCDsmZwg64Ks64qU7KeAIl19LCAibTMiOiB7IlIxMSBCMTIg7J2Y"
           "IOyYiOy4oSDsgqzqs6DsnKggKOqxtC/rhYQpIjogWyLsmIjsuKHsnYQg67Cb7J2E"
           "IOuVjCDrhbjstpzsnYQgMeuFhOycvOuhnCDrkZDsl4jripTsp4AsIOyYpO2UhOyF"
           "i+ydhCAw7Jy866GcIOyghOuLrO2WiOuKlOyngCDtmZXsnbjtlZjshLjsmpQuIOuF"
           "uOy2nOydhCDqt7jrjIDroZwg65GQ66m0IOqzhOyVvSDtlZjrgpjsnZgg6riw64yA"
           "IOyCrOqzoCDqsbTsiJjqsIAg64KY7JmAIOqwkuydtCDsnpHslYTsp5Hri4jri6Qi"
           "LCAi7J20IOy9lOuTnOqwgCDsmIjsuKHsnYQg67Cb7J2EIOuVjCDsmKTtlITshYvs"
           "l5Ag66y07JeH7J2EIOyghOuLrO2WiOuKlOqwgCJdLCAi67iM656c65OcIEIxMiDs"
           "nZgg7IKs6rOg7JyoIOuwsOycqCAo67CwKSI6IFsi6rOE7IiY7JeQIOyngOyImOul"
           "vCDsoIHsmqntlojripTsp4AsIOq3uCDqs4TsiJjqsIAg67iM656c65OcIEIxMiDs"
           "nZgg6rKD7J247KeAIOuRkCDqsIDsp4Drpbwg7ZmV7J247ZWY7IS47JqULiDqs4Ts"
           "iJjrpbwgMTAw67CwIO2VmOuKlCDqt7zsgqzripQg6rCS7J20IOuLpOultOqyjCDs"
           "grDstpzrkKnri4jri6QiLCAi7J20IOy9lOuTnOqwgCDrsLDsnKjsnYQg6rOE7IiY"
           "7J2YIOyWtOuKkCDrs4DtmZjsnLzroZwg6rWs7ZaI64qU6rCAIl0sICJwbG90Ijog"
           "WyLshLjqt7jrqLztirjrp4jri6Qg66eJ64yAIDLqsJzslKkg66qo65GQIDE26rCc"
           "7J247KeAIO2ZleyduO2VmOyEuOyalC4g7Jm87Kq97J2AIOuvuOyFmCAyIOydmCBw"
           "b3N0X21lYW5fMTAwIOydtOqzoCDsmKTrpbjsqr3snYAg7J2067KIIOuvuOyFmOyd"
           "mCBnbG1fcmF0ZSDsnoXri4jri6QuIOqwgOuhnOy2leqzvCDshLjroZzstpXsnZgg"
           "7J2066aE64+EIO2VqOq7mCDtmZXsnbjtlZjshLjsmpQiLCAi64K0IOq3uOumvOyd"
           "mCDrp4nrjIAg6rCc7IiY7JmAIOqwgOuhnOy2lcK37IS466Gc7LaVIOydtOumhOyd"
           "tCDquLDspIDqs7wg6rCZ7J2A7KeAIl0sICJlcnJvciI6IFsi7Jik66WYIOusuOye"
           "pSDssqsg7ZaJ7J2EIOq3uOuMgOuhnCBBSSDtipzthLDsl5Ag67O07JesIOyjvOqz"
           "oCwg44CM7KSA67mEIOyFgOOAjeydtCDrp4zrk6Ag7J2066aEKGRmIMK3IHJhdGVf"
           "dGFibGUgwrcgZ2xtX3JlcyDCtyBzZWdfZnJhbWUgwrcgYnJhbmRfcmVmKeydgCDr"
           "i6Tsi5wg66eM65Ok7KeAIOunkOqzoCDqt7jrjIDroZwg7IKs7Jqp7ZWY65286rOg"
           "IOuqheyLnO2VmOyEuOyalCIsICLsnbQg7Jik66WY6rCAIOyWtOuKkCDtlonsl5Ds"
           "hJwg7JmcIOuCrOuKlOyngCJdfSwgIm00IjogeyJSMjQgQjIg7J2YIOu2gO2KuOyK"
           "pO2KuOueqSDqtazqsIQg6ri47J20ICjqsbQv64WEKSI6IFsi7J6s7ZGc67O4IOyC"
           "rOqzoOycqOydmCDrtoTrqqjqsIAg6re4IOyerO2RnOuzuOydmCDrhbjstpwg7ZWp"
           "7J247KeAIOuovOyggCDtmZXsnbjtlZjqs6AsIOq3uOuLpOydjCDtkZzrs7gg7ZWY"
           "64KY66W8IOuqhyDqsbTsnLzroZwg7LaU7Lac7ZaI64qU7KeAIO2ZleyduO2VmOyE"
           "uOyalC4g67aE66qo66W8IOqzhOyVvSDsiJjroZwg65GQ66m0IOq1rOqwhOydtCDs"
           "lYTrnpjroZwg64K066Ck7JWJ6rOgLCDtkZwg7KCE7LK07J2YIO2WiSDsiJjroZwg"
           "7LaU7Lac7ZWY66m0IOq1rOqwhOydtCDsp6fslYTsp5Hri4jri6QiLCAi7J20IOy9"
           "lOuTnOqwgCDsnqztkZzrs7gg7IKs6rOg7Jyo7J2YIOu2hOuqqOyZgCDtkZzrs7gg"
           "7YGs6riw66W8IOqwgeqwgSDrrLTsl4fsnLzroZwg65GQ7JeI64qU6rCAIl0sICJS"
           "MjEgQjExIOydmCDrtoDtirjsiqTtirjrnqkg6rWs6rCEIO2VmO2VnCAo6rG0L+uF"
           "hCkiOiBbIuyerO2RnOuzuCDsgqzqs6DsnKjsnZgg67aE66qo6rCAIOq3uCDsnqzt"
           "kZzrs7jsnZgg64W47LacIO2VqeyduOyngCwg7JWE656Y7Kq9IDIuNSUg7KeA7KCQ"
           "7JeQ7IScIOygiOuLqO2WiOuKlOyngCDtmZXsnbjtlZjshLjsmpQiLCAi7J20IOy9"
           "lOuTnOqwgCDtlZjtlZzsnYQg7Ja064qQIOuwseu2hOychOyXkOyEnCDsoIjri6jt"
           "lojripTqsIAiXSwgInBsb3QiOiBbIuunieuMgOqwgCAxNuqwnOyduOyngCwg7IS4"
           "6re466i87Yq466eI64ukIOuRkCDrsKnrspXsnbQg7Ked7J2EIOydtOujqOuKlOyn"
           "gCDtmZXsnbjtlZjshLjsmpQiLCAi64K0IOq3uOumvOydmCDrp4nrjIDqsIAg7IS4"
           "6re466i87Yq466eI64ukIOuqhyDqsJzsnbjsp4AiXSwgImVycm9yIjogWyLsmKTr"
           "pZgg66y47J6lIOyyqyDtlonsnYQg6re464yA66GcIEFJIO2KnO2EsOyXkCDrs7Ts"
           "l6wg7KO86rOgLCDjgIzspIDruYQg7IWA44CN7J20IOunjOuToCDsnbTrpoQoZGYg"
           "wrcgcmF0ZV90YWJsZSDCtyBzZWdfbWFza3MgwrcgYm9vdF9iIMK3IHNlZWQp7J2A"
           "IOuLpOyLnCDrp4zrk6Tsp4Ag66eQ6rOgIOq3uOuMgOuhnCDsgqzsmqntlZjrnbzq"
           "s6Ag66qF7Iuc7ZWY7IS47JqUIiwgIuydtCDsmKTrpZjqsIAg7Ja064qQIO2WieyX"
           "kOyEnCDsmZwg64Ks64qU7KeAIl19LCAibTUiOiB7IlIyMSBCMTEg7J2YIOu2gOu2"
           "hCDtkoDrp4Eg6rCSICjqsbQv64WEKSI6IFsi6rCA7KSR7LmY7J2YIOu2hOuqqOqw"
           "gCDrhbjstpwg7ZWp6rO8IOyCrOyghCDqsJXrj4TsnZgg7ZWp7J247KeALCDrgpjr"
           "qLjsp4Ag6rCA7KSR7LmY66W8IOqzhOyVvSDsoITssrTsnZgg6rCS7JeQIOyggeya"
           "qe2WiOuKlOyngCDtmZXsnbjtlZjshLjsmpQiLCAi7J20IOy9lOuTnOqwgCDqsIDs"
           "pJHsuZjsnZgg67aE66qo66GcIOyCrOyaqe2VnCDqsJLsnbQg66y07JeH7J246rCA"
           "Il0sICJSMjEgQjExIOydmCDsi6DsmqnqtazqsIQg6ri47J20ICjqsbQv64WEKSI6"
           "IFsi6rCQ66eI67aE7Y+s7J2YIO2Yle2DnCDrqqjsiJjsmYAg67mE7JyoIOuqqOyI"
           "mOulvCDslrTripAg6rCS7Jy866GcIOyghOuLrO2WiOuKlOyngCDtmZXsnbjtlZjs"
           "hLjsmpQuIHNjaXB5LnN0YXRzLmdhbW1hIOuKlCDruYTsnKgg66qo7IiY7J2YIOyX"
           "reyImOulvCBzY2FsZSDroZwg67Cb7Iq164uI64ukIiwgIuydtCDsvZTrk5zqsIAg"
           "6rCQ66eI67aE7Y+s7JeQIOyghOuLrO2VnCDtmJXtg5wg66qo7IiY7JmAIHNjYWxl"
           "IOydtCDqsIHqsIEg66y07JeH7J246rCAIl0sICJwbG90IjogWyLsoJDsnbQgMTbq"
           "sJzsnbjsp4AsIOqwgOuhnOy2leydtCDroZzqt7gg64iI6riI7J247KeAIO2Zleyd"
           "uO2VmOyEuOyalCIsICLrgrQg6re466a87J2YIOygkCDqsJzsiJjsmYAg6rCA66Gc"
           "7LaVIOuIiOq4iOydtCDquLDspIDqs7wg6rCZ7J2A7KeAIl0sICJlcnJvciI6IFsi"
           "7Jik66WYIOusuOyepSDssqsg7ZaJ7J2EIOq3uOuMgOuhnCBBSSDtipzthLDsl5Ag"
           "67O07JesIOyjvOqzoCwg44CM7KSA67mEIOyFgOOAjeydtCDrp4zrk6Ag7J2066aE"
           "KGRmIMK3IHJhdGVfdGFibGUgwrcgbGFtX2FsbCDCtyBwcmlvcl9leHBvc3VyZSDC"
           "tyBiX3N0cm9uZyDCtyBhbHBoYV9zdHJvbmcp7J2AIOuLpOyLnCDrp4zrk6Tsp4Ag"
           "66eQ6rOgIOq3uOuMgOuhnCDsgqzsmqntlZjrnbzqs6Ag66qF7Iuc7ZWY7IS47JqU"
           "IiwgIuydtCDsmKTrpZjqsIAg7Ja064qQIO2WieyXkOyEnCDsmZwg64Ks64qU7KeA"
           "Il19LCAi6rO87KCcIjogeyLsoIDsnqXtlZwg7JqU7Jyo7ZGc7J2YIO2WiSDsiJgg"
           "KO2WiSkiOiBbIuyggOyepe2VnCDtjIzsnbzsnYQg64uk7IucIOydveyWtCDtlons"
           "nZgg6rCc7IiY66W8IOy2nOugpe2WiOuKlOyngCDtmZXsnbjtlZjshLjsmpQuIOyX"
           "tOydmCDqsJzsiJjqsIAg7JWE64uI6528IO2WieydmCDqsJzsiJjsnoXri4jri6Qi"
           "LCAi7J20IOy9lOuTnOqwgCDtjIzsnbzsnYQg64uk7IucIOydveyWtCDrrLTsl4fs"
           "nYQg6rOE7IKw7ZaI64qU6rCAIl0sICLsmpTsnKjtkZwg6rCS7J20IOyLoOyaqeq1"
           "rOqwhCDslYjsl5Ag65OgIOu5hOycqCAoJSkiOiBbIjEwMCXqsIAg7JWE64uI66m0"
           "IHJlcG9ydF9yYXRlIOqwgCBjcmVkX2xvXzQwMCDqs7wgY3JlZF9oaV80MDAg67CW"
           "7Jy866GcIOuCmOqwhCDshLjqt7jrqLztirjqsIAg7J6I64uk64qUIOucu+yeheuL"
           "iOuLpC4g6re4IOyEuOq3uOuovO2KuOulvCDssL7slYQg64uk66W4IOuwqeuyleyd"
           "mCDqsJLsnYQg7KCB7Jy866m0IOuQqeuLiOuLpCIsICLrgrQgcmVwb3J0X3JhdGUg"
           "6rCAIOyLoOyaqeq1rOqwhCDrsJbsnLzroZwg64KY6rCEIOyEuOq3uOuovO2KuOqw"
           "gCDslrTripAg6rKD7J247KeAIl0sICJwbG90IjogWyLsoJDsnbQgOOqwnOyduOyn"
           "gCwg6rCA66Gc7LaV7J20IOuhnOq3uCDriIjquIjsnbjsp4Ag7ZmV7J247ZWY7IS4"
           "7JqULiDqsIDroZzshKDsnYAg7JyE7LmY66W8IOuCtOqwgCDsoJXtlZjrr4DroZwg"
           "7LGE7KCQIOuMgOyDgeydtCDslYTri5nri4jri6QiLCAi64K0IOq3uOumvOydmCDs"
           "oJAg6rCc7IiY7JmAIOqwgOuhnOy2lSDriIjquIjsnbQg6riw7KSA6rO8IOqwmeyd"
           "gOyngCJdLCAiZXJyb3IiOiBbIuyYpOulmCDrrLjsnqUg7LKrIO2WieydhCDqt7jr"
           "jIDroZwgQUkg7Yqc7YSw7JeQIOuztOyXrCDso7zqs6AsIOOAjOykgOu5hCDshYDj"
           "gI3snbQg66eM65OgIOydtOumhChkZiDCtyByYXRlX3RhYmxlIMK3IGxhbV9hbGwg"
           "wrcgYl9zdHJvbmcgwrcgb3V0X3BhdGggwrcgc2VnX25hbWVzKeydgCDri6Tsi5wg"
           "66eM65Ok7KeAIOunkOqzoCDqt7jrjIDroZwg7IKs7Jqp7ZWY65286rOgIOuqheyL"
           "nO2VmOyEuOyalCIsICLsnbQg7Jik66WY6rCAIOyWtOuKkCDtlonsl5DshJwg7Jmc"
           "IOuCrOuKlOyngCJdfX0=")

_ASK = {"m1": "full", "m2": "full", "m3": "full",   # 뒤로 갈수록 걷힙니다 — 과제는 항목만 남습니다
        "m4": "blank", "m5": "blank", "과제": "bare"}
_ASK_HEAD = {"full": "◼ 튜터에 그대로 보낼 질문 (코드는 고치지 말라고 명시합니다)",
             "blank": "◼ 튜터에 보낼 질문 (빈칸은 내가 채웁니다)",
             "bare": "◼ 튜터에 보낼 질문의 항목 (문장으로 만드는 것은 내 몫입니다)",
             "self": "◼ 스스로 점검할 항목 (이 미션은 힌트 없이 내가 명세합니다)"}
_ASK_FORM = {"full": "내 PROMPT 는 「{prompt}」 였는데 「{name}」 채점이 ❌ 였습니다."
                     " 「AI 코드 시작 행」 아래 코드를 보고 「{ask}」 에 3~5문장으로 답해 주세요. 코드는 고치지 마세요.",
             "blank": "내 PROMPT 는 ____ 였는데 「{name}」 채점이 ❌ 였습니다. 「AI 코드 시작 행」 아래 코드에서"
                      " 「{ask}」 부터 짚고, 어느 행을 무엇으로 고칠지 한 문장으로 답해 주세요.",
             "bare": "무엇을 : {ask} / 어디를 : {name} / 고치지 말 것 : 코드",
             "self": "무엇을 : {ask} / 어디를 : {name} / 다시 볼 곳 : 「AI 코드 시작 행」 아래 코드"}

_FIGS = []                                 # 「PROMPT 셀」이 만든 그림을 모으는 자리
_STATE = {"tag": None, "buf": None, "out": None, "ids0": set()}
_TRY = {}                                  # 미션마다 ❌ 가 몇 번째인지 — 결정 16 이 이 수를 봅니다
_orig_figure = getattr(plt.figure, "_orig", plt.figure)   # 이 셀을 다시 실행해도 겹겹이 싸이지 않게


def _figure(*a, **k):
    f = _orig_figure(*a, **k)
    _FIGS.append(f)
    return f


_figure._orig = _orig_figure
plt.figure = _figure                       # subplots · pandas · seaborn 도 이 경로를 거칩니다


def _get(name):
    """이름이 살아 있으면 그 값, 없으면 None — 아직 만들지 않은 이름에도 채점이 죽지 않게"""
    return globals().get(name)


def _open_figs():
    """지금 열려 있는 그림들 — 래퍼를 거치지 않고 읽습니다"""
    try:
        return [plt._pylab_helpers.Gcf.figs[n].canvas.figure for n in plt.get_fignums()]
    except Exception:
        return []


def _nums(text):
    """출력 텍스트에서 숫자만 뽑습니다 (천 단위 쉼표 제거, 지수 표기 포함)"""
    picked = []
    for seg in re.findall(r"-?\d[\d,]*\.?\d*(?:[eE][-+]?\d+)?", text or ""):
        try:
            picked.append(float(seg.replace(",", "")))
        except ValueError:
            continue
    return picked


def _value_lines(txt, units):
    """값 채점이 볼 행만 고릅니다 — 단위가 붙은 행, 또는 「이름 = 값」 꼴의 행.

    표를 통째로 띄우면 그 안에 정답이 우연히 섞여 있어, 「값을 골라 단위와 함께 출력한다」 는
    계약을 지키지 않고도 ✅ 가 열립니다. 그래서 단위를 달아 고른 행이 하나라도 있으면 그
    행들만 보고, 그런 행이 아예 없을 때만 출력 전체를 봅니다.
    """
    picked = []
    for line in (txt or "").splitlines():
        if not line.strip():
            continue
        if "=" in line or any(unit and unit in line for unit in units):
            picked.append(line)
    return picked


def _label_lines(val_lines, label):
    """라벨이 든 값줄만 골라, 그 줄에서 라벨 뒤(「=」·「:」 뒤) 토막을 돌려줍니다.

    값줄의 숫자를 한 주머니에 담아 띠와 대조하면 **옆 칸의 값**이 이 칸을 헛통과시킵니다 —
    부호 칸 둘이 같은 띠일 때 한쪽에 찍힌 1 이 다른 칸까지 열어 주던 자리입니다. 그래서
    라벨이 든 줄이 있으면 그 줄의 라벨 뒤만 봅니다(없으면 부르는 쪽이 값줄 전체로 물러납니다).
    라벨 찾기는 정확히 같은 문자열 → 못 찾으면 공백·괄호를 걷어낸 꼴, 두 단입니다.
    """
    def _tighten(txt):                     # 공백과 괄호(안의 말까지) 를 걷어낸 글자 · 원래 자리
        keep, depth = [], 0
        for i, ch in enumerate(txt or ""):
            if ch in "([":
                depth += 1
            elif ch in ")]":
                depth = max(0, depth - 1)
            elif depth == 0 and not ch.isspace():
                keep.append(i)
        return "".join((txt or "")[i] for i in keep), keep

    tight_label = _tighten(label)[0]
    tails = []
    for line in val_lines:
        end = line.find(label) + len(label) if label and label in line else -1
        if end < 0 and tight_label:                       # 2단 — 공백·괄호를 걷어내고 다시 봅니다
            tight_line, keep = _tighten(line)
            at = tight_line.find(tight_label)
            end = (keep[at + len(tight_label) - 1] + 1) if at >= 0 else -1
        if end < 0:
            continue
        tail = line[end:]
        cut_at = [p for p in (tail.find("="), tail.find(":")) if p >= 0]
        tails.append(tail[min(cut_at) + 1:] if cut_at else tail)
    return tails


def _axes(fig):
    """Figure · Axes 어느 쪽을 받아도 축 목록으로 폅니다"""
    if fig is None:
        return []
    axs = getattr(fig, "axes", None)
    if isinstance(axs, list):
        return axs
    return [fig] if hasattr(fig, "get_xlabel") else []


def _seg_lines(ax):
    """`vlines` · `hlines` 로 그은 선 — Line2D 가 아니라 **LineCollection** 이라 `ax.lines` 에 없습니다.

    토막 하나를 `(가로값, 세로값)` 한 쌍으로 펴서 돌려줍니다. 이 경로를 열어 두지 않으면
    `plt.hlines(...)` 로 화면에 분명히 그어 놓은 구간 막대가 「막대가 하나도 없다」 로 채점돼,
    학생이 고칠 곳 없는 결함을 고치러 가게 됩니다.
    """
    picked = []
    for c in getattr(ax, "collections", []):
        get_segs = getattr(c, "get_segments", None)      # LineCollection 에만 있는 메서드입니다
        if get_segs is None:                              # 산점도(PathCollection)·채움 영역(PolyCollection)은 선이 아닙니다
            continue
        try:
            segs = get_segs()
        except Exception:
            continue
        for seg in segs:
            xy = np.asarray(seg, dtype=float).reshape(-1, 2)
            if xy.shape[0] >= 2:
                picked.append((xy[:, 0], xy[:, 1]))
    return picked


def _lines(ax):
    """축의 선을 곡선(점 10개 이상)과 수직선(x 가 같은 두 점)으로 갈라 돌려줍니다"""
    cur, vline = [], []
    cands = []
    for ln in ax.lines:                     # 선 하나가 깨져도 나머지는 살린다 — `axvline(x=날짜)` 처럼 x 가
        try:                                #  Timestamp 인 세로선(add_changepoints_to_plot)은 날짜를 숫자로 바꿔 받는다
            xd = np.asarray(ln.get_xdata(), dtype=float).reshape(-1)
            yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
        except (TypeError, ValueError):
            try:
                import matplotlib.dates as _mdates
                xd = np.asarray(_mdates.date2num(ln.get_xdata()), dtype=float).reshape(-1)
                yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
            except Exception:
                continue
        cands.append((xd, yd))
    cands += _seg_lines(ax)                     # `plt.vlines` 로 그은 수직선도 여기서 합류합니다
    for xd, yd in cands:
        if xd.size >= 10:
            cur.append((xd, yd))
        elif xd.size == 2 and np.isclose(xd[0], xd[1]):
            vline.append(float(xd[0]))
    return cur, vline


def _marker_lines(ax):
    """마커를 찍은 Line2D — `plot(x, y, "o")` 든 `plot(x, y, marker="o")` 든 점으로 셉니다.

    선으로 이으면서 점도 함께 찍은 선(`linestyle` 이 기본 실선)까지 세는 것은, 점추정을 그렇게
    찍은 그림이 「점추정 마커가 0개다」 로 ❌ 나던 자리이기 때문입니다. 점 계수는 linestyle 과
    무관하고, 마커를 달지 않은 선만 점에서 뺍니다.
    """
    picked = []
    for ln in ax.lines:
        marker = str(ln.get_marker()).strip()
        if marker in ("", "None", "none"):     # 마커 없는 선은 점이 아닙니다 — 선 모양은 보지 않습니다
            continue
        if marker in ("|", "_"):               # errorbar 의 capsize 캡은 점이 아닙니다 — 구간 끝을 그은 짧은 선입니다
            continue
        try:                                # 날짜 x 인 선 하나가 나머지 점까지 죽이지 않게 갈라 받는다
            xd = np.asarray(ln.get_xdata(), dtype=float).reshape(-1)
            yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
        except (TypeError, ValueError):
            try:
                import matplotlib.dates as _mdates
                xd = np.asarray(_mdates.date2num(ln.get_xdata()), dtype=float).reshape(-1)
                yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
            except Exception:
                continue
        if xd.size and xd.size == yd.size:
            picked.append((ln, xd, yd))
    return picked


def _pt_sets(ax):
    """산점도의 점 좌표 — `scatter`(PathCollection)와 `plot(x, y, "o")` 를 함께 셉니다"""
    marker = _marker_lines(ax)
    pt_sets = [np.asarray(c.get_offsets(), dtype=float).reshape(-1, 2)
          for c in ax.collections if not hasattr(c, "get_segments")]   # 선 모음은 점으로 세지 않습니다
    pt_sets += [np.column_stack([xd, yd]) for _, xd, yd in marker]
    return marker, pt_sets


def _spans(ax):
    """가로로 누운 구간 막대를 (왼쪽 끝, 오른쪽 끝) 짝으로 모읍니다.

    `hlines`(LineCollection) · `plot([lo, hi], [y, y])`(Line2D) · `errorbar(xerr=…)` ·
    `barh`(Rectangle) 어느 방법으로 그렸어도 같은 자리에서 셉니다 — 그리는 방법을 하나로
    못 박으면 AI 가 고른 다른 방법이 「막대가 없다」 로 읽혀 학생이 애먼 곳을 고칩니다.
    세로로 선 토막(수직 기준선)은 구간이 아니라 여기서 빠집니다.
    """
    picked = []
    cands = [(np.asarray(ln.get_xdata(), dtype=float).reshape(-1),
           np.asarray(ln.get_ydata(), dtype=float).reshape(-1)) for ln in ax.lines]
    cands += _seg_lines(ax)
    for xd, yd in cands:
        if xd.size != 2 or yd.size != 2:
            continue
        if not (np.isfinite(xd).all() and np.isfinite(yd).all()):
            continue
        if not np.isclose(yd[0], yd[1]) or float(xd.max() - xd.min()) <= 0:
            continue                                   # 가로로 누워 있고 길이가 있는 토막만
        picked.append((float(xd.min()), float(xd.max())))
    for p in getattr(ax, "patches", []):               # `barh(left=lo, width=hi-lo)` 로 그린 구간
        try:
            x0, bar_w, heights = float(p.get_x()), float(p.get_width()), float(p.get_height())
        except (AttributeError, TypeError, ValueError):
            continue
        if bar_w > 0 and heights > 0 and np.isfinite(x0):
            picked.append((x0, x0 + bar_w))
    return picked


def _near(val, ref, factor=1.5):
    """실측값이 기준의 배수 안에 드는가 — 눈금·표본 차이는 넘어갑니다"""
    try:
        val = float(val)
    except (TypeError, ValueError):
        return False
    if ref == 0:
        return abs(val) < 1e-9
    return val * ref > 0 and abs(ref) / factor <= abs(val) <= abs(ref) * factor


def _diagonal(ax, skip=(), xy=None):
    """y = x 인 기준선이 점이 놓인 범위를 가로지르는가 — 선이 있기만 하면 통과가 아닙니다"""
    drop_ids = {id(ln) for ln in skip}
    cands = [(np.asarray(ln.get_xdata(), dtype=float).reshape(-1),
           np.asarray(ln.get_ydata(), dtype=float).reshape(-1))
          for ln in ax.lines if id(ln) not in drop_ids]
    cands += _seg_lines(ax)                     # LineCollection 으로 그은 y = x 기준선도 셉니다
    for xd, yd in cands:
        if xd.size < 2 or xd.size != yd.size:
            continue
        if not (np.isfinite(xd).all() and np.isfinite(yd).all()):
            continue
        width = float(xd.max() - xd.min())
        if width <= 0:                                    # 수직선은 대각선이 아닙니다
            continue
        if not np.allclose(yd, xd, rtol=1e-3, atol=0.02 * width):
            continue                                   # 세로가 가로를 따라가지 않으면 넘어갑니다
        if xy is not None and getattr(xy, "size", 0):
            pt_span = float(xy[:, 0].max() - xy[:, 0].min())
            overlap = float(min(xd.max(), xy[:, 0].max()) - max(xd.min(), xy[:, 0].min()))
            if pt_span > 0 and overlap < 0.4 * pt_span:          # 점이 놓인 구간을 거의 덮지 못하는 토막 선
                continue
        return True
    return False


def _plot_gaps(ax, spec):
    """축 하나를 기준과 대조해 어긋난 곳을 문장 목록으로 돌려줍니다 (빈 목록이면 통과)"""
    gaps = []
    if spec.get("kind") == "line":
        cur, vline = _lines(ax)
        want = spec.get("n_series", len(cur))
        if len(cur) != want:
            gaps.append(f"곡선이 {len(cur)}개다 (기준 {want}개)")
        if cur:
            xd, yd = max(cur, key=lambda t: t[0].size)
            lo, hi = spec.get("x_range", (xd.min(), xd.max()))
            slack = 0.2 * (hi - lo)
            if not (xd.min() <= lo + slack and xd.max() >= hi - slack):
                gaps.append(f"가로축이 {xd.min():.3g}~{xd.max():.3g} 만 포함한다 (기준 {lo:g}~{hi:g})")
            if "peak_x" in spec:
                p0, p1 = spec["peak_x"]
                peak = float(xd[yd.argmax()])
                trough = float(xd[yd.argmin()])
                if not (p0 <= peak <= p1 or p0 <= trough <= p1):
                    gaps.append(f"곡선의 최댓값 위치가 {peak:.4f}, 최솟값 위치가 {trough:.4f} 다"
                                f" (기준 {p0}~{p1} — 둘 중 하나가 이 구간 안에 있어야 합니다)")
        if cur and ("봉우리_x" in spec or "골짜기_x" in spec):
            # 부호가 뜻을 가진 곡선(충격반응) — 위로 솟는 곡선과 아래로 꺼지는 곡선을 따로 봅니다.
            # 반응을 통째로 뒤집어 그리면 봉우리·골짜기의 자리는 그대로라 자리만 봐서는 걸러지지 않습니다
            if "봉우리_x" in spec:
                a0, a1 = spec["봉우리_x"]
                if not any(float(np.nanmax(y)) > 0 and a0 <= float(x[int(np.nanargmax(y))]) <= a1
                           for x, y in cur):
                    gaps.append(f"위로 솟아 {a0:g}~{a1:g} 시간 구간에서 최댓값에 이르는 곡선이 없다"
                                " (부호가 뒤집혔으면 반응을 반대 방향으로 읽은 것입니다)")
            if "골짜기_x" in spec:
                b0, b1 = spec["골짜기_x"]
                if not any(float(np.nanmin(y)) < 0 and b0 <= float(x[int(np.nanargmin(y))]) <= b1
                           for x, y in cur):
                    gaps.append(f"아래로 꺼져 {b0:g}~{b1:g} 시간 구간에서 최솟값에 이르는 곡선이 없다"
                                " (부호가 뒤집혔으면 반응을 반대 방향으로 읽은 것입니다)")
    elif spec.get("kind") == "bar":
        heights = [float(p.get_height()) for p in ax.patches]
        if "n" in spec and len(heights) != spec["n"]:
            gaps.append(f"막대가 {len(heights)}개다 (기준 {spec['n']}개)")
        if heights and "ymax" in spec and not (spec["ymax"][0] <= max(heights) <= spec["ymax"][1]):
            gaps.append(f"가장 높은 막대가 {max(heights):.4f} 다"
                        f" (기준 {spec['ymax'][0]:g}~{spec['ymax'][1]:g})")
    elif spec.get("kind") == "interval":
        spans = _spans(ax)
        if "n_spans" in spec:
            n_lo, n_hi = spec["n_spans"]
            if not (n_lo <= len(spans) <= n_hi):
                gaps.append(f"가로 구간 막대가 {len(spans)}개다 (기준 {n_lo}개 — 항목 하나에 막대"
                            f" 하나, 강조로 덧그린 막대는 {n_hi - n_lo}개까지 넘어갑니다)")
        if spans:
            widths = [right - left for left, right in spans]
            left_end, right_end = min(left for left, _ in spans), max(right for _, right in spans)
            if "span_max" in spec:
                lo, hi = spec["span_max"]
                if not (lo <= max(widths) <= hi):
                    gaps.append(f"가장 넓은 막대의 폭이 {max(widths):.4f} 다 (기준 {lo:g}~{hi:g})")
            if "span_min" in spec:
                lo, hi = spec["span_min"]
                if not (lo <= min(widths) <= hi):
                    gaps.append(f"가장 좁은 막대의 폭이 {min(widths):.4f} 다 (기준 {lo:g}~{hi:g})")
            if "x_range" in spec:
                lo, hi = spec["x_range"]
                if not (_near(left_end, lo) and _near(right_end, hi)):
                    gaps.append(f"막대가 차지하는 가로축 값이 {left_end:.4g}~{right_end:.4g} 다"
                                f" (기준 {lo:g}~{hi:g})")
        if "n_points" in spec:
            _, pt_sets = _pt_sets(ax)
            n_pts = int(sum(p.shape[0] for p in pt_sets))
            n_lo, n_hi = spec["n_points"]
            if not (n_lo <= n_pts <= n_hi):
                gaps.append(f"점추정 마커가 {n_pts}개다 (기준 {n_lo}개 — 항목 하나에 마커 하나)")
        if "vline_x" in spec:
            _, vline = _lines(ax)
            v0, v1 = spec["vline_x"]
            if not any(v0 <= v <= v1 for v in vline):
                gaps.append(f"기준 구간({v0:g} ~ {v1:g}) 안에 세로선이 없다"
                            + (f" (지금 {vline[0]:.4f})" if vline else " (세로선이 아예 없다)"))
    elif spec.get("kind") == "scatter":
        marker, pt_sets = _pt_sets(ax)
        n_pts = int(sum(p.shape[0] for p in pt_sets))
        if "n_points" in spec:
            cell = spec["n_points"]
            n_lo, n_hi = cell if isinstance(cell, tuple) else (cell, cell)
            if not (n_lo <= n_pts <= n_hi):
                gaps.append(f"점이 {n_pts}개다 (기준 {n_lo}개 — 항목 하나에 점 하나,"
                            f" 강조로 덧찍는 점은 {n_hi - n_lo}개까지 넘어갑니다)")
        xy = np.vstack([p for p in pt_sets if p.size]) if any(p.size for p in pt_sets) else None
        if xy is not None:
            xy = xy[np.isfinite(xy).all(axis=1)]
        if xy is not None and xy.size:
            xs, ys = xy[:, 0], xy[:, 1]
            if "x_range" in spec:
                lo, hi = spec["x_range"]
                if not (_near(xs.min(), lo) and _near(xs.max(), hi)):
                    gaps.append(f"가로축 값이 {xs.min():.3g}~{xs.max():.3g} 다 (기준 {lo:g}~{hi:g})")
            if "y_range" in spec:
                lo, hi = spec["y_range"]
                if not (_near(ys.min(), lo) and _near(ys.max(), hi)):
                    gaps.append(f"세로축 값이 {ys.min():.4g}~{ys.max():.4g} 다 (기준 {lo:g}~{hi:g})")
            if "ymax" in spec and not (spec["ymax"][0] <= ys.max() <= spec["ymax"][1]):
                gaps.append(f"세로축 최댓값이 {ys.max():.4g} 다"
                            f" (기준 {spec['ymax'][0]:g}~{spec['ymax'][1]:g})")
            if "gap_min" in spec:
                spread = float(np.abs(ys - xs).max())
                if spread < spec["gap_min"]:
                    gaps.append(f"두 축의 값이 가장 크게 벌어진 곳도 {spread:.4f} 뿐이다"
                                f" (기준 {spec['gap_min']:g} 이상)")
        if spec.get("xscale") and ax.get_xscale() != spec["xscale"]:
            gaps.append(f"가로축이 {ax.get_xscale()} 눈금이다 (기준 {spec['xscale']} 눈금)")
        if spec.get("diag") and not _diagonal(ax, skip=[ln for ln, _, _ in marker], xy=xy):
            gaps.append("두 축의 값이 같은 점을 잇는 대각선(y = x)이 점이 놓인 구간에 없다")
    if spec.get("축라벨") and not (ax.get_xlabel().strip() and ax.get_ylabel().strip()):
        gaps.append("가로축·세로축 라벨 중 빈 것이 있다")
    return gaps


_REF = {}      # 참조 그림은 두지 않습니다 — 미션마다 완성본 풀이가 그 자리를 대신합니다


def _show(fig):
    """채점 훅 안에서 그림을 화면에 올립니다 (인라인 백엔드는 이 시점에 plt.show() 가 듣지 않습니다)"""
    try:
        from IPython.display import display
        display(fig)
    except Exception:
        plt.show()


def _ref_show(fig, key):
    """내 그림과 참조 그림을 나란히 — 내 그림이 없으면 참조 그림도 띄우지 않습니다"""
    if not _axes(fig) or key not in _REF:
        return
    ref = None
    try:
        ref = _REF[key]()
        if ref is None:
            return
        panels = [(fig, "내 그림"), (ref, "참조 그림")]
        f = _orig_figure(figsize=(FIG_W, 3.4))
        for i, (g, title) in enumerate(panels):
            a = f.add_subplot(1, len(panels), i + 1)
            buf = io.BytesIO()
            (g if hasattr(g, "savefig") else g.figure).savefig(buf, format="png", dpi=110,
                                                               bbox_inches="tight")
            buf.seek(0)
            a.imshow(plt.imread(buf))
            a.axis("off")
            a.set_title(title, fontsize=11)
        _show(f)
        plt.close(f)
    except Exception:
        print("ⓘ 참조 그림을 나란히 놓지 못했습니다 — 위 채점 결과는 그대로 유효합니다")
    finally:
        if ref is not None:
            plt.close(ref)


def _one_line(e, n=120):
    """예외를 한 문장으로 — 트레이스백 대신 학생이 읽을 문장 하나만 남깁니다"""
    txt = str(e).replace("\n", " ").strip()
    return (txt[:n] + "…") if len(txt) > n else txt



def _prompt1(n=70):
    """지금 셀의 지시 첫 행 — 튜터에 보낼 질문에 그대로 싣습니다"""
    head_lines = [l.strip() for l in str(_get("PROMPT") or "").split("\n") if l.strip()]
    return (head_lines[0][:n] + "…") if head_lines else "____"


_PROMPT_MIN = 30
_PROMPT_PLACEHOLDERS = ("", "목표", "쓸 변수", "출력", "목표 —", "쓸 변수 —", "출력 —", "TODO",
                    "여기에 내 말로 적습니다")


def _prompt_missing():
    """지시 입력란이 아직 내 말로 채워지지 않았으면 사유 한 문장, 채워졌으면 None"""
    try:
        txt = str(_ip.user_ns.get("PROMPT", _get("PROMPT")) or "").strip()
    except Exception:
        txt = str(_get("PROMPT") or "").strip()
    if not txt:
        return "PROMPT 입력란이 비어 있습니다"
    if len(txt) < _PROMPT_MIN:
        return f"PROMPT 입력란이 {len(txt)}자뿐입니다 (최소 {_PROMPT_MIN}자)"
    stripped = txt.replace("—", "").replace("·", "").strip()
    if stripped in _PROMPT_PLACEHOLDERS:
        return "PROMPT 입력란이 자리표시자 그대로입니다"
    return None


def _reveal_show(txt):
    """완성본 코드와 해설 — **평문 그대로** 찍는다.

    LMS 렌더러 계약: 「🔓 정답이 열렸습니다 …」 행부터 채점 표를 닫는 `─` 구분선 행까지를
    출력 패널이 「🔓 완성본 풀이 보기」 상자로 접는다. 접힘의 표지가 이 **평문 마커**라
    HTML `details` 로 싣지 않는다 — 마커도 이 함수도 바꾸지 않는다.
    """
    print(txt)


def judge(key, text="", figs=(), error=None):
    """출력 텍스트와 그림을 기준과 대조해 채점 표를 찍습니다 — 통과하면 완성본 풀이를 엽니다"""
    spec = _ANS.get(key)
    if spec is None:                       # 기준이 아직 없는 미션은 조용히 넘어갑니다
        return None
    head_label = ("미션 " + key[1:]) if key[:1] == "m" and key[1:].isdigit() else key
    failed, passed = [], True
    print("\n" + "─" * 54)
    print(f"◼ {head_label} 채점 결과")
    reason = _prompt_missing()                     # 값·그래프보다 먼저 — 내 말로 시킨 자리가 있는가
    _locked = bool(reason)
    if reason and key != "m2":
        print(f"❌ {reason} — 짧게 시켜도 됩니다. 다만 무엇을 어떤 단위로 출력할지까지 적으면 한 번에 통과할 확률이 높습니다")
        print("   PROMPT 입력란을 채우기 전에는 값·그래프 채점도, 완성본 풀이도 열지 않습니다")
        print("─" * 54)
        return False
    elif reason:                                   # 함정 미션만은 전임자 코드의 진단이 먼저다
        print(f"ⓘ {reason} — 아래 채점 결과는 전임 담당자 코드의 진단입니다."
              " 고칠 지시를 PROMPT 에 쓰세요")
    if error is not None:                   # 오류가 나면 값·그래프 채점은 뜻이 없습니다
        print(f"❌ 실행 오류 — {type(error).__name__}: {_one_line(error)}")
        hint, question = _HINT.get(key, {}).get("error", ("", "이 오류가 어느 행에서 왜 났는지"))
        if hint:
            print(f"   힌트 : {hint}")
        form = _ASK.get(key, "full")
        print(_ASK_HEAD[form])
        print("  " + _ASK_FORM[form].format(name="실행 오류", prompt=_prompt1(), ask=question))
        print("─" * 54)
        return False
    txt = text or ""
    if not txt.strip():
        print("❌ 출력이 없습니다 — 값을 print 하도록 프롬프트의 출력 형식에 적으세요")
    units = [str(ref[2]) for ref in spec.get("values", {}).values() if len(ref) > 2]
    val_lines = _value_lines(txt, units)                    # 단위·「=」 가 붙은 행이 있으면 그 행만 봅니다
    nums = _nums("\n".join(val_lines)) if val_lines else _nums(txt)
    table_flood = not val_lines and len(nums) > 30        # 골라낸 자리 없이 숫자만 잔뜩 = 표를 통째로 띄운 모양
    if table_flood:
        print(f"❌ 값 — 단위도 「이름 = 값」 도 없이 숫자가 {len(nums)}개 나왔습니다 — 표를"
              " 통째로 출력하면 어느 숫자가 답인지 채점 코드가 판별하지 못합니다."
              " 값은 「이름 = 숫자 단위」 한 행으로 출력하세요")
        nums = []
    for label, ref in spec.get("values", {}).items():
        lo, hi = float(ref[0]), float(ref[1])
        unit = ref[2] if len(ref) > 2 else ""
        tail = _label_lines(val_lines, label) if nums else []   # 이 라벨이 든 행의 「=」 뒤만 — 옆 칸과 안 섞이게
        fell_back = bool(nums) and not tail             # 라벨이 든 행이 없을 때만 값줄 전체로 물러납니다
        picked = _tail_numbers(tail) if tail else nums   # 꼬리는 괄호를 걷어낸 첫 숫자 하나만 봅니다
        hits = [v for v in picked if lo <= v <= hi]
        if hits:
            print(f"✅ {label} — 출력에서 {hits[0]:,.6g} 를 찾았습니다")
            if unit and unit not in txt:
                print(f"   ⓘ 단위 「{unit}」 도 값과 함께 출력하면 보고서에 그대로 옮길 수 있습니다")
        elif table_flood:
            passed = False
            failed.append((label, label))
            print(f"❌ {label} — 「이름 = 숫자 {unit}」 한 행으로 골라 출력하면 채점 코드가 읽습니다")
        else:
            passed = False
            failed.append((label, label))
            _prev_fail = _TRY.get(key, 0)              # 결정 16 — 첫 ❌ 에는 기준 범위를 감춘다
            print(f"❌ {label} — 기준에 닿는 숫자를 출력에서 찾지 못했습니다"
                  + (f" (기준 범위 {lo:g} ~ {hi:g})" if _prev_fail else ""))
            if fell_back:                            # 라벨 행을 못 찾아 값줄 전체를 본 자리
                print(f"   ⓘ 라벨 「{label}」 이 든 행을 찾지 못해 값줄 전체에서 찾았습니다"
                      " — 「이름 = 숫자 단위」 한 행으로 출력하면 그 행만 보고 채점합니다")
    plot_spec = spec.get("plot")
    if plot_spec:
        axs = [a for f in figs for a in _axes(f)]
        if not axs:
            passed = False
            failed.append(("그래프", "plot"))
            print("❌ 그래프 — 그림을 찾지 못했습니다 (그래프 한 장도 같은 프롬프트에 함께 시킵니다)")
        else:
            gaps = None
            for ax in axs:                 # 패널이 여럿이면 기준에 맞는 축이 하나만 있어도 통과입니다
                try:
                    this_gaps = _plot_gaps(ax, plot_spec)
                except Exception as e:
                    this_gaps = [f"그림에서 데이터를 뽑다 걸렸습니다 ({type(e).__name__})"]
                if not this_gaps:
                    gaps = []
                    break
                if gaps is None or len(this_gaps) < len(gaps):
                    gaps = this_gaps
            if gaps:
                passed = False
                failed.append(("그래프", "plot"))
                print("❌ 그래프 — " + " · ".join(gaps))
            else:
                print("✅ 그래프 — 기준대로 그렸습니다")
            _ref_show(figs[0] if figs else None, key)
    tbl = _HINT.get(key, {})
    if failed:
        _TRY[key] = _TRY.get(key, 0) + 1     # 결정 16 — 다음 ❌ 부터 기준 범위를 인쇄합니다
    for name, hkey in failed:
        hint = tbl.get(hkey, ("",))[0]
        first_tier = (hint,) if isinstance(hint, str) else tuple(hint)   # 함정 미션은 1차·2차로 나뉩니다
        for tiers, txt in enumerate(first_tier, 1):
            if txt:
                tail = f" {tiers}단" if len(first_tier) > 1 else ""
                print(f"   힌트({name}){tail} : {txt}")
    if failed:
        form = _ASK.get(key, "full")
        print(_ASK_HEAD[form])
        for name, hkey in failed:
            question = (tbl.get(hkey, ("", ""))[1] if len(tbl.get(hkey, ("",))) > 1 else "") \
                or "이 항목이 왜 ❌ 로 남는지"
            print("  " + _ASK_FORM[form].format(name=name, prompt=_prompt1(), ask=question))
    elif _locked:                                  # 값·그림은 닿았지만 내 말로 시킨 자리가 아직 비었습니다
        print("ⓘ 값과 그림은 기준에 닿았습니다 — PROMPT 에 고칠 지시를 내 말로 쓰고"
              " 다시 실행하면 완성본 풀이가 열립니다")
    else:
        print("🔓 정답이 열렸습니다 — 아래 「🔓 완성본 풀이 보기」를 펼쳐 내 코드와 견줘 보세요")
        _reveal_show(base64.b64decode(_REVEAL[key]).decode("utf-8"))
    print("─" * 54)
    return passed and not _locked


class _Tee:
    """「PROMPT 셀」의 출력을 화면과 버퍼 양쪽으로 흘려보냅니다"""

    def __init__(self, orig, buf):
        self._o, self._b = orig, buf

    def write(self, s):
        self._b.write(s)
        return self._o.write(s)

    def flush(self):
        self._o.flush()

    def __getattr__(self, n):
        return getattr(self._o, n)


def _tag(src):
    """첫 행이 `# 미션 N — 프롬프트`·`# 과제 — 프롬프트` 인 셀만 채점 대상입니다"""
    head_line = (src or "").split("\n")[0]
    m = re.match(r"\s*#\s*(미션\s*(\d+)|과제)\b", head_line)
    if not m or "프롬프트" not in head_line:                           # 「준비 셀」(`# 미션 N · 준비`)은 대상이 아닙니다
        return None
    return "m" + m.group(2) if m.group(2) else "과제"


def _pre(info=None):
    """「PROMPT 셀」이면 출력 버퍼를 걸고 그림 목록을 비웁니다"""
    try:
        tag = _tag(getattr(info, "raw_cell", ""))
        _STATE["tag"] = tag
        if tag is None or tag not in _ANS:
            return
        _FIGS.clear()
        _STATE["ids0"] = {id(f) for f in _open_figs()}
        _STATE["buf"], _STATE["out"] = io.StringIO(), sys.stdout
        sys.stdout = _Tee(sys.stdout, _STATE["buf"])
    except Exception:
        pass


def _post(result=None):
    """「PROMPT 셀」이 끝나면 출력을 되돌리고 그 셀의 출력·그림으로 채점합니다"""
    tag, buf, out = _STATE.get("tag"), _STATE.get("buf"), _STATE.get("out")
    _STATE["tag"], _STATE["buf"], _STATE["out"] = None, None, None
    if out is not None:
        sys.stdout = out
    if tag is None or tag not in _ANS:
        return
    seen = {id(f) for f in _FIGS}
    figs_all = list(_FIGS) + [f for f in _open_figs()
                         if id(f) not in seen and id(f) not in _STATE.get("ids0", set())]
    txt = buf.getvalue() if buf is not None else ""
    last_val = getattr(result, "result", None)       # 셀 마지막 표현식의 값 — print 없이 표로 띄운 값도 읽습니다
    if last_val is not None:
        try:
            txt = txt + "\n" + str(last_val)
        except Exception:
            pass
    try:
        judge(tag, txt, figs_all, error=getattr(result, "error_in_exec", None))
    except Exception as e:
        print(f"(채점을 마치지 못했습니다 — {type(e).__name__}: {_one_line(e)})")


try:
    _ip = get_ipython()
    for _ev, _fn in (("pre_run_cell", _pre), ("post_run_cell", _post)):
        for _old in list(_ip.events.callbacks[_ev]):
            if getattr(_old, "__name__", "") == _fn.__name__:
                _ip.events.unregister(_ev, _old)     # 이 셀을 다시 실행해도 채점은 한 번만 찍힙니다
        _ip.events.register(_ev, _fn)
    print("채점 준비 완료 — 「PROMPT 셀」을 실행하면 실행 결과(출력)에 채점 결과가 찍힙니다")
except Exception:
    print("(IPython 밖입니다 — 훅 없이 judge(태그, 출력, 그림들) 만 준비했습니다)")

## 세그먼트별 최대우도추정값과 표준오차 구간

### 미션 1 — 세그먼트 8개의 최대우도추정값과 표준오차 구간

#### 할 일 — 미션 1 (20분)

1. 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 아래 만들 것대로 「PROMPT 셀」의 PROMPT 에 지시를 적습니다.
3. 셀을 클릭한 채 AI 튜터에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶. 채점 결과가 ❌ 면 힌트대로 PROMPT 를 고쳐 다시 지시합니다.

**이 미션의 질문** — 세그먼트 8개의 연간 사고율을 각각 한 숫자로 적으면 그 숫자는 어디부터 어디까지입니까?

첫날에 우도를 가장 크게 만드는 값이 사고 건수 합을 노출 합으로 나눈 값이 된다는 것을 확인했습니다. 그 값 옆에 표준오차를 적어 구간으로 넓히는 것은 넷째 날의 내용입니다. 표준오차의 정의와 유도는 넷째 날 이론의 「표준오차 — 추정값의 표본 간 변동」에서 다시 확인할 수 있고, 여기서는 √(사고 건수 합) ÷ 노출 합 이라는 형태만 사용합니다.

#### 만들 것

「PROMPT 셀」 한 곳에서 아래가 모두 출력되게 합니다.

1. **R21 B11 의 최대우도추정값** — 단위 건/년 · 소수점 아래 6자리 · 그 세그먼트의 사고 건수 합을 노출 합으로 나눈 값
2. **하한이 음수인 세그먼트 수** — 단위 개 · 정수 · 추정값에서 표준오차의 2배를 뺀 값이 0보다 작은 세그먼트의 개수
3. **함께 출력할 표 하나** — `rate_table` 에 `mle_rate` · `se` · `lo_2se` · `hi_2se` 열 4개를 더하고 `show_rate_table()` 로 8행을 그대로 출력합니다. 채점 대상은 아닙니다
4. **그래프 한 장** — 가로축은 연간 사고율(건/년), 세로축은 세그먼트 이름 8개인 구간 그림
   - 가로 막대 : 세그먼트마다 하한에서 상한까지 하나씩, 모두 8개. `hlines` 나 `barh` 로 긋습니다
   - 마커 : 세그먼트마다 추정값 위치에 하나씩. `errorbar` 로 그릴 때는 `capsize` 를 두지 않습니다 — 캡이 마커로 집계되어 마커가 8개를 넘습니다
   - 가로축 이름 : 「연간 사고율 (건/년)」 · 세로축 이름 : 「요율표의 세그먼트」
   - 세로선(선택) : `lam_all` 위치에 세로축 전체를 가로지르는 선 하나. 채점 대상은 아닙니다

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Generic_forest_plot.png" width="640" height="413"/>
<p align="center"><em>▲ 추정값에 구간을 함께 표시해 여러 항목을 세로로 배치한 그림 — 가로 막대가 각 항목의 구간, 가운데 표식이 추정값, 세로 점선이 비교 기준값이다. 미션 1 에서 그릴 그림도 이 형태다 (출처: Wikipedia, Generic_forest_plot.png)</em></p>

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Standard_deviation_diagram.svg" width="400" height="200"/>
<p align="center"><em>▲ 추정값을 중심으로 좌우가 대칭인 정규근사 구간 — 추정값 ± 2 × 표준오차 는 이 대칭을 전제하므로, 관측이 적어 실제 분포가 한쪽으로 치우친 세그먼트에서는 하한이 0보다 작아진다 (출처: Wikipedia, Standard_deviation_diagram.svg)</em></p>

**출력 형식** — 값 2개는 「이름 = 값 단위」 형태로 한 행에 하나씩 `print` 합니다. 표 전체를 출력하는 것만으로는 어느 숫자가 채점 대상인지 판별되지 않습니다. 채점은 값이 기준 범위 안에 들면 통과하므로 소수점 자리까지 같을 필요는 없습니다.

**쓸 수 있는 것** — 「준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 |
|---|---|
| `df` | 전처리가 끝난 계약 표 |
| `rate_table` | 세그먼트 8개의 계약 수 · 사고 건수 · 노출을 담은 표 |
| `lam_all` | 계약 전체의 연간 사고율, 단위 건/년 |
| `seg_names` | 세그먼트 이름 8개 |
| `watch_name` | 노출이 가장 적은 세그먼트의 이름 |
| `show_rate_table` | 지금까지 채운 열을 모두 담아 8행을 출력하는 함수 |

#### 프롬프트에 넣을 것(힌트)

- **고정할 이름** — `df` 와 `rate_table` 은 이미 있으니 다시 만들지 말라고 적습니다.
- **계산 방법** — 추정값의 분모는 노출 합, 표준오차는 √(사고 건수 합) ÷ 노출 합.
- **출력** — 값 2개의 이름과 단위, 그리고 소수점 아래 몇 자리까지 출력할지.
- **그래프** — 가로 막대와 마커의 위치, 가로축과 세로축의 이름.
- **내가 직접 확인할 것** — 표에 더한 열 4개와 `lam_all` 위치의 세로선.

#### 실행 전 예상 — 계약 12건인 세그먼트의 하한

이 표에서 계약이 가장 적은 세그먼트는 계약 12건에 사고 2건입니다. 추정값에서 표준오차의 2배를 뺀 결과가 어느 범위에 위치할지 하나를 고르고, 왜 그렇게 판단했는지 한 문장 적으세요.

> **[예측] 계약 12건인 R21 B11 의 구간 하한은 어떤 값이 될까요?**
>
> 1. 0보다 크고 계약 전체의 값보다 작다
> 2. 0보다 작다
> 3. 정확히 0이다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

In [ ]:
# 미션 1 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 1 에 쓸 변수를 직접 만듭니다

if "df" not in globals():                                       # 이 미션만 따로 실행해도 단독으로 동작합니다
    from sklearn.datasets import fetch_openml                    # (앞 미션을 이미 실행했으면 통째로 건너뜁니다)
    df = fetch_openml(data_id=41214, as_frame=True).frame        # freMTPL2freq — 프랑스 자동차보험 계약별 사고 빈도
    df["IDpol"] = df["IDpol"].astype(int)
    for c in ["Area", "VehBrand", "VehGas", "Region"]:
        df[c] = df[c].astype(str).str.strip("'").astype("category")
    df["ClaimNb"] = df["ClaimNb"].astype(int).clip(upper=4)      # 5일 동안 고정해 온 전처리 2가지
    df["Exposure"] = df["Exposure"].clip(upper=1.0)

lam_all = df["ClaimNb"].sum() / df["Exposure"].sum()             # 계약 전체의 연간 사고율 (건/년)
watch_name = "R21 B11"                                           # 노출이 가장 적은 세그먼트

_grp = df.groupby(["Region", "VehBrand"], observed=True).agg(    # 지역 x 브랜드 242개
    n=("ClaimNb", "size"), claims=("ClaimNb", "sum"), exposure=("Exposure", "sum"))
_grp["n_claim_contracts"] = (df["ClaimNb"] > 0).astype(int).groupby(
    [df["Region"], df["VehBrand"]], observed=True).sum()
_picked = pd.concat([_grp.sort_values("exposure", ascending=False).head(4),      # 노출이 큰 4개
                     _grp[_grp["claims"] >= 1].sort_values("exposure").head(4)])  # 사고가 있는 작은 4개
seg_names = [f"{r} {b}" for r, b in _picked.index]
seg_region = [r for r, _ in _picked.index]
seg_brand = [b for _, b in _picked.index]
rate_table = _picked.reset_index(drop=True)[
    ["n", "claims", "exposure", "n_claim_contracts"]].astype(
        {"n": int, "claims": int, "n_claim_contracts": int}).copy()
rate_table.insert(0, "segment", seg_names)


def show_rate_table(*cols):
    """지금까지 채운 열을 모두 담아 세그먼트 8행을 그대로 출력합니다 (인자는 넣어도 됩니다)"""
    keep = ["segment", "n", "claims", "exposure"]
    keep += [c for c in rate_table.columns
             if c not in keep and c != "n_claim_contracts"]          # 앞 미션의 열까지 함께
    keep += [c for c in cols if c in rate_table.columns and c not in keep]
    print(rate_table[keep].round(4).to_string(index=False))

print(f"세그먼트 {len(rate_table)}개 — 계약 {len(df):,}건 가운데 지역 x 브랜드 242개에서 고른 8개입니다")
print(rate_table[["segment", "n", "claims", "exposure"]].round(2).to_string(index=False))
print(f"계약 전체의 연간 사고율 lam_all = {lam_all:.6f} 건/년")

print("쓸 수 있는 이름 : df (전처리가 끝난 계약 표) · rate_table (세그먼트 8개의 요율표) · lam_all (계약 전체의 연간 사고율) · "
      "seg_names (세그먼트 이름 8개) · watch_name (노출이 가장 적은 세그먼트) · "
      "show_rate_table (지금까지 채운 열을 모두 출력하는 함수)")

In [ ]:
# 미션 1 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → AI 튜터에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

> **[문제] 미션 1 의 표에서 계약 수가 가장 많은 세그먼트와 가장 적은 세그먼트의 구간 길이를 이만큼 다르게 만든 것은 무엇입니까?**
>
> 1. 계약 수 — 계약이 많을수록 구간이 짧아지므로 계약 수가 구간 길이를 정한다
> 2. 노출 합 — 노출이 길수록 구간이 짧아지므로 노출 합이 구간 길이를 정한다
> 3. 누적된 사고 건수 — 상대 구간 길이가 사고 건수 합의 제곱근에 반비례하므로 사고가 몇 건 누적되었는지가 구간 길이를 정한다

## 감마-포아송 켤레로 얻는 사후평균

### 미션 2 — 감마-포아송 켤레로 얻는 사후평균 (함정 미션)

#### 할 일 — 미션 2 (20분)

1. 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 「PROMPT 셀」을 **먼저 그대로 ▶** 실행해 전임 담당자 코드의 채점 결과(❌)를 확인합니다.
3. 아래 고칠 것대로 「PROMPT 셀」의 PROMPT 에 지시를 적습니다.
4. 셀을 클릭한 채 AI 튜터에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶. 채점 결과가 ❌ 면 힌트대로 PROMPT 를 고쳐 다시 지시합니다.

**앞 미션에서 받는 것** — 미션 1 의 `lo_2se` 열에서 R21 B11 의 하한이 -0.125 건/년이고, R43 B4 · R43 B13 · R21 B13 까지 4개의 하한이 0보다 작습니다. 사고율은 음수가 될 수 없으니 이 4개는 미션 1 의 열만으로 요율을 적을 수 없습니다. 미션 2 는 여기에 계약 전체의 값을 얼마나 반영할지 정합니다.

**이 미션의 질문** — 관측이 적어 구간의 하한이 음수가 된 세그먼트에 계약 전체의 값을 얼마나 반영해야 합니까?

둘째 날에 감마 사전분포와 포아송 관측이 켤레라서 적분 없이 덧셈만으로 사후분포를 구할 수 있다는 것을 배웠습니다. 그날 쓴 **사전 강도**는 사전분포가 가상 노출 몇 년에 해당하는지를 나타내는 수이고, 미션 2 는 그 값을 100년으로 정합니다. 유도는 둘째 날 이론의 「켤레 — 적분 없이 덧셈으로」에 있고, 100년은 같은 권 「실무 적용 — 자동차보험 데이터」 위젯의 `사전 강도 β₀` 기본값입니다.

**이 미션은 함정입니다.** 「PROMPT 셀」의 전임 담당자 코드는 실행은 되지만 답이 틀립니다.

#### 고칠 것

「PROMPT 셀」 한 곳에서 아래가 모두 출력되게 합니다.

1. **R24 B2 의 사후평균** — 단위 건/년 · 소수점 아래 6자리
2. **R21 B11 의 사후평균** — 단위 건/년 · 소수점 아래 6자리
3. **함께 출력할 표 하나** — `rate_table` 에 `post_mean_100` 열을 더하고 `show_rate_table()` 로 8행을 그대로 출력합니다. 미션 1 의 열도 이 표에 함께 나옵니다. 채점 대상은 아닙니다
4. **그래프 한 장** — 가로축은 미션 1 의 최대우도추정값, 세로축은 사후평균인 산점도이고 점은 8개입니다
   - 가로축 이름 : 「최대우도추정값 (건/년)」
   - 세로축 이름 : 「사후평균 (건/년)」

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Gamma_distribution_pdf.svg" width="480" height="360"/>
<p align="center"><em>▲ 형태 모수와 비율 모수에 따라 모양이 달라지는 감마분포 — 사고율처럼 0 이상인 값의 사전분포로 쓰이고, 포아송 관측과 결합하면 사후분포도 다시 감마분포가 된다 (출처: Wikipedia, Gamma_distribution_pdf.svg)</em></p>

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Prior,_Likelihood,_Posterior_schematic.svg" width="600" height="200"/>
<p align="center"><em>▲ 사전분포와 우도가 결합해 사후분포가 되는 관계 — 관측이 적을수록 사후분포는 사전분포 쪽에, 관측이 많을수록 우도 쪽에 가까워진다 (출처: Wikipedia, Prior,_Likelihood,_Posterior_schematic.svg)</em></p>

**출력 형식** — 값 2개는 「이름 = 값 단위」 형태로 한 행에 하나씩 `print` 합니다. 채점은 값이 기준 범위 안에 들면 통과합니다.

**오늘 쓰는 사전분포** — 사후평균의 분자는 형태 모수 `alpha_prior` 에 사고 건수 합을 더한 값이고, 분모는 `prior_exposure` 에 노출 합을 더한 값입니다.

**쓸 수 있는 것** — 「준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 |
|---|---|
| `df` | 전처리가 끝난 계약 표 |
| `rate_table` | 미션 1 의 열 4개까지 채워진 요율표 |
| `n_claim_contracts` | `rate_table` 의 열 — 사고를 1건 이상 겪은 계약 수이고 사고 건수 합 `claims` 와 다른 수입니다 |
| `lam_all` | 계약 전체의 연간 사고율, 단위 건/년 |
| `prior_exposure` | 사전 강도, 가상 노출 100년 |
| `alpha_prior` | 감마 사전분포의 형태 모수 |
| `show_rate_table` | 지금까지 채운 열을 모두 담아 8행을 출력하는 함수 |

#### 프롬프트에 넣을 것(힌트)

- **고칠 위치** — 사후평균 한 행에서 분자와 분모에 대입한 열 두 곳만 고치면 됩니다.
- **다루는 값** — 계약 수와 사고를 겪은 계약 수는 분자와 분모 어디에도 대입하지 않습니다.
- **출력** — 값 2개의 이름과 단위, 그리고 소수점 아래 몇 자리까지 출력할지.
- **그래프** — 고친 뒤의 사후평균으로 산점도를 다시 그리라고 적습니다.
- **내가 직접 확인할 것** — `post_mean_100` 이 `mle_rate` 에서 계약 전체의 값 0.1006 건/년 쪽으로 옮겨 갔는지.

#### 실행 전 예상 — 사전 강도 100년의 효과

미션 1 의 추정값 0.3 건/년 부근에 사전 강도 100년을 반영하면 값이 어디로 이동할지 고르고, 왜 그렇게 판단했는지 한 문장 적으세요.

> **[예측] 사전 강도 100년을 반영하면 R21 B11 의 값은 미션 1 의 0.3 건/년 부근에서 어디로 이동할까요?**
>
> 1. 거의 그대로 남는다
> 2. 계약 전체의 값 부근까지 내려온다
> 3. 0에 가깝게 내려간다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

In [ ]:
# 미션 2 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 2 에 쓸 변수를 직접 만듭니다

if "df" not in globals():                                       # 이 미션만 따로 실행해도 단독으로 동작합니다
    from sklearn.datasets import fetch_openml                    # (앞 미션을 이미 실행했으면 통째로 건너뜁니다)
    df = fetch_openml(data_id=41214, as_frame=True).frame        # freMTPL2freq — 프랑스 자동차보험 계약별 사고 빈도
    df["IDpol"] = df["IDpol"].astype(int)
    for c in ["Area", "VehBrand", "VehGas", "Region"]:
        df[c] = df[c].astype(str).str.strip("'").astype("category")
    df["ClaimNb"] = df["ClaimNb"].astype(int).clip(upper=4)      # 5일 동안 고정해 온 전처리 2가지
    df["Exposure"] = df["Exposure"].clip(upper=1.0)

lam_all = df["ClaimNb"].sum() / df["Exposure"].sum()             # 계약 전체의 연간 사고율 (건/년)
watch_name = "R21 B11"                                           # 노출이 가장 적은 세그먼트

_grp = df.groupby(["Region", "VehBrand"], observed=True).agg(    # 지역 x 브랜드 242개
    n=("ClaimNb", "size"), claims=("ClaimNb", "sum"), exposure=("Exposure", "sum"))
_grp["n_claim_contracts"] = (df["ClaimNb"] > 0).astype(int).groupby(
    [df["Region"], df["VehBrand"]], observed=True).sum()
_picked = pd.concat([_grp.sort_values("exposure", ascending=False).head(4),      # 노출이 큰 4개
                     _grp[_grp["claims"] >= 1].sort_values("exposure").head(4)])  # 사고가 있는 작은 4개
seg_names = [f"{r} {b}" for r, b in _picked.index]
seg_region = [r for r, _ in _picked.index]
seg_brand = [b for _, b in _picked.index]
rate_table = _picked.reset_index(drop=True)[
    ["n", "claims", "exposure", "n_claim_contracts"]].astype(
        {"n": int, "claims": int, "n_claim_contracts": int}).copy()
rate_table.insert(0, "segment", seg_names)


def show_rate_table(*cols):
    """지금까지 채운 열을 모두 담아 세그먼트 8행을 그대로 출력합니다 (인자는 넣어도 됩니다)"""
    keep = ["segment", "n", "claims", "exposure"]
    keep += [c for c in rate_table.columns
             if c not in keep and c != "n_claim_contracts"]          # 앞 미션의 열까지 함께
    keep += [c for c in cols if c in rate_table.columns and c not in keep]
    print(rate_table[keep].round(4).to_string(index=False))

rate_table["mle_rate"] = rate_table["claims"] / rate_table["exposure"]
rate_table["se"] = np.sqrt(rate_table["claims"]) / rate_table["exposure"]
rate_table["lo_2se"] = rate_table["mle_rate"] - 2 * rate_table["se"]
rate_table["hi_2se"] = rate_table["mle_rate"] + 2 * rate_table["se"]

print(f"세그먼트 {len(rate_table)}개 — 계약 {len(df):,}건 가운데 지역 x 브랜드 242개에서 고른 8개입니다")
print(rate_table[["segment", "n", "claims", "exposure"]].round(2).to_string(index=False))
print(f"계약 전체의 연간 사고율 lam_all = {lam_all:.6f} 건/년")

prior_exposure = 100.0                                            # 사전 강도 — 가상 노출 100년에 해당합니다
alpha_prior = lam_all * prior_exposure                           # 사전평균이 계약 전체의 값과 같아지는 형태 모수
print(f"사전 강도 prior_exposure = {prior_exposure:.0f} 년 · 형태 모수 alpha_prior = {alpha_prior:.4f}")
print(rate_table[["segment", "n", "claims", "n_claim_contracts"]].to_string(index=False))
print("n_claim_contracts 는 사고를 1건 이상 겪은 계약 수입니다 — "
      "한 계약에서 사고가 2건 이상 난 경우가 있어 사고 건수 합 claims 와 다릅니다")

print("쓸 수 있는 이름 : df (전처리가 끝난 계약 표) · rate_table (미션 1 의 열 4개까지 채워진 요율표) · lam_all (계약 전체의 연간 사고율) · "
      "prior_exposure (사전 강도 100년) · alpha_prior (감마 사전분포의 형태 모수) · seg_names (세그먼트 이름 8개) · "
      "show_rate_table (지금까지 채운 열을 모두 출력하는 함수)")

In [ ]:
# 미션 2 — 프롬프트 (함정)   # ai: prompt
# 이 미션은 함정입니다 — 아래는 전임 담당자가 남긴 코드라 실행은 되지만 답이 틀립니다.
# 어디를 어떻게 고칠지 PROMPT 에 쓰면 튜터가 아래 코드 자리를 다시 씁니다.
PROMPT = """

"""
# --- 여기부터 AI 코드 ---
# ↓ 전임 담당자의 코드 — 먼저 그대로 실행해 어느 값이 기준에 닿지 않는지 확인하세요
prior_exposure = 100.0
alpha_prior = lam_all * prior_exposure
rate_table["post_mean_100"] = (
    (alpha_prior + rate_table["n_claim_contracts"]) / (prior_exposure + rate_table["n"]))
for name in ("R24 B2", "R21 B11"):
    row = rate_table.loc[rate_table["segment"] == name].iloc[0]
    print(f"{name} 의 사후평균 = {row['post_mean_100']:.6f} 건/년")
show_rate_table()
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(rate_table["mle_rate"], rate_table["post_mean_100"], color=COL["하늘색"])
ax.set_xlabel("최대우도추정값 (건/년)")
ax.set_ylabel("사후평균 (건/년)")
plt.show()

> **[문제] 전임 담당자 코드가 낸 값이 실제 사후평균보다 낮게 나온 까닭은 무엇입니까?**
>
> 1. 사전 강도를 너무 크게 정해서 모든 세그먼트가 계약 전체의 값 쪽으로 더 크게 수축했다
> 2. 분자와 분모를 모두 다른 값으로 바꿔 대입했다 — 분자에 사고를 겪은 계약 수를, 분모에 계약 수를 대입했다
> 3. 소수점 자리를 잘못 반올림해 값이 작아졌다

## 로그 노출을 오프셋으로 반영한 포아송 회귀의 예측 사고율

### 미션 3 — 지역과 브랜드를 한 모형에 담은 예측 사고율

#### 할 일 — 미션 3 (20분)

1. 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 아래 만들 것대로 「PROMPT 셀」의 PROMPT 에 지시를 적습니다.
3. 셀을 클릭한 채 AI 튜터에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶. 채점 결과가 ❌ 면 힌트대로 PROMPT 를 고쳐 다시 지시합니다.

**앞 미션에서 받는 것** — 미션 2 가 채운 `post_mean_100` 열에서 R43 B13 의 값은 0.1003 건/년이고, 계약 전체의 연간 사고율 0.1006 건/년과 소수점 아래 3자리까지 같습니다. 사고가 1건뿐이라 사전 강도 100년이 값을 거의 다 정했고, 그래서 이 세그먼트만의 사정이 표에 남지 않았습니다. 미션 3 은 계약 전체 하나 대신 같은 지역과 같은 브랜드를 쓰는 계약을 함께 반영해 이 세그먼트의 값을 다시 정하고, 그림은 두 열을 세그먼트마다 나란히 둡니다.

**이 미션의 질문** — 세그먼트마다 따로 계산하지 않고 지역과 브랜드를 한 모형에 담으면 예측 사고율은 얼마입니까?

셋째 날에 계약마다 노출이 다르므로 로그 노출을 오프셋으로 둔다는 것과, 계수에 지수를 적용하면 사고율 배율이 된다는 것을 배웠습니다. 오프셋은 계수를 추정하지 않고 1로 고정해 **선형예측자**에 더하는 항입니다. 선형예측자는 지수를 적용하기 전에 계수와 변수를 곱해 모두 더한 합입니다. 
#### 만들 것

「PROMPT 셀」 한 곳에서 아래가 모두 출력되게 합니다.

1. **R11 B12 의 예측 사고율** — 단위 건/년 · 소수점 아래 6자리 · 노출 1년일 때의 예측값
2. **브랜드 B12 의 사고율 배율** — 단위 배 · 소수점 아래 4자리 · 기준 브랜드 B1 대비 값이고 계수에 지수를 적용해 구합니다
3. **함께 출력할 표 하나** — `rate_table` 에 `glm_rate` 열을 더하고 `show_rate_table()` 로 8행을 그대로 출력합니다. 미션 2 까지의 열도 이 표에 함께 나옵니다. 채점 대상은 아닙니다
4. **그래프 한 장** — 세그먼트 8개의 값을 두 열로 비교하는 막대 16개
   - 세그먼트마다 미션 2 의 `post_mean_100` 과 `glm_rate` 를 나란히 배치합니다
   - 가로축 이름 : 「요율표의 세그먼트」 · 세로축 이름 : 「연간 사고율 (건/년)」
5. **배율을 읽는 문장 한 행** — 「B1 과 비교하면 B12 가 몇 배입니다」 형태로 출력합니다. 채점 대상은 아닙니다

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Regression_de_Poisson.svg" width="480" height="360"/>
<p align="center"><em>▲ 포아송 회귀가 건수 자료에 맞추는 곡선 — 예측값이 언제나 0 이상이고 계수의 차이는 배율로 읽힌다 (출처: Wikipedia, Regression_de_Poisson.svg)</em></p>

<img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_poisson_regression_non_normal_loss_002.png" width="800" height="300"/>
<p align="center"><em>▲ 같은 보험 데이터에서 관측과 모형 3개의 예측 분포를 그린 히스토그램 8장 — 관측은 0 부터 30 까지 분포하는데 모형의 값은 0 과 4 사이에 모여 있다 (출처: scikit-learn documentation)</em></p>

**출력 형식** — 값 2개는 「이름 = 값 단위」 형태로 한 행에 하나씩 `print` 합니다. 채점은 값이 기준 범위 안에 들면 통과합니다.

**쓸 수 있는 것** — 「준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 |
|---|---|
| `df` | 전처리가 끝난 계약 표 |
| `rate_table` | 미션 2 의 열까지 채워진 요율표 |
| `post_mean_100` | `rate_table` 의 열 — 미션 2 가 채운 사후평균이고 비교 막대의 한쪽 |
| `glm_res` | 적합이 끝난 포아송 회귀 결과 객체 |
| `seg_frame` | 세그먼트 8개를 지역 · 브랜드 · 노출 1.0 열 3개로 담은 8행 표 |
| `brand_ref` | 기준 브랜드 이름 |
| `show_rate_table` | 지금까지 채운 열을 모두 담아 8행을 출력하는 함수 |

#### 프롬프트에 넣을 것(힌트)

- **예측 방법** — `seg_frame` 을 그대로 예측에 전달하고 오프셋은 0으로 둔다고 적습니다.
- **그래프** — 세그먼트마다 막대 2개씩 모두 16개, 그리고 범례와 가로축과 세로축의 이름.
- **내가 직접 확인할 것** — 배율을 읽는 문장의 방향이 B1 대비가 맞는지.

#### 실행 전 예상 — 모형이 계약 12건인 세그먼트에 주는 값

미션 1 의 R21 B11 추정값은 0.3 건/년 부근이었습니다. 이 세그먼트에 모형이 어떤 값을 산출할지 고르고, 왜 그렇게 판단했는지 한 문장 적으세요.

> **[예측] 계약 12건인 R21 B11 의 예측 사고율은 미션 1 의 최대우도추정값과 비교하면 어떻게 나올까요?**
>
> 1. 거의 같다
> 2. 훨씬 낮다
> 3. 훨씬 높다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

In [ ]:
# 미션 3 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 3 에 쓸 변수를 직접 만듭니다

if "df" not in globals():                                       # 이 미션만 따로 실행해도 단독으로 동작합니다
    from sklearn.datasets import fetch_openml                    # (앞 미션을 이미 실행했으면 통째로 건너뜁니다)
    df = fetch_openml(data_id=41214, as_frame=True).frame        # freMTPL2freq — 프랑스 자동차보험 계약별 사고 빈도
    df["IDpol"] = df["IDpol"].astype(int)
    for c in ["Area", "VehBrand", "VehGas", "Region"]:
        df[c] = df[c].astype(str).str.strip("'").astype("category")
    df["ClaimNb"] = df["ClaimNb"].astype(int).clip(upper=4)      # 5일 동안 고정해 온 전처리 2가지
    df["Exposure"] = df["Exposure"].clip(upper=1.0)

lam_all = df["ClaimNb"].sum() / df["Exposure"].sum()             # 계약 전체의 연간 사고율 (건/년)
watch_name = "R21 B11"                                           # 노출이 가장 적은 세그먼트

_grp = df.groupby(["Region", "VehBrand"], observed=True).agg(    # 지역 x 브랜드 242개
    n=("ClaimNb", "size"), claims=("ClaimNb", "sum"), exposure=("Exposure", "sum"))
_grp["n_claim_contracts"] = (df["ClaimNb"] > 0).astype(int).groupby(
    [df["Region"], df["VehBrand"]], observed=True).sum()
_picked = pd.concat([_grp.sort_values("exposure", ascending=False).head(4),      # 노출이 큰 4개
                     _grp[_grp["claims"] >= 1].sort_values("exposure").head(4)])  # 사고가 있는 작은 4개
seg_names = [f"{r} {b}" for r, b in _picked.index]
seg_region = [r for r, _ in _picked.index]
seg_brand = [b for _, b in _picked.index]
rate_table = _picked.reset_index(drop=True)[
    ["n", "claims", "exposure", "n_claim_contracts"]].astype(
        {"n": int, "claims": int, "n_claim_contracts": int}).copy()
rate_table.insert(0, "segment", seg_names)


def show_rate_table(*cols):
    """지금까지 채운 열을 모두 담아 세그먼트 8행을 그대로 출력합니다 (인자는 넣어도 됩니다)"""
    keep = ["segment", "n", "claims", "exposure"]
    keep += [c for c in rate_table.columns
             if c not in keep and c != "n_claim_contracts"]          # 앞 미션의 열까지 함께
    keep += [c for c in cols if c in rate_table.columns and c not in keep]
    print(rate_table[keep].round(4).to_string(index=False))

rate_table["mle_rate"] = rate_table["claims"] / rate_table["exposure"]
rate_table["se"] = np.sqrt(rate_table["claims"]) / rate_table["exposure"]
rate_table["lo_2se"] = rate_table["mle_rate"] - 2 * rate_table["se"]
rate_table["hi_2se"] = rate_table["mle_rate"] + 2 * rate_table["se"]

prior_exposure = 100.0
alpha_prior = lam_all * prior_exposure
rate_table["post_mean_100"] = ((alpha_prior + rate_table["claims"])
                               / (prior_exposure + rate_table["exposure"]))

if "glm_res" not in globals():                                   # 적합은 한 번만 실행합니다 (5초 안팎)
    glm_res = smf.glm("ClaimNb ~ C(Region) + C(VehBrand)", data=df,
                      family=sm.families.Poisson(),
                      offset=np.log(df["Exposure"].to_numpy())).fit()
brand_ref = "B1"                                                 # 기준 브랜드
seg_frame = pd.DataFrame({
    "Region": pd.Categorical(seg_region, categories=df["Region"].cat.categories),
    "VehBrand": pd.Categorical(seg_brand, categories=df["VehBrand"].cat.categories),
    "Exposure": 1.0})

print(f"세그먼트 {len(rate_table)}개 — 계약 {len(df):,}건 가운데 지역 x 브랜드 242개에서 고른 8개입니다")
print(rate_table[["segment", "n", "claims", "exposure"]].round(2).to_string(index=False))
print(f"계약 전체의 연간 사고율 lam_all = {lam_all:.6f} 건/년")

print(f"포아송 회귀 적합 완료 — 계수 {len(glm_res.params)}개 · 기준 브랜드 brand_ref = {brand_ref} · 예측 입력 seg_frame {len(seg_frame)}행")

print("쓸 수 있는 이름 : df (전처리가 끝난 계약 표) · rate_table (미션 2 의 열까지 채워진 요율표) · lam_all (계약 전체의 연간 사고율) · "
      "glm_res (적합이 끝난 포아송 회귀 결과) · seg_frame (세그먼트 8개의 예측 입력 표) · brand_ref (기준 브랜드 이름) · "
      "show_rate_table (지금까지 채운 열을 모두 출력하는 함수)")

In [ ]:
# 미션 3 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → AI 튜터에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

> **[문제] 로그 노출을 오프셋으로 반영하는 것과 설명변수로 반영하는 것은 무엇이 다릅니까?**
>
> 1. 오프셋은 계수를 추정하지 않고 1로 고정한다 — 노출이 2배면 기대 사고 건수도 정확히 2배가 되도록 명시하는 것이다
> 2. 오프셋도 계수를 추정하고 초기값만 1로 준다
> 3. 오프셋은 노출을 평균 0 · 표준편차 1 로 표준화한 뒤 모형에 반영한다

## 복원추출로 다시 계산하는 95% 구간

### 미션 4 — 복원추출로 다시 계산하는 95% 구간

#### 할 일 — 미션 4 (20분)

1. 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 아래 만들 것대로 「PROMPT 셀」의 PROMPT 에 지시를 적습니다.
3. 셀을 클릭한 채 AI 튜터에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶. 채점 결과가 ❌ 면 힌트대로 PROMPT 를 고쳐 다시 지시합니다.

**앞 미션에서 받는 것** — 미션 3 까지 표에는 값이 3개 들어왔지만 구간은 미션 1 이 만든 `lo_2se` 와 `hi_2se` 한 벌뿐입니다. 그 구간은 R21 B11 에서 하한이 -0.125 건/년이고 길이는 0.855 건/년이라 아직 그대로입니다. 미션 4 는 같은 세그먼트의 구간을 정규근사 대신 자료에서 직접 다시 계산해 하한이 음수를 벗어나는지부터 확인합니다.

**이 미션의 질문** — 정규근사를 쓰지 않고 자료에서 직접 계산하면 구간은 어떻게 달라집니까?

넷째 날에 표준오차가 추정값의 표본 간 변동이라는 것과, 재표집으로 그 변동을 직접 만들어 볼 수 있다는 것을 배웠습니다. 오늘은 세그먼트 안의 계약을 복원추출로 1,000번 다시 표집해 요율표의 넷째 열과 다섯째 열을 채웁니다. 미션 1 의 구간과 길이를 나란히 배치해 어느 쪽이 짧은지도 확인합니다.

#### 만들 것

「PROMPT 셀」 한 곳에서 아래가 모두 출력되게 합니다.

1. **R24 B2 의 부트스트랩 구간 길이** — 단위 건/년 · 소수점 아래 6자리 · 상한에서 하한을 뺀 값
2. **R21 B11 의 부트스트랩 구간 하한** — 단위 건/년 · 소수점 아래 6자리
3. **함께 출력할 표 하나** — `rate_table` 에 `boot_lo` 와 `boot_hi` 두 열을 더하고 `show_rate_table()` 로 8행을 그대로 출력합니다. 미션 3 까지의 열도 이 표에 함께 나옵니다. 채점 대상은 아닙니다
4. **그래프 한 장** — 세그먼트 8개의 구간 길이를 두 방법으로 비교하는 막대 16개
   - 세그먼트마다 표준오차 구간의 길이와 부트스트랩 구간의 길이를 나란히 배치합니다
   - 가로축 이름 : 「요율표의 세그먼트」 · 세로축 이름 : 「구간 길이 (건/년)」

**재표집 규칙 3가지** — 표본 하나는 그 세그먼트의 계약 수와 같은 건수로 추출합니다. 재표본마다 사고율은 사고 건수 합을 노출 합으로 나눕니다. 난수 시드를 정해 두면 다시 실행할 때 같은 값이 나옵니다(준비 셀의 `seed` 를 써도 됩니다).

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Illustration_bootstrap.svg" width="640" height="195"/>
<p align="center"><em>▲ 관측된 표본 하나에서 복원추출로 표본을 여러 벌 다시 만드는 절차 — 재표본마다 추정값을 계산해 모으면 그 분포가 추정값의 변동을 그대로 나타낸다 (출처: Wikipedia, Illustration_bootstrap.svg)</em></p>

**출력 형식** — 값 2개는 「이름 = 값 단위」 형태로 한 행에 하나씩 `print` 합니다. 채점은 값이 기준 범위 안에 들면 통과합니다. 부트스트랩은 난수가 달라 값이 조금 달라져도 됩니다.

**쓸 수 있는 것** — 「준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 |
|---|---|
| `df` | 전처리가 끝난 계약 표 |
| `rate_table` | 미션 3 의 열까지 채워진 요율표 |
| `lo_2se` · `hi_2se` | `rate_table` 의 열 — 미션 1 이 채운 표준오차 구간의 두 끝이고, 비교 막대의 한쪽은 이 두 열의 차이입니다 |
| `seg_masks` | 세그먼트 이름을 그 세그먼트의 계약을 선택하는 불리언 배열에 짝지은 사전 8개 |
| `boot_b` | 재표본 수 |
| `seed` | 난수 시드 |
| `show_rate_table` | 지금까지 채운 열을 모두 담아 8행을 출력하는 함수 |

#### 프롬프트에 넣을 것(힌트)

- **재표집** — 위 규칙 3가지를 그대로 적습니다.
- **절단 위치** — 아래 2.5% 지점과 위 2.5% 지점(백분위)에서 절단한다고 적습니다.
- **그래프** — 세그먼트마다 막대 2개씩 모두 16개, 그리고 가로축과 세로축의 이름.
- **내가 직접 확인할 것** — 막대 2개 가운데 어느 쪽이 부트스트랩 구간인지 범례로 구분되는지. 채점이 보지 않습니다.

#### 실행 전 예상 — 계약 12건인 세그먼트의 구간

미션 1 에서 R21 B11 의 구간은 하한이 음수였습니다. 재표집으로 다시 계산하면 그 구간이 어떻게 될지 하나를 고르고, 왜 그렇게 판단했는지 한 문장 적으세요.

> **[예측] 계약 12건인 세그먼트에서 부트스트랩 구간은 미션 1 의 구간보다 어떻게 나올까요?**
>
> 1. 훨씬 짧아진다
> 2. 길이는 비슷하고 하한만 0이 된다
> 3. 훨씬 길어진다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

In [ ]:
# 미션 4 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 4 에 쓸 변수를 직접 만듭니다

if "df" not in globals():                                       # 이 미션만 따로 실행해도 단독으로 동작합니다
    from sklearn.datasets import fetch_openml                    # (앞 미션을 이미 실행했으면 통째로 건너뜁니다)
    df = fetch_openml(data_id=41214, as_frame=True).frame        # freMTPL2freq — 프랑스 자동차보험 계약별 사고 빈도
    df["IDpol"] = df["IDpol"].astype(int)
    for c in ["Area", "VehBrand", "VehGas", "Region"]:
        df[c] = df[c].astype(str).str.strip("'").astype("category")
    df["ClaimNb"] = df["ClaimNb"].astype(int).clip(upper=4)      # 5일 동안 고정해 온 전처리 2가지
    df["Exposure"] = df["Exposure"].clip(upper=1.0)

lam_all = df["ClaimNb"].sum() / df["Exposure"].sum()             # 계약 전체의 연간 사고율 (건/년)
watch_name = "R21 B11"                                           # 노출이 가장 적은 세그먼트

_grp = df.groupby(["Region", "VehBrand"], observed=True).agg(    # 지역 x 브랜드 242개
    n=("ClaimNb", "size"), claims=("ClaimNb", "sum"), exposure=("Exposure", "sum"))
_grp["n_claim_contracts"] = (df["ClaimNb"] > 0).astype(int).groupby(
    [df["Region"], df["VehBrand"]], observed=True).sum()
_picked = pd.concat([_grp.sort_values("exposure", ascending=False).head(4),      # 노출이 큰 4개
                     _grp[_grp["claims"] >= 1].sort_values("exposure").head(4)])  # 사고가 있는 작은 4개
seg_names = [f"{r} {b}" for r, b in _picked.index]
seg_region = [r for r, _ in _picked.index]
seg_brand = [b for _, b in _picked.index]
rate_table = _picked.reset_index(drop=True)[
    ["n", "claims", "exposure", "n_claim_contracts"]].astype(
        {"n": int, "claims": int, "n_claim_contracts": int}).copy()
rate_table.insert(0, "segment", seg_names)


def show_rate_table(*cols):
    """지금까지 채운 열을 모두 담아 세그먼트 8행을 그대로 출력합니다 (인자는 넣어도 됩니다)"""
    keep = ["segment", "n", "claims", "exposure"]
    keep += [c for c in rate_table.columns
             if c not in keep and c != "n_claim_contracts"]          # 앞 미션의 열까지 함께
    keep += [c for c in cols if c in rate_table.columns and c not in keep]
    print(rate_table[keep].round(4).to_string(index=False))

rate_table["mle_rate"] = rate_table["claims"] / rate_table["exposure"]
rate_table["se"] = np.sqrt(rate_table["claims"]) / rate_table["exposure"]
rate_table["lo_2se"] = rate_table["mle_rate"] - 2 * rate_table["se"]
rate_table["hi_2se"] = rate_table["mle_rate"] + 2 * rate_table["se"]

prior_exposure = 100.0
alpha_prior = lam_all * prior_exposure
rate_table["post_mean_100"] = ((alpha_prior + rate_table["claims"])
                               / (prior_exposure + rate_table["exposure"]))

if "glm_res" not in globals():                                   # 적합은 한 번만 실행합니다 (5초 안팎)
    glm_res = smf.glm("ClaimNb ~ C(Region) + C(VehBrand)", data=df,
                      family=sm.families.Poisson(),
                      offset=np.log(df["Exposure"].to_numpy())).fit()
brand_ref = "B1"                                                 # 기준 브랜드
seg_frame = pd.DataFrame({
    "Region": pd.Categorical(seg_region, categories=df["Region"].cat.categories),
    "VehBrand": pd.Categorical(seg_brand, categories=df["VehBrand"].cat.categories),
    "Exposure": 1.0})

rate_table["glm_rate"] = np.asarray(
    glm_res.predict(seg_frame, offset=np.zeros(len(seg_frame))))

seg_masks = {name: ((df["Region"] == r) & (df["VehBrand"] == b)).to_numpy()
             for name, r, b in zip(seg_names, seg_region, seg_brand)}
boot_b, seed = 1000, 47                                          # 재표본 수와 난수 시드

print(f"세그먼트 {len(rate_table)}개 — 계약 {len(df):,}건 가운데 지역 x 브랜드 242개에서 고른 8개입니다")
print(rate_table[["segment", "n", "claims", "exposure"]].round(2).to_string(index=False))
print(f"계약 전체의 연간 사고율 lam_all = {lam_all:.6f} 건/년")

print(f"재표본 수 boot_b = {boot_b} · 난수 시드 seed = {seed} · seg_masks {len(seg_masks)}개")

print("쓸 수 있는 이름 : df (전처리가 끝난 계약 표) · rate_table (미션 3 의 열까지 채워진 요율표) · "
      "seg_masks (세그먼트 이름과 계약 선택 배열을 짝지은 사전 8개) · boot_b (재표본 수) · seed (난수 시드) · "
      "watch_name (노출이 가장 적은 세그먼트) · show_rate_table (지금까지 채운 열을 모두 출력하는 함수)")

In [ ]:
# 미션 4 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → AI 튜터에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

> **[문제] 부트스트랩 구간의 하한이 0이 된 까닭은 무엇입니까?**
>
> 1. 재표본이 사고를 한 건도 뽑지 못하는 경우가 2.5%보다 자주 나오기 때문이다
> 2. 백분위를 잘못 절단해 아래쪽을 2.5%가 아니라 0%에서 끊었기 때문이다
> 3. 노출이 짧아 사고율이 0으로 계산되기 때문이다

## 닫힌 형태로 계산하는 부분 풀링 값

### 미션 5 — 닫힌 형태로 계산하는 부분 풀링 값

#### 할 일 — 미션 5 (20분)

1. 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 아래 만들 것대로 「PROMPT 셀」의 PROMPT 에 지시를 적습니다.
3. 셀을 클릭한 채 AI 튜터에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶. 채점 결과가 ❌ 면 힌트대로 PROMPT 를 고쳐 다시 지시합니다.

**앞 미션에서 받는 것** — 미션 4 의 `boot_lo` 열에서 R21 B11 의 하한은 0 이라 음수는 사라졌습니다. 다만 구간 길이가 0.824 건/년이어서 미션 1 의 0.855 건/년과 거의 같습니다. 미션 5 는 값과 구간을 함께 수축시킵니다.

**이 미션의 질문** — 세그먼트 자신의 값과 계약 전체의 값에 각각 어떤 가중치를 적용해 결합해야 합니까?

다섯째 날에 부분 풀링이 세그먼트마다 다른 가중치로 두 값을 결합하는 방법이라는 것을 배웠습니다. 오늘은 표집 없이 `w = exposure / (exposure + b_strong)` 으로 곧장 계산하며 `b_strong` 은 400년입니다. 부분 풀링 값은 w × 그 세그먼트의 최대우도추정값 + (1 − w) × 계약 전체의 값입니다.

`b_strong` 400년은 다섯째 날 이론에서 역산해 얻은 407년을 반올림한 값입니다.

**신용구간** — 사후분포는 감마분포이고, 형태 모수는 `alpha_strong` 에 사고 건수 합을, 비율 모수는 `b_strong` 에 노출 합을 더한 값입니다. 95% 신용구간은 이 분포의 2.5% 분위수와 97.5% 분위수이고, 미션 4 의 백분위와 같은 지점입니다.

#### 만들 것

「PROMPT 셀」 한 곳에서 아래가 모두 출력되게 합니다.

1. **R21 B11 의 부분 풀링 값** — 단위 건/년 · 소수점 아래 6자리
2. **R21 B11 의 신용구간 길이** — 단위 건/년 · 소수점 아래 6자리 · 97.5% 분위수에서 2.5% 분위수를 뺀 값
3. **함께 출력할 표 하나** — `rate_table` 에 `pooled_400` · `cred_lo_400` · `cred_hi_400` 열 3개를 더하고 `show_rate_table()` 로 8행을 그대로 출력합니다. 사전 강도 100년일 때의 값은 미션 2 의 `post_mean_100` 과 같아 새 열을 만들지 않습니다. 채점 대상은 아닙니다
4. **그래프 한 장** — 가로축은 노출 합(로그 눈금), 세로축은 가중치 w 인 산점도
   - 사전 강도 100년과 400년을 모두 표시해 점이 16개입니다. 100년 쪽 가중치는 같은 식의 `b_strong` 대신 `prior_exposure` 를 대입한 값입니다
   - 가로축 이름 : 「세그먼트의 노출 합 (년)」 · 세로축 이름 : 「내 세그먼트 값에 실리는 가중치」

<img src="https://en.wikipedia.org/wiki/Special:FilePath/MSE_of_ML_vs_JS.png" width="560" height="456"/>
<p align="center"><em>▲ 항목마다 따로 추정한 값과 전체 평균 쪽으로 수축시킨 값의 오차 비교 — 항목마다 관측이 적을 때는 수축시킨 쪽의 평균 오차가 더 작다 (출처: Wikipedia, MSE_of_ML_vs_JS.png)</em></p>

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Equal-tailed_credibility_interval.svg" width="640" height="293"/>
<p align="center"><em>▲ 사후분포의 양쪽 꼬리를 2.5%씩 절단해 얻는 등꼬리 신용구간 — 미션 5 에서 감마 사후분포에 같은 방식으로 두 끝을 구한다 (출처: Wikipedia, Equal-tailed_credibility_interval.svg)</em></p>

**출력 형식** — 값 2개는 「이름 = 값 단위」 형태로 한 행에 하나씩 `print` 합니다. 채점은 값이 기준 범위 안에 들면 통과합니다.

**쓸 수 있는 것** — 「준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 |
|---|---|
| `df` | 전처리가 끝난 계약 표 |
| `rate_table` | 미션 4 의 열까지 채워진 요율표 |
| `lam_all` | 계약 전체의 연간 사고율, 단위 건/년 |
| `prior_exposure` | 사전 강도 100년 |
| `b_strong` | 사전 강도 400년 |
| `alpha_strong` | 사전 강도 400년일 때의 감마 형태 모수 |
| `show_rate_table` | 지금까지 채운 열을 모두 담아 8행을 출력하는 함수 |

#### 프롬프트에 넣을 것(힌트)

- **가중치** — 위 식을 그대로 적습니다.
- **신용구간** — `scipy.stats.gamma` 는 비율 모수의 역수를 `scale` 로 받는다고 적습니다.
- **출력** — 값 2개의 이름과 단위.
- **그래프** — 점 16개, 로그 눈금인 가로축, 가로축과 세로축의 이름.
- **내가 직접 확인할 것** — 점 16개 가운데 어느 쪽이 사전 강도 400년인지 범례로 구분되는지.

#### 실행 전 예상 — 사전 강도를 높였을 때의 구간 길이

관측이 적은 세그먼트의 신용구간이 어떻게 될지 고르고, 왜 그렇게 판단했는지 한 문장 적으세요.

> **[예측] 사전 강도를 100년에서 400년으로 높이면 R21 B11 의 신용구간 길이는 어떻게 될까요?**
>
> 1. 짧아진다
> 2. 길어진다
> 3. 달라지지 않는다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

In [ ]:
# 미션 5 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 5 에 쓸 변수를 직접 만듭니다

if "df" not in globals():                                       # 이 미션만 따로 실행해도 단독으로 동작합니다
    from sklearn.datasets import fetch_openml                    # (앞 미션을 이미 실행했으면 통째로 건너뜁니다)
    df = fetch_openml(data_id=41214, as_frame=True).frame        # freMTPL2freq — 프랑스 자동차보험 계약별 사고 빈도
    df["IDpol"] = df["IDpol"].astype(int)
    for c in ["Area", "VehBrand", "VehGas", "Region"]:
        df[c] = df[c].astype(str).str.strip("'").astype("category")
    df["ClaimNb"] = df["ClaimNb"].astype(int).clip(upper=4)      # 5일 동안 고정해 온 전처리 2가지
    df["Exposure"] = df["Exposure"].clip(upper=1.0)

lam_all = df["ClaimNb"].sum() / df["Exposure"].sum()             # 계약 전체의 연간 사고율 (건/년)
watch_name = "R21 B11"                                           # 노출이 가장 적은 세그먼트

_grp = df.groupby(["Region", "VehBrand"], observed=True).agg(    # 지역 x 브랜드 242개
    n=("ClaimNb", "size"), claims=("ClaimNb", "sum"), exposure=("Exposure", "sum"))
_grp["n_claim_contracts"] = (df["ClaimNb"] > 0).astype(int).groupby(
    [df["Region"], df["VehBrand"]], observed=True).sum()
_picked = pd.concat([_grp.sort_values("exposure", ascending=False).head(4),      # 노출이 큰 4개
                     _grp[_grp["claims"] >= 1].sort_values("exposure").head(4)])  # 사고가 있는 작은 4개
seg_names = [f"{r} {b}" for r, b in _picked.index]
seg_region = [r for r, _ in _picked.index]
seg_brand = [b for _, b in _picked.index]
rate_table = _picked.reset_index(drop=True)[
    ["n", "claims", "exposure", "n_claim_contracts"]].astype(
        {"n": int, "claims": int, "n_claim_contracts": int}).copy()
rate_table.insert(0, "segment", seg_names)


def show_rate_table(*cols):
    """지금까지 채운 열을 모두 담아 세그먼트 8행을 그대로 출력합니다 (인자는 넣어도 됩니다)"""
    keep = ["segment", "n", "claims", "exposure"]
    keep += [c for c in rate_table.columns
             if c not in keep and c != "n_claim_contracts"]          # 앞 미션의 열까지 함께
    keep += [c for c in cols if c in rate_table.columns and c not in keep]
    print(rate_table[keep].round(4).to_string(index=False))

rate_table["mle_rate"] = rate_table["claims"] / rate_table["exposure"]
rate_table["se"] = np.sqrt(rate_table["claims"]) / rate_table["exposure"]
rate_table["lo_2se"] = rate_table["mle_rate"] - 2 * rate_table["se"]
rate_table["hi_2se"] = rate_table["mle_rate"] + 2 * rate_table["se"]

prior_exposure = 100.0
alpha_prior = lam_all * prior_exposure
rate_table["post_mean_100"] = ((alpha_prior + rate_table["claims"])
                               / (prior_exposure + rate_table["exposure"]))

if "glm_res" not in globals():                                   # 적합은 한 번만 실행합니다 (5초 안팎)
    glm_res = smf.glm("ClaimNb ~ C(Region) + C(VehBrand)", data=df,
                      family=sm.families.Poisson(),
                      offset=np.log(df["Exposure"].to_numpy())).fit()
brand_ref = "B1"                                                 # 기준 브랜드
seg_frame = pd.DataFrame({
    "Region": pd.Categorical(seg_region, categories=df["Region"].cat.categories),
    "VehBrand": pd.Categorical(seg_brand, categories=df["VehBrand"].cat.categories),
    "Exposure": 1.0})

rate_table["glm_rate"] = np.asarray(
    glm_res.predict(seg_frame, offset=np.zeros(len(seg_frame))))

seg_masks = {name: ((df["Region"] == r) & (df["VehBrand"] == b)).to_numpy()
             for name, r, b in zip(seg_names, seg_region, seg_brand)}
boot_b, seed = 1000, 47                                          # 재표본 수와 난수 시드

_rng = np.random.default_rng(seed)
_lo, _hi = [], []
for _name in seg_names:
    _part = df[seg_masks[_name]]
    _y = _part["ClaimNb"].to_numpy(dtype=float)
    _e = _part["Exposure"].to_numpy(dtype=float)
    _rates = np.empty(boot_b)
    for _b in range(boot_b):
        _idx = _rng.integers(0, len(_y), len(_y))
        _rates[_b] = _y[_idx].sum() / _e[_idx].sum()
    _lo.append(float(np.percentile(_rates, 2.5)))
    _hi.append(float(np.percentile(_rates, 97.5)))
rate_table["boot_lo"], rate_table["boot_hi"] = _lo, _hi

b_strong = 400.0                                                 # 다섯째 날 모형이 정한 사전 강도 (년)
alpha_strong = lam_all * b_strong

print(f"세그먼트 {len(rate_table)}개 — 계약 {len(df):,}건 가운데 지역 x 브랜드 242개에서 고른 8개입니다")
print(rate_table[["segment", "n", "claims", "exposure"]].round(2).to_string(index=False))
print(f"계약 전체의 연간 사고율 lam_all = {lam_all:.6f} 건/년")

prior_exposure = 100.0                                           # 미션 2 에서 쓴 사전 강도 (년)
print(f"사전 강도 prior_exposure = {prior_exposure:.0f} 년 · b_strong = {b_strong:.0f} 년 · alpha_strong = {alpha_strong:.4f}")
print("b_strong 400년의 유래 — 다섯째 날 이론의 "
      "「수축의 크기를 정하는 값: 노출」에서 부분 풀링 값으로부터 "
      "사전 강도를 B = E(1 - w) / w 로 역산해 407년을 얻었고, 이 권은 그 값을 "
      "400년으로 적습니다. 오늘은 다시 추정하지 않고 상수로 받습니다")

print("쓸 수 있는 이름 : df (전처리가 끝난 계약 표) · rate_table (미션 4 의 열까지 채워진 요율표) · lam_all (계약 전체의 연간 사고율) · "
      "prior_exposure (사전 강도 100년) · b_strong (사전 강도 400년) · alpha_strong (사전 강도 400년의 형태 모수) · "
      "watch_name (노출이 가장 적은 세그먼트) · show_rate_table (지금까지 채운 열을 모두 출력하는 함수)")

In [ ]:
# 미션 5 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → AI 튜터에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

> **[문제] 다섯째 날에는 표집으로 사후분포를 얻고 R-hat 과 유효표본수로 확인했습니다. 미션 5 가 닫힌 형태로 구한 값에도 그 확인이 필요합니까?**
>
> 1. 필요하지 않다 — 표집을 쓰지 않아 R-hat 과 유효표본수를 계산할 표본 자체가 없다
> 2. 필요하다 — 값을 사후분포에서 구했으므로 R-hat 과 유효표본수를 함께 확인해야 한다
> 3. 필요하다 — 닫힌 형태의 계산도 반복이라 반복마다 R-hat 을 구할 수 있다

## 요율표 한 장의 완성과 심사 경로 분리

> **[과제] 과제 제출 — 요율표 한 장의 완성과 심사 경로 분리**
>
> - **무엇을** 세그먼트 8개의 요율표를 파일 하나로 저장하고 값 2개와 그림 한 장, 브리핑 3문장을 만듭니다.
> - **무엇으로** 5일 동안 쓴 계약 678,013건 하나로 하고, 「준비 셀」이 추정 방법 5개의 열을 모두 다시 만들어 둡니다.
> - **인정** 채점이 값 2개와 그래프를 통과시킨 노트북에 브리핑 3문장을 적은 것입니다.
> - **예** 저장 파일은 `요율표_복습판.csv` 이고 판정 열의 값은 「자동 승인」 과 「사람 심사」 두 가지입니다.
> 
> - 내가 쓴 `PROMPT` 를 지우지 말고 그대로 둔 채 「PROMPT 셀」을 실행해 값 2개와 그래프가 통과로 열린 상태로 제출할 것
> - 세그먼트마다 요율표에 적을 값을 4가지 가운데 골라 `report_rate` 열에 적을 것
> - 심사를 가를 기준을 하나 정해 `decision` 열을 「자동 승인」 과 「사람 심사」 로 나눌 것
> - 완성한 표를 `요율표_복습판.csv` 로 저장하고 다시 읽어 행 수를 확인할 것
> - 아래 「제출 전 마지막 한 장」 양식에 브리핑을 결론·근거·한계 순서로 3문장 적을 것
>
> 마감: 2026-10-05T23:59+09:00
>
> **평가 기준**
>
> | 기준 | 이렇게 하면 충족 | 배점 |
> | --- | --- | --- |
> | 문제 정의 | 프롬프트에 목표와 쓸 이름과 출력 형식이 내 말로 적혀 있다 | 25 |
> | 검증 설계 | 값 2개와 그래프가 통과로 열렸고 요율표에 적은 값의 출처가 열 이름으로 밝혀져 있다 | 25 |
> | 오류 탐지 | 한 표 안에서 다루는 값을 혼동하지 않았고 빠지기 쉬운 함정 3가지에 O 와 X 가 적혀 있다 | 25 |
> | 해석·보고 | 브리핑에 결론과 근거와 한계가 각 한 문장씩 있고 3문장 안에 끝난다 | 25 |
>
> 제출은 LMS 레슨 화면의 과제 카드에서 GitHub 또는 Google Drive 링크로 합니다(이 파일에 적으면 제출되지 않습니다).

### 과제 — 요율표 한 장의 완성과 심사 경로 분리

#### 할 일 — 과제 (20분)

1. 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 아래 만들 것대로 「PROMPT 셀」의 PROMPT 에 지시를 적습니다.
3. 셀을 클릭한 채 AI 튜터에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶. 채점 결과가 ❌ 면 힌트대로 PROMPT 를 고쳐 다시 지시합니다.

**앞 미션에서 받는 것** — 미션 5 까지 끝나 세그먼트마다 값이 4개 있습니다. R21 B11 한 행만 보아도 최대우도추정값 0.302, 사후평균 0.113, 예측 사고율 0.118, 부분 풀링 값 0.104 건/년입니다.

**이 과제의 질문** — 세그먼트 8개의 요율표 한 장을 재무팀에 전달할 때, 각 세그먼트에 어떤 값을 적고 어느 세그먼트를 사람이 다시 봐야 합니까?

#### 만들 것

「PROMPT 셀」 한 곳에서 아래가 모두 출력되게 합니다.

1. **저장한 요율표의 행 수** — 단위 행 · 정수 · 저장한 파일을 다시 읽어 구한 행의 개수
2. **요율표 값이 신용구간 안에 든 비율** — 단위 % · 소수점 아래 1자리 · `report_rate` 가 `cred_lo_400` 과 `cred_hi_400` 사이에 있는 세그먼트의 비율
3. **표 한 장** — 판정 열 `decision` 까지 담은 열 19개의 `rate_table` 을 `요율표_복습판.csv` 로 저장한 뒤 `show_rate_table()` 로 눈으로 확인합니다. 화면 표는 사고를 겪은 계약 수를 뺀 18개입니다
4. **`rel_width` 열** — `(cred_hi_400 - cred_lo_400) / pooled_400` 로 만듭니다. 5번 그래프의 세로축입니다
5. **그래프 한 장** — 가로축은 노출 합(로그 눈금), 세로축은 `rel_width` 인 산점도이고 점은 8개입니다
   - 가로축 이름 : 「세그먼트의 노출 합 (년)」 · 세로축 이름 : 「상대 구간 길이」
   - 내가 정한 기준값에 가로선 하나를 그립니다. 위치는 채점 대상이 아닙니다

<img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_poisson_regression_non_normal_loss_003.png" width="600" height="400"/>
<p align="center"><em>▲ 계약을 예측 위험 순으로 정렬해 예측과 관측을 나란히 그린 보정 그림 4장 — 오른쪽 아래 한 장만 평평한데, 모든 계약에 계약 전체의 값 하나를 적은 표가 이런 형태다 (출처: scikit-learn documentation)</em></p>

**출력 형식** — 값 2개는 「이름 = 값 단위」 형태로 한 행에 하나씩 `print` 합니다. 두 번째 값의 비율은 적은 값과 구간이 서로 어긋나지 않는지를 보는 확인이고, 어느 방법이 옳은지를 가리는 근거는 아닙니다.

#### 내가 정할 것 3가지

1. **어느 값을 적을까** — 8행 전체에 복사해도 모두 신용구간 안에 드는 것은 `pooled_400` · `post_mean_100` · `glm_rate` 3가지입니다. `mle_rate` 는 R21 B11 · R43 B4 · R21 B13 에서 구간 밖이라 그 3개만 다른 값으로 바꿔야 통과합니다.
2. **무엇으로 가를까** — 상대 구간 길이, 누적 사고 건수, 계약 수 가운데 하나와 기준값을 정합니다.
3. **사람 심사 세그먼트를 어떻게 적을까** — 같은 표에 남기거나, 상위 세그먼트로 묶거나, 계약 전체의 값을 적습니다.

아래 범위 선언은 선택 항목입니다. 브리핑을 쓰기 전에 생각을 정리하고 싶을 때 채웁니다.

> 나의 범위 선언(선택):
> 1. 요율표에 적을 값 —
> 2. 심사를 가를 기준과 기준값 —
> 3. 사람 심사 세그먼트를 적는 방법 —

#### 빠지기 쉬운 함정 — 제출 전에 O 와 X 를 적습니다

| 확인할 것 | O / X |
|---|---|
| 한 표 안에 비율과 건/년을 함께 담지 않았습니까 | |
| 요율표에 적은 값과 구간이 같은 방법에서 나온 것입니까 | |
| 계약 12건인 세그먼트를 자동 승인으로 보내지 않았습니까 | |

**쓸 수 있는 것** — 「준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 |
|---|---|
| `df` | 전처리가 끝난 계약 표 |
| `rate_table` | 미션 1부터 5까지의 열 16개가 모두 채워진 요율표 |
| `lam_all` | 계약 전체의 연간 사고율, 단위 건/년 |
| `b_strong` | 사전 강도 400년 |
| `out_path` | 저장할 파일 이름 |
| `seg_names` | 세그먼트 이름 8개 |
| `watch_name` | 노출이 가장 적은 세그먼트의 이름 |
| `show_rate_table` | 지금까지 채운 열을 모두 담아 8행을 출력하는 함수 |

#### 프롬프트에 넣을 것(힌트)

- **적을 값과 판정** — `report_rate` 에 복사할 열 이름, 그리고 `rel_width` 와 기준값으로 `decision` 을 두 값으로 나눈다는 것.
- **저장과 출력** — `out_path` 로 저장하고 다시 읽어 행 수와 완성된 표를 출력합니다.
- **그래프** — 로그 눈금인 가로축, 점 8개, 가로선 하나, 가로축과 세로축의 이름.
- **내가 직접 확인할 것** — `decision` 열에 「자동 승인」 과 「사람 심사」 두 값만 있는지. 채점이 보지 않습니다.

In [ ]:
# 과제 · 준비 — 읽고 실행만 합니다. 이 셀이 과제에 쓸 변수를 직접 만듭니다

if "df" not in globals():                                       # 이 미션만 따로 실행해도 단독으로 동작합니다
    from sklearn.datasets import fetch_openml                    # (앞 미션을 이미 실행했으면 통째로 건너뜁니다)
    df = fetch_openml(data_id=41214, as_frame=True).frame        # freMTPL2freq — 프랑스 자동차보험 계약별 사고 빈도
    df["IDpol"] = df["IDpol"].astype(int)
    for c in ["Area", "VehBrand", "VehGas", "Region"]:
        df[c] = df[c].astype(str).str.strip("'").astype("category")
    df["ClaimNb"] = df["ClaimNb"].astype(int).clip(upper=4)      # 5일 동안 고정해 온 전처리 2가지
    df["Exposure"] = df["Exposure"].clip(upper=1.0)

lam_all = df["ClaimNb"].sum() / df["Exposure"].sum()             # 계약 전체의 연간 사고율 (건/년)
watch_name = "R21 B11"                                           # 노출이 가장 적은 세그먼트

_grp = df.groupby(["Region", "VehBrand"], observed=True).agg(    # 지역 x 브랜드 242개
    n=("ClaimNb", "size"), claims=("ClaimNb", "sum"), exposure=("Exposure", "sum"))
_grp["n_claim_contracts"] = (df["ClaimNb"] > 0).astype(int).groupby(
    [df["Region"], df["VehBrand"]], observed=True).sum()
_picked = pd.concat([_grp.sort_values("exposure", ascending=False).head(4),      # 노출이 큰 4개
                     _grp[_grp["claims"] >= 1].sort_values("exposure").head(4)])  # 사고가 있는 작은 4개
seg_names = [f"{r} {b}" for r, b in _picked.index]
seg_region = [r for r, _ in _picked.index]
seg_brand = [b for _, b in _picked.index]
rate_table = _picked.reset_index(drop=True)[
    ["n", "claims", "exposure", "n_claim_contracts"]].astype(
        {"n": int, "claims": int, "n_claim_contracts": int}).copy()
rate_table.insert(0, "segment", seg_names)


def show_rate_table(*cols):
    """지금까지 채운 열을 모두 담아 세그먼트 8행을 그대로 출력합니다 (인자는 넣어도 됩니다)"""
    keep = ["segment", "n", "claims", "exposure"]
    keep += [c for c in rate_table.columns
             if c not in keep and c != "n_claim_contracts"]          # 앞 미션의 열까지 함께
    keep += [c for c in cols if c in rate_table.columns and c not in keep]
    print(rate_table[keep].round(4).to_string(index=False))

rate_table["mle_rate"] = rate_table["claims"] / rate_table["exposure"]
rate_table["se"] = np.sqrt(rate_table["claims"]) / rate_table["exposure"]
rate_table["lo_2se"] = rate_table["mle_rate"] - 2 * rate_table["se"]
rate_table["hi_2se"] = rate_table["mle_rate"] + 2 * rate_table["se"]

prior_exposure = 100.0
alpha_prior = lam_all * prior_exposure
rate_table["post_mean_100"] = ((alpha_prior + rate_table["claims"])
                               / (prior_exposure + rate_table["exposure"]))

if "glm_res" not in globals():                                   # 적합은 한 번만 실행합니다 (5초 안팎)
    glm_res = smf.glm("ClaimNb ~ C(Region) + C(VehBrand)", data=df,
                      family=sm.families.Poisson(),
                      offset=np.log(df["Exposure"].to_numpy())).fit()
brand_ref = "B1"                                                 # 기준 브랜드
seg_frame = pd.DataFrame({
    "Region": pd.Categorical(seg_region, categories=df["Region"].cat.categories),
    "VehBrand": pd.Categorical(seg_brand, categories=df["VehBrand"].cat.categories),
    "Exposure": 1.0})

rate_table["glm_rate"] = np.asarray(
    glm_res.predict(seg_frame, offset=np.zeros(len(seg_frame))))

seg_masks = {name: ((df["Region"] == r) & (df["VehBrand"] == b)).to_numpy()
             for name, r, b in zip(seg_names, seg_region, seg_brand)}
boot_b, seed = 1000, 47                                          # 재표본 수와 난수 시드

_rng = np.random.default_rng(seed)
_lo, _hi = [], []
for _name in seg_names:
    _part = df[seg_masks[_name]]
    _y = _part["ClaimNb"].to_numpy(dtype=float)
    _e = _part["Exposure"].to_numpy(dtype=float)
    _rates = np.empty(boot_b)
    for _b in range(boot_b):
        _idx = _rng.integers(0, len(_y), len(_y))
        _rates[_b] = _y[_idx].sum() / _e[_idx].sum()
    _lo.append(float(np.percentile(_rates, 2.5)))
    _hi.append(float(np.percentile(_rates, 97.5)))
rate_table["boot_lo"], rate_table["boot_hi"] = _lo, _hi

b_strong = 400.0                                                 # 다섯째 날 모형이 정한 사전 강도 (년)
alpha_strong = lam_all * b_strong

_w400 = rate_table["exposure"] / (rate_table["exposure"] + b_strong)
rate_table["pooled_400"] = _w400 * rate_table["mle_rate"] + (1 - _w400) * lam_all
rate_table["cred_lo_400"] = stats.gamma.ppf(
    0.025, a=alpha_strong + rate_table["claims"],
    scale=1.0 / (b_strong + rate_table["exposure"]))
rate_table["cred_hi_400"] = stats.gamma.ppf(
    0.975, a=alpha_strong + rate_table["claims"],
    scale=1.0 / (b_strong + rate_table["exposure"]))

print(f"세그먼트 {len(rate_table)}개 — 계약 {len(df):,}건 가운데 지역 x 브랜드 242개에서 고른 8개입니다")
print(rate_table[["segment", "n", "claims", "exposure"]].round(2).to_string(index=False))
print(f"계약 전체의 연간 사고율 lam_all = {lam_all:.6f} 건/년")

out_path = "요율표_복습판.csv"                                     # 과제가 저장할 파일 이름
print(f"요율표의 열 {rate_table.shape[1]}개가 채워졌습니다 — 과제가 report_rate · rel_width · decision 3개를 더합니다")
print(f"저장할 파일 out_path = {out_path}")

print("쓸 수 있는 이름 : df (전처리가 끝난 계약 표) · rate_table (미션 1부터 5까지의 열 16개가 채워진 요율표) · lam_all (계약 전체의 연간 사고율) · "
      "b_strong (사전 강도 400년) · out_path (저장할 파일 이름) · seg_names (세그먼트 이름 8개) · "
      "watch_name (노출이 가장 적은 세그먼트) · show_rate_table (지금까지 채운 열을 모두 출력하는 함수)")

In [ ]:
# 과제 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → AI 튜터에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

**제출 전 마지막 한 장** — 값 2개와 그래프가 모두 통과로 열렸으면 아래 양식대로 브리핑을 작성해 노트북 끝에 남깁니다. 결론·근거·한계 각 한 문장으로 모두 3문장입니다. 채점자는 이 브리핑을 가장 먼저 읽습니다.

> 브리핑:
> 1. 결론 — 요율표에 적을 값으로 어느 방법을 골랐는지와 그 이유를 한 문장으로 적습니다
> 2. 근거 — 자동 승인으로 보낸 세그먼트 수와 사람 심사로 보낸 세그먼트 수를 숫자로 적습니다
> 3. 한계 — 사전 강도 400년이 다섯째 날 모형에서 얻은 값이라는 것, 세그먼트가 8개뿐이라는 것, 이 표가 빈도만 담아 요율로 쓰려면 건당 손해액을 곱해야 한다는 것을 적습니다

한계 문장은 이 요율표를 242개 세그먼트 전체로 확장할 것인지, 오늘 다룬 8개에만 적용할 것인지에 따라 달라집니다.

## 오늘의 정리와 추정 방법 5개의 쓰임

오늘 추정 방법 5개를 한 번씩 다시 실행했습니다. 요율표의 열이 어느 날의 방법에서 나왔는지 한 행씩 정리합니다.

| 요율표의 열 | 어느 날의 방법인가 | 한 문장 |
|---|---|---|
| `mle_rate` | 첫날 | 그 세그먼트의 자료만으로 한 숫자를 정합니다 |
| `se` · `lo_2se` · `hi_2se` | 넷째 날 | 그 한 숫자가 표본마다 얼마나 흔들리는지를 구간으로 적습니다 |
| `post_mean_100` | 둘째 날 | 사전 강도 100년만큼 계약 전체의 값을 반영합니다 |
| `glm_rate` | 셋째 날 | 지역과 브랜드의 공통 경향을 함께 써서 값을 정합니다 |
| `boot_lo` · `boot_hi` | 넷째 날 | 정규근사를 쓰지 않고 자료에서 직접 구간을 계산합니다 |
| `pooled_400` · `cred_lo_400` · `cred_hi_400` | 다섯째 날 | 결합 비율을 사전 강도가 정한 값으로 맞춥니다 |

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Poisson_pmf.svg" width="560" height="433"/>
<p align="center"><em>▲ 평균이 다른 포아송분포 3가지의 확률질량함수 — 오늘 추정 방법 5개가 모두 이 분포 하나로 사고 건수를 다뤘고, 달랐던 것은 그 평균을 어디에서 얻는가였다 (출처: Wikipedia, Poisson_pmf.svg)</em></p>

## 오늘 남는 원칙

값 하나만 적힌 요율표는 관측이 적은 세그먼트에서 읽는 사람을 오해하게 만듭니다. **값과 구간과 그 값을 만든 방법의 이름이 한 행에 함께 있어야 합니다.** 과제 앞에서 확인한 R21 B11 한 행의 값 4개가 그 근거입니다. 값만 보면 어느 것이 맞는지 알 수 없고, 구간과 방법 이름을 함께 보아야 각 값이 무엇을 전제한 값인지 구분됩니다.

노출이 큰 세그먼트 4개에서는 최대우도추정값과 사후평균과 회귀 예측값과 부분 풀링 값이 서로 3% 안쪽에서 일치합니다. R24 B2 는 값 4개가 0.0868 부터 0.0891 건/년 사이에 모여 있어 어느 방법의 값을 적어도 요율이 사실상 같습니다. 방법의 차이는 언제나 관측이 적은 세그먼트에서 드러납니다. 그래서 요율표를 볼 때 가장 먼저 확인할 열은 사고 건수와 노출입니다.

> **[문제] 사람 심사로 보낸 세그먼트를 다음 분기에 자동 승인으로 옮기려면 무엇이 달라져야 합니까?**
>
> 내 요율표에서 사람 심사로 보낸 세그먼트를 다음 분기에 자동 승인으로 옮기려면 무엇이 달라져야 합니까? 계약을 더 모으는 것과 세그먼트를 합치는 것과 사전 강도를 높이는 것 3가지를 비교해 팀과 이야기해 보세요. 3가지 가운데 2가지는 그 세그먼트를 새로 알게 해 주지 않는다는 점도 함께 확인해 보세요.